# A Modular Episodic Framework for Cross-Design Comparison in Multi-Mode Industrial Time Series

**Authors:** Ján Skalka, Małgorzata Przybyła-Kasperek, Ľubomír Benko, Martin Drlík, Dominik Halvoník, Kacper Książek

This notebook accompanies the article and follows the same stage-based workflow used in the manuscript.

**Folder convention**
- `data/` – preparation, Stage 1, and Stage 2 intermediate and final outputs
- `Stage3/` – episodic standardization, boundary stabilization, prototypes, and meta-mode alignment
- `Stage4/` – cross-design episodic evaluation outputs


## Preparation

Join the original CSV files into one combined dataset.

In [1]:
from pathlib import Path
import pandas as pd
import re

# -----------------------------
# Settings
# -----------------------------
DATA_DIR = Path("data")
OUTPUT_CSV = DATA_DIR / "combined_dataset.csv"

# -----------------------------
# Helper functions
# -----------------------------
def natural_key(text):
    """
    Natural sorting:
    file2.csv < file10.csv
    """
    return [int(c) if c.isdigit() else c.lower() for c in re.split(r"(\d+)", str(text))]

def read_csv_flexible(path):
    """
    Robust CSV loading while preserving raw text exactly.
    Important: dtype=str keeps DateTime unchanged, including seconds if present.
    """
    # try separator autodetection first
    try:
        return pd.read_csv(
            path,
            sep=None,
            engine="python",
            dtype=str,
            keep_default_na=False,
            na_filter=False,
            encoding="utf-8-sig"
        )
    except Exception:
        # fallback separators
        for sep in [",", ";", "\t", "|"]:
            try:
                return pd.read_csv(
                    path,
                    sep=sep,
                    dtype=str,
                    keep_default_na=False,
                    na_filter=False,
                    encoding="utf-8-sig"
                )
            except Exception:
                pass
    raise ValueError(f"Could not read file: {path}")

def clean_column_names(df):
    df.columns = [
        str(c).strip() if str(c).strip() else f"unnamed_{idx}"
        for idx, c in enumerate(df.columns)
    ]
    return df

def strip_string_cells(df):
    """
    Strip whitespace from all string cells.
    Keeps raw content otherwise unchanged.
    """
    return df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

def inspect_datetime_column(df, file_name):
    """
    Quick check whether DateTime appears to contain seconds.
    """
    if "DateTime" not in df.columns:
        print(f"[WARN] {file_name}: missing DateTime column")
        return

    sample = df["DateTime"].dropna().astype(str).head(5).tolist()
    has_seconds = df["DateTime"].astype(str).str.contains(r"\d{1,2}:\d{2}:\d{2}$", regex=True).any()

    print(f"\n{file_name}")
    print(f"  sample DateTime values: {sample}")
    print(f"  contains seconds: {has_seconds}")

# -----------------------------
# Find input files
# -----------------------------
csv_files = sorted(
    [f for f in DATA_DIR.glob("*.csv") if f.name != OUTPUT_CSV.name],
    key=natural_key
)

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR.resolve()}")

print("Found files:")
for i, f in enumerate(csv_files, start=1):
    print(f"{i:02d}. {f.name}")

# -----------------------------
# Load files and collect columns
# -----------------------------
dfs = []
all_columns = set()

for session_id, file_path in enumerate(csv_files, start=1):
    df = read_csv_flexible(file_path)
    df = clean_column_names(df)
    df = strip_string_cells(df)

    # Inspect DateTime before anything else
    inspect_datetime_column(df, file_path.name)

    # Preserve raw DateTime exactly as present in source
    # Optional: duplicate into DateTime_raw for later debugging
    if "DateTime" in df.columns:
        df["DateTime_raw"] = df["DateTime"]

    # Add identifiers
    df["session_id"] = str(session_id)
    df["source_file"] = file_path.name

    dfs.append(df)
    all_columns.update(df.columns)

# -----------------------------
# Reorder columns
# session_id, source_file first
# DateTime, DateTime_raw early if present
# -----------------------------
all_columns = sorted(all_columns)

priority_cols = ["session_id", "source_file", "DateTime", "DateTime_raw"]
for col in priority_cols:
    if col in all_columns:
        all_columns.remove(col)

final_columns = [c for c in priority_cols if c in {"session_id", "source_file", "DateTime", "DateTime_raw"} and any(c in df.columns for df in dfs)] + all_columns

# -----------------------------
# Align and concatenate
# -----------------------------
aligned_dfs = [df.reindex(columns=final_columns) for df in dfs]
combined_df = pd.concat(aligned_dfs, ignore_index=True)

# -----------------------------
# Save
# -----------------------------
print("\n--- Summary ---")
print(f"Number of files: {len(csv_files)}")
print(f"Total rows: {len(combined_df)}")
print(f"Total columns: {len(combined_df.columns)}")

print("\nFirst 15 columns:")
print(combined_df.columns[:15].tolist())

print("\nSample merged DateTime values:")
if "DateTime" in combined_df.columns:
    print(combined_df["DateTime"].head(10).tolist())

# Save as CSV while preserving raw DateTime strings
combined_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"\nCombined dataset saved to: {OUTPUT_CSV}")

Found files:
01. Data_from_2021-11-26.csv
02. Data_from_2021-12-07.csv
03. Data_from_2021-12-17.csv
04. Data_from_2022-01-20.csv
05. Data_from_2022-01-21.csv
06. Data_from_2022-02-14.csv
07. Data_from_2022-02-16.csv
08. Data_from_2022-02-18.csv
09. Data_from_2022-02-22.csv
10. Data_from_2022-03-02.csv
11. Data_from_2022-03-15.csv
12. Data_from_2022-04-05.csv
13. Data_from_2022-04.csv

Data_from_2021-11-26.csv
  sample DateTime values: ['2021-11-26 07:15:36', '2021-11-26 07:15:38', '2021-11-26 07:15:40', '2021-11-26 07:15:41', '2021-11-26 07:15:42']
  contains seconds: True

Data_from_2021-12-07.csv
  sample DateTime values: ['2021-12-07 07:39:13', '2021-12-07 07:39:14', '2021-12-07 07:39:16', '2021-12-07 07:39:18', '2021-12-07 07:39:20']
  contains seconds: True

Data_from_2021-12-17.csv
  sample DateTime values: ['2021-12-17 14:00:01', '2021-12-17 14:00:03', '2021-12-17 14:00:05', '2021-12-17 14:00:07', '2021-12-17 14:00:09']
  contains seconds: True

Data_from_2022-01-20.csv
  sample

## Stage 1 — Shared 1 Hz operating-state representation

### 1.1 Build the raw 1 Hz dataset

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# RAW 1 Hz dataset builder from combined_dataset.csv
# - no scaling
# - no transformation
# - no windowing
# - valid contiguous runs preserved
# ============================================================

# -----------------------------
# Settings
# -----------------------------
DATA_DIR = Path("data")
INPUT_FILE = DATA_DIR / "combined_dataset.csv"
OUTPUT_FILE = DATA_DIR / "dataset_1Hz_raw.csv"

# Native data are mostly 1-3 s apart.
# Gap larger than this starts a new valid contiguous run.
GAP_THRESHOLD_SECONDS = 3
    
# Keep optional extended/context variables and labels?
KEEP_EXTENDED = True
KEEP_LABELS = True

# -----------------------------
# Variable groups
# -----------------------------
CORE_STATE_VARS = [
    "DriveStatus",
    "DoorStatusMain",
    "DoorStatusTooling",
    "CoolantFlow",
]

CORE_CONT_VARS = [
    "OverrideFeed",
    "FeedRate",
    "SpindleSpeed",
]

EXTENDED_VARS = [
    "ProgramStatus",
    "Warmup",
    "ToolNumber",
]

LABEL_VARS = [
    "Production",
    "Phase_compressed",
    "Phase",
]

OPTIONAL_KEEP_VARS = []

# -----------------------------
# Helpers
# -----------------------------
def try_cast_integerish(df, cols):
    """
    Convert float columns back to Int64 if values are integer-like.
    Keeps NA support.
    """
    for c in cols:
        if c in df.columns:
            s = df[c]
            non_null = s.dropna()
            if len(non_null) > 0 and np.all(np.isclose(non_null, np.round(non_null))):
                df[c] = np.round(df[c]).astype("Int64")
    return df

def build_run_ids(df):
    """
    New valid contiguous run starts when time gap > threshold
    within each (session_id, source_file).
    """
    df = df.sort_values(["session_id", "source_file", "DateTime"]).copy()

    gap_s = df.groupby(["session_id", "source_file"])["DateTime"].diff().dt.total_seconds()

    df["valid_run_id"] = (
        gap_s.gt(GAP_THRESHOLD_SECONDS)
             .fillna(False)
             .groupby([df["session_id"], df["source_file"]])
             .cumsum() + 1
    )

    return df

def process_single_run(run_df, session_id, source_file, run_id):
    """
    Build raw 1 Hz series for one valid contiguous run.
    """
    run_df = run_df.sort_values("DateTime").copy()
    run_df["tick"] = run_df["DateTime"].dt.floor("s")

    agg_map = {}

    state_vars = list(CORE_STATE_VARS)
    if KEEP_EXTENDED:
        state_vars += [c for c in EXTENDED_VARS if c in run_df.columns]
    if KEEP_LABELS:
        state_vars += [c for c in LABEL_VARS if c in run_df.columns]
    state_vars += [c for c in OPTIONAL_KEEP_VARS if c in run_df.columns]

    # State-like variables -> last within second
    for c in state_vars:
        if c in run_df.columns:
            agg_map[c] = "last"

    # Continuous process-intensity variables -> mean within second
    for c in CORE_CONT_VARS:
        if c in run_df.columns:
            agg_map[c] = "mean"

    tick_df = (
        run_df.groupby("tick", as_index=True)
              .agg(agg_map)
              .sort_index()
    )

    # Full 1 Hz grid for this contiguous run
    full_index = pd.date_range(
        start=tick_df.index.min(),
        end=tick_df.index.max(),
        freq="1s"
    )

    tick_df = tick_df.reindex(full_index)

    # Forward-fill within contiguous run
    tick_df[list(agg_map.keys())] = tick_df[list(agg_map.keys())].ffill()

    tick_df.index.name = "DateTime"
    tick_df = tick_df.reset_index()

    tick_df.insert(0, "valid_run_id", int(run_id))
    tick_df.insert(0, "source_file", source_file)
    tick_df.insert(0, "session_id", int(session_id))

    return tick_df

# -----------------------------
# Load combined dataset
# -----------------------------
df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")

# Clean column names
df.columns = [str(c).strip() for c in df.columns]

# Drop unnamed helper/index columns if present
drop_cols = [c for c in df.columns if c.lower().startswith("unnamed")]
if drop_cols:
    df = df.drop(columns=drop_cols)

# Required columns
required_cols = ["session_id", "source_file", "DateTime"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

# Parse DateTime
df["DateTime"] = pd.to_datetime(df["DateTime"], dayfirst=True, errors="coerce")
bad_dt = df["DateTime"].isna().sum()
if bad_dt > 0:
    print(f"[WARN] Invalid DateTime rows dropped: {bad_dt}")
    df = df.dropna(subset=["DateTime"]).copy()

# Keep only relevant columns if they exist
wanted_cols = (
    ["session_id", "source_file", "DateTime"] +
    [c for c in CORE_STATE_VARS if c in df.columns] +
    [c for c in CORE_CONT_VARS if c in df.columns] +
    ([c for c in EXTENDED_VARS if c in df.columns] if KEEP_EXTENDED else []) +
    ([c for c in LABEL_VARS if c in df.columns] if KEEP_LABELS else []) +
    [c for c in OPTIONAL_KEEP_VARS if c in df.columns]
)
df = df[wanted_cols].copy()

# Convert non-metadata columns to numeric where possible
for c in df.columns:
    if c not in ["session_id", "source_file", "DateTime"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Normalize session_id type
df["session_id"] = pd.to_numeric(df["session_id"], errors="coerce").astype("Int64")

# Sort and split into valid contiguous runs
df = build_run_ids(df)

# -----------------------------
# Build raw 1 Hz dataset
# -----------------------------
all_runs = []

for (session_id, source_file, run_id), run_df in df.groupby(
    ["session_id", "source_file", "valid_run_id"],
    sort=False
):
    out_run = process_single_run(
        run_df=run_df,
        session_id=session_id,
        source_file=source_file,
        run_id=run_id
    )
    all_runs.append(out_run)

raw_1hz = pd.concat(all_runs, ignore_index=True)

# Recast integer-like columns
integerish_cols = CORE_STATE_VARS.copy()
if KEEP_EXTENDED:
    integerish_cols += EXTENDED_VARS
if KEEP_LABELS:
    integerish_cols += LABEL_VARS
integerish_cols += OPTIONAL_KEEP_VARS

raw_1hz = try_cast_integerish(raw_1hz, integerish_cols)

# Final column order
ordered_cols = (
    ["session_id", "source_file", "valid_run_id", "DateTime"] +
    [c for c in CORE_STATE_VARS if c in raw_1hz.columns] +
    [c for c in CORE_CONT_VARS if c in raw_1hz.columns] +
    ([c for c in EXTENDED_VARS if c in raw_1hz.columns] if KEEP_EXTENDED else []) +
    ([c for c in LABEL_VARS if c in raw_1hz.columns] if KEEP_LABELS else []) +
    [c for c in OPTIONAL_KEEP_VARS if c in raw_1hz.columns]
)

raw_1hz = raw_1hz[ordered_cols]

# Save
raw_1hz.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

# -----------------------------
# Sanity checks
# -----------------------------
print("--- Summary ---")
print(f"Input rows total: {len(df):,}")
print(f"Raw 1 Hz rows total: {len(raw_1hz):,}")
print(f"Number of valid contiguous runs: {raw_1hz[['session_id', 'source_file', 'valid_run_id']].drop_duplicates().shape[0]:,}")
print(f"Saved to: {OUTPUT_FILE}")

print("\nColumns kept:")
print(raw_1hz.columns.tolist())

print("\nHead:")
print(raw_1hz.head())

C:\Users\Sinus\AppData\Local\Temp\ipykernel_21028\1690671494.py:165: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["DateTime"] = pd.to_datetime(df["DateTime"], dayfirst=True, errors="coerce")


--- Summary ---
Input rows total: 190,046
Raw 1 Hz rows total: 331,630
Number of valid contiguous runs: 331
Saved to: data\dataset_1Hz_raw.csv

Columns kept:
['session_id', 'source_file', 'valid_run_id', 'DateTime', 'DriveStatus', 'DoorStatusMain', 'DoorStatusTooling', 'CoolantFlow', 'OverrideFeed', 'FeedRate', 'SpindleSpeed', 'ProgramStatus', 'Warmup', 'ToolNumber', 'Production', 'Phase_compressed', 'Phase']

Head:
   session_id               source_file  valid_run_id            DateTime  \
0           1  Data_from_2021-11-26.csv             1 2021-11-26 07:15:36   
1           1  Data_from_2021-11-26.csv             1 2021-11-26 07:15:37   
2           1  Data_from_2021-11-26.csv             1 2021-11-26 07:15:38   
3           1  Data_from_2021-11-26.csv             1 2021-11-26 07:15:39   
4           1  Data_from_2021-11-26.csv             1 2021-11-26 07:15:40   

   DriveStatus  DoorStatusMain  DoorStatusTooling  CoolantFlow  OverrideFeed  \
0            1               0       

### 1.2 Core harmonization

Build the harmonized 1 Hz core representation used by selected downstream blocks.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

# ============================================================
# Stage 1 harmonization from raw 1 Hz dataset
# - only core input columns are kept
# - binary machine-state variables remain unchanged
# - continuous process-intensity variables are harmonized
# ============================================================

# -----------------------------
# Settings
# -----------------------------
DATA_DIR = Path("data")
INPUT_FILE = DATA_DIR / "dataset_1Hz_raw.csv"
OUTPUT_FILE = DATA_DIR / "dataset_1Hz_core_harmonized.csv"

# Keep raw continuous columns for audit/debug?
KEEP_RAW_CONTINUOUS = True

# Clipping percentiles
FEEDRATE_LOWER_Q = 0.005
FEEDRATE_UPPER_Q = 0.995
SPINDLE_UPPER_Q = 0.995

# -----------------------------
# Core input columns
# -----------------------------
META_COLS = ["session_id", "source_file", "valid_run_id", "DateTime"]

CORE_STATE_VARS = [
    "DriveStatus",
    "DoorStatusMain",
    "DoorStatusTooling",
    "CoolantFlow",
]

CORE_CONT_VARS = [
    "OverrideFeed",
    "FeedRate",
    "SpindleSpeed",
]

# -----------------------------
# Helpers
# -----------------------------
def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))

def ensure_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# -----------------------------
# Load raw 1 Hz dataset
# -----------------------------
df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
df.columns = [str(c).strip() for c in df.columns]

required_cols = META_COLS + CORE_STATE_VARS + CORE_CONT_VARS
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["DateTime"] = pd.to_datetime(df["DateTime"], errors="coerce")
df = ensure_numeric(df, CORE_STATE_VARS + CORE_CONT_VARS)

# Keep only core input + metadata
stage1 = df[META_COLS + CORE_STATE_VARS + CORE_CONT_VARS].copy()

# -----------------------------
# 1) Binary state variables
# Keep unchanged
# -----------------------------
for c in CORE_STATE_VARS:
    stage1[c] = stage1[c].astype("Int64")

# -----------------------------
# 2) Continuous variables: harmonization
# -----------------------------

# Optional backups of raw values
if KEEP_RAW_CONTINUOUS:
    for c in CORE_CONT_VARS:
        stage1[f"{c}_raw"] = stage1[c]

# ---- OverrideFeed
# bounded, usually less skewed -> robust scaling only
stage1["OverrideFeed_h"] = stage1["OverrideFeed"].astype(float)

# ---- FeedRate
# large range, may be negative -> clip + signed_log1p + robust scaling
feed_lo = stage1["FeedRate"].quantile(FEEDRATE_LOWER_Q)
feed_hi = stage1["FeedRate"].quantile(FEEDRATE_UPPER_Q)
stage1["FeedRate_h"] = stage1["FeedRate"].clip(lower=feed_lo, upper=feed_hi)
stage1["FeedRate_h"] = signed_log1p(stage1["FeedRate_h"])

# ---- SpindleSpeed
# nonnegative, often right-skewed -> clip + log1p + robust scaling
sp_hi = stage1["SpindleSpeed"].quantile(SPINDLE_UPPER_Q)
stage1["SpindleSpeed_h"] = stage1["SpindleSpeed"].clip(lower=0, upper=sp_hi)
stage1["SpindleSpeed_h"] = np.log1p(stage1["SpindleSpeed_h"])

# -----------------------------
# 3) Robust scaling of continuous harmonized columns
# Fit once globally to preserve comparability
# -----------------------------
harm_cols = ["OverrideFeed_h", "FeedRate_h", "SpindleSpeed_h"]

scaler = RobustScaler()
stage1[harm_cols] = scaler.fit_transform(stage1[harm_cols])

# -----------------------------
# 4) Build final harmonized dataset
# Replace original continuous names by harmonized versions
# -----------------------------
final_df = stage1[META_COLS + CORE_STATE_VARS].copy()

final_df["OverrideFeed"] = stage1["OverrideFeed_h"]
final_df["FeedRate"] = stage1["FeedRate_h"]
final_df["SpindleSpeed"] = stage1["SpindleSpeed_h"]

# Optional raw backups
if KEEP_RAW_CONTINUOUS:
    final_df["OverrideFeed_raw"] = stage1["OverrideFeed_raw"]
    final_df["FeedRate_raw"] = stage1["FeedRate_raw"]
    final_df["SpindleSpeed_raw"] = stage1["SpindleSpeed_raw"]

# -----------------------------
# Save
# -----------------------------
final_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

# -----------------------------
# Short report
# -----------------------------
print("--- Stage 1 core harmonization completed ---")
print(f"Input rows: {len(df):,}")
print(f"Output rows: {len(final_df):,}")
print(f"Saved to: {OUTPUT_FILE}")

print("\nCore state variables kept unchanged:")
print(CORE_STATE_VARS)

print("\nContinuous variables harmonized:")
print({
    "OverrideFeed": "robust scaling",
    "FeedRate": "clip -> signed_log1p -> robust scaling",
    "SpindleSpeed": "clip -> log1p -> robust scaling",
})

print("\nFinal columns:")
print(final_df.columns.tolist())

print("\nHead:")
print(final_df.head())

--- Stage 1 core harmonization completed ---
Input rows: 331,630
Output rows: 331,630
Saved to: data\dataset_1Hz_core_harmonized.csv

Core state variables kept unchanged:
['DriveStatus', 'DoorStatusMain', 'DoorStatusTooling', 'CoolantFlow']

Continuous variables harmonized:
{'OverrideFeed': 'robust scaling', 'FeedRate': 'clip -> signed_log1p -> robust scaling', 'SpindleSpeed': 'clip -> log1p -> robust scaling'}

Final columns:
['session_id', 'source_file', 'valid_run_id', 'DateTime', 'DriveStatus', 'DoorStatusMain', 'DoorStatusTooling', 'CoolantFlow', 'OverrideFeed', 'FeedRate', 'SpindleSpeed', 'OverrideFeed_raw', 'FeedRate_raw', 'SpindleSpeed_raw']

Head:
   session_id               source_file  valid_run_id            DateTime  \
0           1  Data_from_2021-11-26.csv             1 2021-11-26 07:15:36   
1           1  Data_from_2021-11-26.csv             1 2021-11-26 07:15:37   
2           1  Data_from_2021-11-26.csv             1 2021-11-26 07:15:38   
3           1  Data_from_20

## Stage 2 — Alternative segmentation designs

### 2.1 K-means screening

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# --------------------------------------------------
# Load harmonized Stage-1 core dataset
# --------------------------------------------------
DATA_PATH = "data/dataset_1Hz_raw.csv"

df = pd.read_csv(DATA_PATH)

# Core inputs for illustrative k-means
features = [
    "DriveStatus",
    "DoorStatusMain",
    "DoorStatusTooling",
    "CoolantFlow",
    "OverrideFeed",
    "FeedRate",
    "SpindleSpeed",
]

X = df[features].copy()

# Drop missing just in case
X = X.dropna().reset_index(drop=True)

print("Shape used for clustering:", X.shape)

# --------------------------------------------------
# K range
# --------------------------------------------------
k_values = list(range(5, 35))

results = []

# Silhouette can be expensive on large data
# Use a sample for speed while keeping CH and DB on full data
silhouette_sample_size = min(50000, len(X))

for k in k_values:
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20,
        max_iter=300
    )

    labels = km.fit_predict(X)

    sil = silhouette_score(
        X,
        labels,
        sample_size=silhouette_sample_size,
        random_state=42
    )
    ch = calinski_harabasz_score(X, labels)
    db = davies_bouldin_score(X, labels)

    results.append({
        "k": k,
        "silhouette": sil,
        "calinski_harabasz": ch,
        "davies_bouldin": db,
        "inertia": km.inertia_
    })

results_df = pd.DataFrame(results)

print(results_df)

# --------------------------------------------------
# Best values
# --------------------------------------------------
best_sil_k = results_df.loc[results_df["silhouette"].idxmax(), "k"]
best_ch_k = results_df.loc[results_df["calinski_harabasz"].idxmax(), "k"]
best_db_k = results_df.loc[results_df["davies_bouldin"].idxmin(), "k"]

print("\nBest by Silhouette:", best_sil_k)
print("Best by Calinski-Harabasz:", best_ch_k)
print("Best by Davies-Bouldin:", best_db_k)

# --------------------------------------------------
# Plots
# --------------------------------------------------
plt.figure(figsize=(8, 4))
plt.plot(results_df["k"], results_df["silhouette"], marker="o")
plt.xlabel("k")
plt.ylabel("Silhouette")
plt.title("K-means: Silhouette by k")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(results_df["k"], results_df["calinski_harabasz"], marker="o")
plt.xlabel("k")
plt.ylabel("Calinski-Harabasz")
plt.title("K-means: Calinski-Harabasz by k")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(results_df["k"], results_df["davies_bouldin"], marker="o")
plt.xlabel("k")
plt.ylabel("Davies-Bouldin")
plt.title("K-means: Davies-Bouldin by k")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(results_df["k"], results_df["inertia"], marker="o")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.title("K-means: Inertia by k")
plt.grid(True)
plt.show()

### 2.2 Windowed local representations

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# Windowed local representations from 1 Hz harmonized dataset
# - windows: 5 s, 10 s, 15 s
# - rolling within each valid contiguous run only
# - state vars -> occupancy proportion in window
# - continuous vars -> mean and std in window
# ============================================================

# -----------------------------
# Settings
# -----------------------------
DATA_DIR = Path("data")
INPUT_FILE = DATA_DIR / "dataset_1Hz_raw.csv"

WINDOW_SIZES = [5, 10, 15]

GROUP_COLS = ["session_id", "source_file", "valid_run_id"]
TIME_COL = "DateTime"

STATE_VARS = [
    "DriveStatus",
    "DoorStatusMain",
    "DoorStatusTooling",
    "CoolantFlow",
]

CONT_VARS = [
    "OverrideFeed",
    "FeedRate",
    "SpindleSpeed",
]

# -----------------------------
# Load input
# -----------------------------
df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
df.columns = [str(c).strip() for c in df.columns]
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")

required_cols = GROUP_COLS + [TIME_COL] + STATE_VARS + CONT_VARS
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.dropna(subset=[TIME_COL]).copy()

for c in STATE_VARS + CONT_VARS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.sort_values(GROUP_COLS + [TIME_COL]).reset_index(drop=True)

# -----------------------------
# Window builder
# -----------------------------
def build_windowed_dataset(df_in, window_size):
    parts = []

    for keys, g in df_in.groupby(GROUP_COLS, sort=False):
        g = g.sort_values(TIME_COL).reset_index(drop=True).copy()

        out = g[GROUP_COLS + [TIME_COL]].copy()
        out["window_size_s"] = window_size
        out["window_start"] = out[TIME_COL] - pd.to_timedelta(window_size - 1, unit="s")
        out["window_end"] = out[TIME_COL]

        # state vars -> occupancy proportion in window
        for c in STATE_VARS:
            out[f"{c}_occ"] = (
                g[c]
                .rolling(window=window_size, min_periods=window_size)
                .mean()
            )

        # continuous vars -> local mean and local std in window
        for c in CONT_VARS:
            out[f"{c}_mean"] = (
                g[c]
                .rolling(window=window_size, min_periods=window_size)
                .mean()
            )
            out[f"{c}_std"] = (
                g[c]
                .rolling(window=window_size, min_periods=window_size)
                .std(ddof=0)
            )

        # keep only complete windows
        first_feature_col = f"{STATE_VARS[0]}_occ"
        out = out.dropna(subset=[first_feature_col]).reset_index(drop=True)

        # std can be NaN only in pathological cases; replace with 0 if any appear
        std_cols = [f"{c}_std" for c in CONT_VARS if f"{c}_std" in out.columns]
        if std_cols:
            out[std_cols] = out[std_cols].fillna(0.0)

        parts.append(out)

    return pd.concat(parts, ignore_index=True)

# -----------------------------
# Build and save all window sizes
# -----------------------------
for w in WINDOW_SIZES:
    out_df = build_windowed_dataset(df, window_size=w)

    output_file = DATA_DIR / f"dataset_1Hz_windowed_{w}s.csv"
    out_df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print(f"Saved {w}s windowed dataset to: {output_file}")
    print(f"Rows: {len(out_df):,}")
    print(f"Columns: {len(out_df.columns)}")
    print("-" * 60)

print("Done.")

Saved 5s windowed dataset to: data\dataset_1Hz_windowed_5s.csv
Rows: 330,362
Columns: 17
------------------------------------------------------------
Saved 10s windowed dataset to: data\dataset_1Hz_windowed_10s.csv
Rows: 328,882
Columns: 17
------------------------------------------------------------
Saved 15s windowed dataset to: data\dataset_1Hz_windowed_15s.csv
Rows: 327,438
Columns: 17
------------------------------------------------------------
Done.


### 2.3 K-means model selection

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.preprocessing import RobustScaler

# ============================================================
# K-means screening for preprocessed windowed datasets
# - windows: 5 s, 10 s, 15 s
# - evaluate k = 3..35
# - use 35,000 random windows from each dataset
# - preprocessing:
#     * occupancy features: unchanged
#     * selected nonnegative continuous features: log1p
#     * signed FeedRate_mean: asinh
#     * all features: RobustScaler
# ============================================================

# -----------------------------
# Settings
# -----------------------------
DATA_DIR = Path("data")
OUT_DIR = DATA_DIR / "kmeans_screening_preprocessed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_SIZES = [5, 10, 15]
K_RANGE = range(3, 36)
SAMPLE_N = 35000
RANDOM_STATE = 42

META_COLS = [
    "session_id",
    "source_file",
    "valid_run_id",
    "DateTime",
    "window_size_s",
    "window_start",
    "window_end",
]

# Exact feature set expected from the window-construction step
FEATURE_COLS = [
    "DriveStatus_occ",
    "DoorStatusMain_occ",
    "DoorStatusTooling_occ",
    "CoolantFlow_occ",
    "OverrideFeed_mean",
    "OverrideFeed_std",
    "FeedRate_mean",
    "FeedRate_std",
    "SpindleSpeed_mean",
    "SpindleSpeed_std",
]

OCC_COLS = [
    "DriveStatus_occ",
    "DoorStatusMain_occ",
    "DoorStatusTooling_occ",
    "CoolantFlow_occ",
]

LOG1P_COLS = [
    "OverrideFeed_mean",
    "OverrideFeed_std",
    "FeedRate_std",
    "SpindleSpeed_mean",
    "SpindleSpeed_std",
]

ASINH_COLS = [
    "FeedRate_mean",
]

# -----------------------------
# Helpers
# -----------------------------
def preprocess_window_features(df_window: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    """
    Returns:
        X_proc_df: dataframe after transformations, before scaling
        X_scaled: numpy array after RobustScaler
    """
    X = df_window[FEATURE_COLS].copy()

    # numeric conversion
    for c in FEATURE_COLS:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    # drop incomplete rows
    X = X.dropna().reset_index(drop=True)

    # defensive cleaning for occupancy
    for c in OCC_COLS:
        X[c] = X[c].clip(lower=0.0, upper=1.0)

    # log1p for nonnegative skew-prone features
    for c in LOG1P_COLS:
        # small negative numerical artifacts are clipped to 0
        if (X[c] < 0).any():
            n_neg = int((X[c] < 0).sum())
            print(f"[WARN] {c}: found {n_neg} negative values before log1p; clipping to 0.")
            X[c] = X[c].clip(lower=0.0)
        X[c] = np.log1p(X[c])

    # asinh for signed feature
    for c in ASINH_COLS:
        X[c] = np.arcsinh(X[c])

    # robust scaling on all features
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X[FEATURE_COLS])

    X_proc_df = pd.DataFrame(X_scaled, columns=FEATURE_COLS)
    return X, X_proc_df


def rank_and_consensus(results_df: pd.DataFrame) -> pd.DataFrame:
    """
    Add per-index ranks and consensus rank sum.
    Lower consensus_rank_sum is better.
    """
    out = results_df.copy()
    out["rank_silhouette"] = out["silhouette"].rank(ascending=False, method="min")
    out["rank_calinski_harabasz"] = out["calinski_harabasz"].rank(ascending=False, method="min")
    out["rank_davies_bouldin"] = out["davies_bouldin"].rank(ascending=True, method="min")
    out["consensus_rank_sum"] = (
        out["rank_silhouette"]
        + out["rank_calinski_harabasz"]
        + out["rank_davies_bouldin"]
    )
    return out


def save_metric_plot(results_df: pd.DataFrame, x_col: str, y_col: str, title: str, out_file: Path):
    plt.figure(figsize=(8, 4))
    plt.plot(results_df[x_col], results_df[y_col], marker="o")
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(out_file, dpi=150)
    plt.close()


# -----------------------------
# Screening loop
# -----------------------------
all_results = []
summary_rows = []

for w in WINDOW_SIZES:
    file_path = DATA_DIR / f"dataset_1Hz_windowed_{w}s.csv"
    df = pd.read_csv(file_path, encoding="utf-8-sig")
    df.columns = [str(c).strip() for c in df.columns]

    missing = [c for c in FEATURE_COLS + META_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"{file_path.name}: missing required columns: {missing}")

    # keep only metadata + feature columns
    df = df[META_COLS + FEATURE_COLS].copy()

    # preprocess full usable matrix
    X_raw_transformed, X_scaled_df = preprocess_window_features(df)

    # sample 35k rows (or all if smaller) AFTER preprocessing prep and dropna
    n_sample = min(SAMPLE_N, len(X_scaled_df))
    sample_idx = X_scaled_df.sample(n=n_sample, random_state=RANDOM_STATE).index

    Xs = X_scaled_df.loc[sample_idx].reset_index(drop=True)
    Xs_np = Xs.to_numpy(dtype=float)

    # save sampled preprocessed matrix for audit / reproducibility
    sample_out = OUT_DIR / f"window_{w}s_preprocessed_sample_{n_sample}.csv"
    Xs.to_csv(sample_out, index=False, encoding="utf-8-sig")

    print(f"\n=== Window {w}s ===")
    print(f"Input rows:           {len(df):,}")
    print(f"Usable rows:          {len(X_scaled_df):,}")
    print(f"Sample rows:          {len(Xs):,}")
    print(f"Features used:        {len(FEATURE_COLS)}")
    print("Feature order:")
    print(FEATURE_COLS)
    print(f"Saved sample to:      {sample_out}")

    window_results = []

    for k in K_RANGE:
        km = KMeans(
            n_clusters=k,
            random_state=RANDOM_STATE,
            n_init=20,
            max_iter=300
        )

        labels = km.fit_predict(Xs_np)

        sil = silhouette_score(Xs_np, labels)
        ch = calinski_harabasz_score(Xs_np, labels)
        db = davies_bouldin_score(Xs_np, labels)

        row = {
            "window_s": w,
            "sample_n": len(Xs_np),
            "k": k,
            "silhouette": sil,
            "calinski_harabasz": ch,
            "davies_bouldin": db,
            "inertia": km.inertia_,
        }
        window_results.append(row)
        all_results.append(row)

    results_df = pd.DataFrame(window_results)
    results_df = rank_and_consensus(results_df)

    # save per-window results
    per_window_out = OUT_DIR / f"kmeans_screening_window_{w}s.csv"
    results_df.to_csv(per_window_out, index=False, encoding="utf-8-sig")

    # console summary
    print("\nTop by silhouette:")
    print(
        results_df.sort_values("silhouette", ascending=False)
        .head(5)[["k", "silhouette", "calinski_harabasz", "davies_bouldin", "consensus_rank_sum"]]
    )

    print("\nTop by Calinski-Harabasz:")
    print(
        results_df.sort_values("calinski_harabasz", ascending=False)
        .head(5)[["k", "silhouette", "calinski_harabasz", "davies_bouldin", "consensus_rank_sum"]]
    )

    print("\nTop by Davies-Bouldin (lower is better):")
    print(
        results_df.sort_values("davies_bouldin", ascending=True)
        .head(5)[["k", "silhouette", "calinski_harabasz", "davies_bouldin", "consensus_rank_sum"]]
    )

    print("\nTop by consensus rank sum (lower is better):")
    print(
        results_df.sort_values(["consensus_rank_sum", "davies_bouldin", "silhouette"], ascending=[True, True, False])
        .head(5)[["k", "silhouette", "calinski_harabasz", "davies_bouldin", "consensus_rank_sum"]]
    )

    # save plots
    save_metric_plot(
        results_df, "k", "silhouette",
        f"{w}s window: Silhouette (preprocessed)",
        OUT_DIR / f"window_{w}s_silhouette.png"
    )

    save_metric_plot(
        results_df, "k", "calinski_harabasz",
        f"{w}s window: Calinski-Harabasz (preprocessed)",
        OUT_DIR / f"window_{w}s_calinski_harabasz.png"
    )

    save_metric_plot(
        results_df, "k", "davies_bouldin",
        f"{w}s window: Davies-Bouldin (preprocessed)",
        OUT_DIR / f"window_{w}s_davies_bouldin.png"
    )

    save_metric_plot(
        results_df, "k", "inertia",
        f"{w}s window: Inertia (preprocessed)",
        OUT_DIR / f"window_{w}s_inertia.png"
    )

    # summary row
    best_sil_k = int(results_df.loc[results_df["silhouette"].idxmax(), "k"])
    best_ch_k = int(results_df.loc[results_df["calinski_harabasz"].idxmax(), "k"])
    best_db_k = int(results_df.loc[results_df["davies_bouldin"].idxmin(), "k"])
    best_consensus_k = int(
        results_df.sort_values(
            ["consensus_rank_sum", "davies_bouldin", "silhouette"],
            ascending=[True, True, False]
        ).iloc[0]["k"]
    )

    summary_rows.append({
        "window_s": w,
        "best_k_silhouette": best_sil_k,
        "best_k_calinski_harabasz": best_ch_k,
        "best_k_davies_bouldin": best_db_k,
        "best_k_consensus": best_consensus_k,
        "sample_n": len(Xs_np),
        "results_file": per_window_out.name,
    })

# -----------------------------
# Save combined results
# -----------------------------
all_results_df = pd.DataFrame(all_results)
all_results_df = rank_and_consensus(all_results_df)

combined_out = OUT_DIR / "kmeans_window_screening_3_to_35_preprocessed.csv"
all_results_df.to_csv(combined_out, index=False, encoding="utf-8-sig")

summary_df = pd.DataFrame(summary_rows)
summary_out = OUT_DIR / "kmeans_window_screening_summary_preprocessed.csv"
summary_df.to_csv(summary_out, index=False, encoding="utf-8-sig")

print("\n============================================================")
print(f"Saved combined results to: {combined_out}")
print(f"Saved summary to:          {summary_out}")
print(f"All per-window files are in: {OUT_DIR.resolve()}")

print("\n=== Summary of best k by window ===")
print(summary_df)


=== Window 5s ===
Input rows:           330,362
Usable rows:          330,362
Sample rows:          35,000
Features used:        10
Feature order:
['DriveStatus_occ', 'DoorStatusMain_occ', 'DoorStatusTooling_occ', 'CoolantFlow_occ', 'OverrideFeed_mean', 'OverrideFeed_std', 'FeedRate_mean', 'FeedRate_std', 'SpindleSpeed_mean', 'SpindleSpeed_std']
Saved sample to:      data\kmeans_screening_preprocessed\window_5s_preprocessed_sample_35000.csv

Top by silhouette:
     k  silhouette  calinski_harabasz  davies_bouldin  consensus_rank_sum
0    3    0.856039      269671.596472        0.208262                25.0
1    4    0.681519      322665.777283        0.453218                 5.0
31  34    0.623106      269914.416322        0.792121                57.0
32  35    0.616479      272742.006770        0.724604                36.0
30  33    0.610031      269954.100573        0.777208                55.0

Top by Calinski-Harabasz:
   k  silhouette  calinski_harabasz  davies_bouldin  consensus_

### 2.4 Final K-means fits

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler

# ============================================================
# Final K-means fits for selected window sizes and K values
# - uses the same preprocessing pipeline as the screening code
# - fits on the full usable dataset for each window size
# - creates one new file per window size
# - keeps all original rows and appends outputs in separate columns
# ============================================================

# -----------------------------
# Settings
# -----------------------------
DATA_DIR = Path("data")

WINDOW_CONFIGS = {
    5: [4, 7],
    10: [4, 10],
    15: [4, 9],
}

RANDOM_STATE = 42
N_INIT = 20
MAX_ITER = 300

FEATURE_COLS = [
    "DriveStatus_occ",
    "DoorStatusMain_occ",
    "DoorStatusTooling_occ",
    "CoolantFlow_occ",
    "OverrideFeed_mean",
    "OverrideFeed_std",
    "FeedRate_mean",
    "FeedRate_std",
    "SpindleSpeed_mean",
    "SpindleSpeed_std",
]

OCC_COLS = [
    "DriveStatus_occ",
    "DoorStatusMain_occ",
    "DoorStatusTooling_occ",
    "CoolantFlow_occ",
]

LOG1P_COLS = [
    "OverrideFeed_mean",
    "OverrideFeed_std",
    "FeedRate_std",
    "SpindleSpeed_mean",
    "SpindleSpeed_std",
]

ASINH_COLS = [
    "FeedRate_mean",
]

ACTIVITY_SCORE_COLS = [
    "OverrideFeed_mean",
    "FeedRate_mean",
    "SpindleSpeed_mean",
]


# -----------------------------
# Helpers
# -----------------------------
def preprocess_window_features(df_window: pd.DataFrame):
    """
    Returns:
        df_full            : original dataframe with numeric-converted feature columns
        keep_mask          : boolean mask of rows usable for modeling
        modeled_df         : original rows retained for modeling (aligned row-for-row)
        X_transformed_df   : transformed features before scaling
        X_scaled_df        : transformed + robust-scaled features
    """
    missing = [c for c in FEATURE_COLS if c not in df_window.columns]
    if missing:
        raise ValueError(f"Missing required feature columns: {missing}")

    df_full = df_window.copy()

    # numeric conversion for features
    for c in FEATURE_COLS:
        df_full[c] = pd.to_numeric(df_full[c], errors="coerce")

    # rows usable for modeling
    keep_mask = df_full[FEATURE_COLS].notna().all(axis=1)
    modeled_df = df_full.loc[keep_mask].reset_index(drop=True)

    X = modeled_df[FEATURE_COLS].copy()

    # occupancy stays in [0, 1]
    for c in OCC_COLS:
        X[c] = X[c].clip(lower=0.0, upper=1.0)

    # log1p for nonnegative skew-prone variables
    for c in LOG1P_COLS:
        if (X[c] < 0).any():
            n_neg = int((X[c] < 0).sum())
            print(f"[WARN] {c}: found {n_neg} negative values before log1p; clipping to 0.")
            X[c] = X[c].clip(lower=0.0)
        X[c] = np.log1p(X[c])

    # asinh for signed feature
    for c in ASINH_COLS:
        X[c] = np.arcsinh(X[c])

    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X[FEATURE_COLS])

    X_transformed_df = X.copy()
    X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURE_COLS)

    return df_full, keep_mask, modeled_df, X_transformed_df, X_scaled_df


def relabel_by_activity(labels: np.ndarray, modeled_df: pd.DataFrame):
    """
    Reorder cluster labels by increasing activity score for readability.
    Activity score is based on cluster means of process-intensity variables.
    """
    tmp = modeled_df.copy()
    tmp["_label_raw"] = labels

    prof = tmp.groupby("_label_raw")[ACTIVITY_SCORE_COLS].mean().fillna(0.0)
    score = (
        prof["OverrideFeed_mean"].rank(method="dense")
        + prof["FeedRate_mean"].rank(method="dense")
        + prof["SpindleSpeed_mean"].rank(method="dense")
    )

    ordered_labels = score.sort_values().index.tolist()
    mapping = {old: new for new, old in enumerate(ordered_labels)}
    relabeled = np.array([mapping[x] for x in labels], dtype=int)
    return relabeled, mapping


# -----------------------------
# Main loop
# -----------------------------
for window_s, k_list in WINDOW_CONFIGS.items():
    input_file = DATA_DIR / f"dataset_1Hz_windowed_{window_s}s.csv"
    df = pd.read_csv(input_file, encoding="utf-8-sig")
    df.columns = [str(c).strip() for c in df.columns]

    print("\n============================================================")
    print(f"WINDOW {window_s}s")
    print(f"Loaded: {input_file}")
    print(f"Raw rows: {len(df):,}")

    df_full, keep_mask, modeled_df, X_transformed_df, X_scaled_df = preprocess_window_features(df)

    print(f"Usable rows after preprocessing: {len(modeled_df):,}")
    print(f"Dropped rows (missing feature values): {(~keep_mask).sum():,}")
    print(f"Feature count: {len(FEATURE_COLS)}")
    print("Fitting selected K values:", k_list)

    # start from the FULL dataframe so original row count is preserved
    out_df = df_full.copy()
    out_df["kmeans_usable_row"] = keep_mask.astype(int)

    # audit columns for transformed/scaled features
    for c in FEATURE_COLS:
        out_df[f"prep_{c}"] = np.nan
        out_df[f"scaled_{c}"] = np.nan
        out_df.loc[keep_mask, f"prep_{c}"] = X_transformed_df[c].values
        out_df.loc[keep_mask, f"scaled_{c}"] = X_scaled_df[c].values

    X_np = X_scaled_df.to_numpy(dtype=float)

    for k in k_list:
        km = KMeans(
            n_clusters=k,
            random_state=RANDOM_STATE,
            n_init=N_INIT,
            max_iter=MAX_ITER,
        )

        raw_labels = km.fit_predict(X_np)
        ordered_labels, mapping = relabel_by_activity(raw_labels, modeled_df)
        dists = km.transform(X_np)
        min_dist = dists.min(axis=1)

        out_df[f"kmeans_raw_k{k}"] = np.nan
        out_df[f"kmeans_state_k{k}"] = np.nan
        out_df[f"kmeans_min_dist_k{k}"] = np.nan
        out_df[f"kmeans_used_k{k}"] = 0

        out_df.loc[keep_mask, f"kmeans_raw_k{k}"] = raw_labels.astype(int)
        out_df.loc[keep_mask, f"kmeans_state_k{k}"] = ordered_labels.astype(int)
        out_df.loc[keep_mask, f"kmeans_min_dist_k{k}"] = min_dist.astype(float)
        out_df.loc[keep_mask, f"kmeans_used_k{k}"] = 1

        print(f"  K={k}: done | inertia={km.inertia_:.4f}")

    output_file = DATA_DIR / f"dataset_1Hz_windowed_{window_s}s_kmeans_selected.csv"
    out_df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print(f"Saved: {output_file}")
    print(f"Columns: {len(out_df.columns)}")
    print(f"Rows:    {len(out_df):,}")

print("\nDone.")


WINDOW 5s
Loaded: data\dataset_1Hz_windowed_5s.csv
Raw rows: 330,362
Usable rows after preprocessing: 330,362
Dropped rows (missing feature values): 0
Feature count: 10
Fitting selected K values: [4, 7]
  K=4: done | inertia=1851427.6297
  K=7: done | inertia=993338.6136
Saved: data\dataset_1Hz_windowed_5s_kmeans_selected.csv
Columns: 46
Rows:    330,362

WINDOW 10s
Loaded: data\dataset_1Hz_windowed_10s.csv
Raw rows: 328,882
Usable rows after preprocessing: 328,882
Dropped rows (missing feature values): 0
Feature count: 10
Fitting selected K values: [4, 10]
  K=4: done | inertia=1923089.3900
  K=10: done | inertia=647921.2260
Saved: data\dataset_1Hz_windowed_10s_kmeans_selected.csv
Columns: 46
Rows:    328,882

WINDOW 15s
Loaded: data\dataset_1Hz_windowed_15s.csv
Raw rows: 327,438
Usable rows after preprocessing: 327,438
Dropped rows (missing feature values): 0
Feature count: 10
Fitting selected K values: [4, 9]
  K=4: done | inertia=1863906.9618
  K=9: done | inertia=713048.3838
Save

### 2.5 K-means article-ready summaries

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# Article-ready summaries from final K-means outputs
# - CSV only
# ============================================================

DATA_DIR = Path("data")
OUT_DIR = DATA_DIR / "kmeans_article_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_CONFIGS = {
    5: [4, 7],
    10: [4, 10],
    15: [4, 9],
}

INPUT_TEMPLATE = "dataset_1Hz_windowed_{window_s}s_kmeans_selected.csv"

GROUP_COLS = ["session_id", "source_file", "valid_run_id"]

FEATURE_COLS = [
    "DriveStatus_occ",
    "DoorStatusMain_occ",
    "DoorStatusTooling_occ",
    "CoolantFlow_occ",
    "OverrideFeed_mean",
    "OverrideFeed_std",
    "FeedRate_mean",
    "FeedRate_std",
    "SpindleSpeed_mean",
    "SpindleSpeed_std",
]

ACTIVITY_COLS = [
    "OverrideFeed_mean",
    "FeedRate_mean",
    "SpindleSpeed_mean",
]

all_overview = []
all_state_profiles = []
all_feature_means = []
all_feature_medians = []
all_distance_summary = []

def entropy_from_counts(counts):
    p = counts / counts.sum()
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())

for window_s, k_list in WINDOW_CONFIGS.items():
    input_file = DATA_DIR / INPUT_TEMPLATE.format(window_s=window_s)
    df = pd.read_csv(input_file, encoding="utf-8-sig")
    df.columns = [str(c).strip() for c in df.columns]

    print(f"\nLoaded: {input_file} | rows={len(df):,}")

    for k in k_list:
        state_col = f"kmeans_state_k{k}"
        raw_col = f"kmeans_raw_k{k}"
        dist_col = f"kmeans_min_dist_k{k}"
        used_col = f"kmeans_used_k{k}"

        required = FEATURE_COLS + [state_col, raw_col, dist_col, used_col] + [c for c in GROUP_COLS if c in df.columns]
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise ValueError(f"{input_file.name} missing columns for K={k}: {missing}")

        d = df[df[used_col] == 1].copy()
        if d.empty:
            raise ValueError(f"{input_file.name} | K={k}: no usable rows found.")

        d[state_col] = pd.to_numeric(d[state_col], errors="coerce").astype("Int64")
        d[raw_col] = pd.to_numeric(d[raw_col], errors="coerce").astype("Int64")
        d[dist_col] = pd.to_numeric(d[dist_col], errors="coerce")

        # -----------------------------
        # Overview for this window x K
        # -----------------------------
        state_counts = d[state_col].value_counts().sort_index()
        shares = state_counts / len(d)

        overview_row = {
            "window_s": window_s,
            "k": k,
            "n_rows": len(d),
            "n_states_found": int(state_counts.shape[0]),
            "smallest_state_size": int(state_counts.min()),
            "largest_state_size": int(state_counts.max()),
            "median_state_size": float(state_counts.median()),
            "smallest_state_share": float(shares.min()),
            "largest_state_share": float(shares.max()),
            "state_entropy": entropy_from_counts(state_counts),
            "mean_min_dist": float(d[dist_col].mean()),
            "median_min_dist": float(d[dist_col].median()),
            "std_min_dist": float(d[dist_col].std(ddof=0)),
            "n_sessions": int(d["session_id"].nunique()) if "session_id" in d.columns else np.nan,
            "n_runs": int(d["valid_run_id"].nunique()) if "valid_run_id" in d.columns else np.nan,
            "source_file_nunique": int(d["source_file"].nunique()) if "source_file" in d.columns else np.nan,
        }
        all_overview.append(overview_row)

        # -----------------------------
        # State-level core profile
        # -----------------------------
        agg_dict = {
            state_col: (state_col, "first"),
            "n_rows": (state_col, "size"),
            "share": (state_col, lambda x: len(x) / len(d)),
            "mean_min_dist": (dist_col, "mean"),
            "median_min_dist": (dist_col, "median"),
            "std_min_dist": (dist_col, lambda x: x.std(ddof=0)),
        }

        if "session_id" in d.columns:
            agg_dict["n_sessions"] = ("session_id", "nunique")
        if "valid_run_id" in d.columns:
            agg_dict["n_runs"] = ("valid_run_id", "nunique")
        if "source_file" in d.columns:
            agg_dict["n_source_files"] = ("source_file", "nunique")

        for c in FEATURE_COLS:
            agg_dict[f"{c}_mean"] = (c, "mean")

        state_profile = (
            d.groupby(state_col, dropna=False)
             .agg(**agg_dict)
             .reset_index(drop=True)
             .sort_values(by=state_col)
             .reset_index(drop=True)
        )

        state_profile["activity_mean_score"] = (
            state_profile["OverrideFeed_mean_mean"]
            + state_profile["FeedRate_mean_mean"]
            + state_profile["SpindleSpeed_mean_mean"]
        )

        state_profile.insert(0, "k", k)
        state_profile.insert(0, "window_s", window_s)
        all_state_profiles.append(state_profile)

        # -----------------------------
        # Feature means by state
        # -----------------------------
        means_df = (
            d.groupby(state_col)[FEATURE_COLS]
             .mean()
             .reset_index()
             .rename(columns={state_col: "state"})
             .sort_values("state")
             .reset_index(drop=True)
        )
        means_df.insert(0, "k", k)
        means_df.insert(0, "window_s", window_s)
        all_feature_means.append(means_df)

        # -----------------------------
        # Feature medians by state
        # -----------------------------
        medians_df = (
            d.groupby(state_col)[FEATURE_COLS]
             .median()
             .reset_index()
             .rename(columns={state_col: "state"})
             .sort_values("state")
             .reset_index(drop=True)
        )
        medians_df.insert(0, "k", k)
        medians_df.insert(0, "window_s", window_s)
        all_feature_medians.append(medians_df)

        # -----------------------------
        # Distance summary by state
        # -----------------------------
        dist_df = (
            d.groupby(state_col)[dist_col]
             .agg(["count", "mean", "median", "std", "min", "max"])
             .reset_index()
             .rename(columns={
                 state_col: "state",
                 "count": "n_rows",
                 "mean": "mean_min_dist",
                 "median": "median_min_dist",
                 "std": "std_min_dist",
                 "min": "min_min_dist",
                 "max": "max_min_dist",
             })
             .sort_values("state")
             .reset_index(drop=True)
        )
        dist_df.insert(0, "k", k)
        dist_df.insert(0, "window_s", window_s)
        all_distance_summary.append(dist_df)

        print(f"  window={window_s}s | K={k}: processed")

# ============================================================
# Combine all outputs
# ============================================================

overview_df = pd.DataFrame(all_overview).sort_values(["window_s", "k"]).reset_index(drop=True)
state_profiles_df = pd.concat(all_state_profiles, ignore_index=True)
feature_means_df = pd.concat(all_feature_means, ignore_index=True)
feature_medians_df = pd.concat(all_feature_medians, ignore_index=True)
distance_summary_df = pd.concat(all_distance_summary, ignore_index=True)

# ============================================================
# Save CSVs
# ============================================================

overview_file = OUT_DIR / "kmeans_article_overview.csv"
state_profiles_file = OUT_DIR / "kmeans_article_state_profiles.csv"
feature_means_file = OUT_DIR / "kmeans_article_feature_means_by_state.csv"
feature_medians_file = OUT_DIR / "kmeans_article_feature_medians_by_state.csv"
distance_summary_file = OUT_DIR / "kmeans_article_distance_summary.csv"

overview_df.to_csv(overview_file, index=False, encoding="utf-8-sig")
state_profiles_df.to_csv(state_profiles_file, index=False, encoding="utf-8-sig")
feature_means_df.to_csv(feature_means_file, index=False, encoding="utf-8-sig")
feature_medians_df.to_csv(feature_medians_file, index=False, encoding="utf-8-sig")
distance_summary_df.to_csv(distance_summary_file, index=False, encoding="utf-8-sig")

print("\nSaved files:")
print(" -", overview_file)
print(" -", state_profiles_file)
print(" -", feature_means_file)
print(" -", feature_medians_file)
print(" -", distance_summary_file)

print("\n=== Overview ===")
print(overview_df)


Loaded: data\dataset_1Hz_windowed_5s_kmeans_selected.csv | rows=330,362
  window=5s | K=4: processed
  window=5s | K=7: processed

Loaded: data\dataset_1Hz_windowed_10s_kmeans_selected.csv | rows=328,882
  window=10s | K=4: processed
  window=10s | K=10: processed

Loaded: data\dataset_1Hz_windowed_15s_kmeans_selected.csv | rows=327,438
  window=15s | K=4: processed
  window=15s | K=9: processed

Saved files:
 - data\kmeans_article_results\kmeans_article_overview.csv
 - data\kmeans_article_results\kmeans_article_state_profiles.csv
 - data\kmeans_article_results\kmeans_article_feature_means_by_state.csv
 - data\kmeans_article_results\kmeans_article_feature_medians_by_state.csv
 - data\kmeans_article_results\kmeans_article_distance_summary.csv

=== Overview ===
   window_s   k  n_rows  n_states_found  smallest_state_size  \
0         5   4  330362               4                16189   
1         5   7  330362               7                  507   
2        10   4  328882              

### 2.6 HDBSCAN screening

In [1]:
# =========================
# HDBSCAN Stage 2 screening on preprocessed windowed data
# - windows: 5 s, 10 s, 15 s
# - NO UMAP
# - NO Stage 3
# - same preprocessing logic as in k-means screening
# =========================

import sys
import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# --- auto-install hdbscan if missing ---
try:
    import hdbscan
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "hdbscan"])
    import hdbscan

warnings.filterwarnings("ignore", category=FutureWarning)

# =========================================================
# CONFIG
# =========================================================

DATA_DIR = Path("data")
OUT_DIR = DATA_DIR / "hdbscan_screening_preprocessed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_SIZES = [15] #[5, 10, 15]

# Set to None to use all usable rows.
# If needed for speed, set e.g. SAMPLE_N = 50000
SAMPLE_N = None
RANDOM_STATE = 42

META_COLS = [
    "session_id",
    "source_file",
    "valid_run_id",
    "DateTime",
    "window_size_s",
    "window_start",
    "window_end",
]

FEATURE_COLS = [
    "DriveStatus_occ",
    "DoorStatusMain_occ",
    "DoorStatusTooling_occ",
    "CoolantFlow_occ",
    "OverrideFeed_mean",
    "OverrideFeed_std",
    "FeedRate_mean",
    "FeedRate_std",
    "SpindleSpeed_mean",
    "SpindleSpeed_std",
]

OCC_COLS = [
    "DriveStatus_occ",
    "DoorStatusMain_occ",
    "DoorStatusTooling_occ",
    "CoolantFlow_occ",
]

LOG1P_COLS = [
    "OverrideFeed_mean",
    "OverrideFeed_std",
    "FeedRate_std",
    "SpindleSpeed_mean",
    "SpindleSpeed_std",
]

ASINH_COLS = [
    "FeedRate_mean",
]

PARAM_GRID = [
    (1500, 50),
    (2000, 50),
    (3000, 50),
    (5000, 50),
    (1500, 100),
    (2000, 100),
    (3000, 100),
    (5000, 100),
    (1500, 200),
    (2000, 200),
    (3000, 200),
    (5000, 200),
]

CLUSTER_SELECTION_METHOD = "eom"
SAVE_LABEL_FILES = True
SAVE_PREPROCESSED_SAMPLE = True

SMALL_CLUSTER_SHARE_THRESHOLD = 0.01   # 1% of all modeled windows

# =========================================================
# HELPERS
# =========================================================

def preprocess_window_features(df_window: pd.DataFrame):
    """
    Returns:
        meta_df: metadata rows aligned with modeled rows
        X_transformed_df: feature dataframe after signal-specific transforms, before scaling
        X_scaled_df: feature dataframe after RobustScaler
    """
    missing = [c for c in META_COLS + FEATURE_COLS if c not in df_window.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    meta_df = df_window[META_COLS].copy()
    X = df_window[FEATURE_COLS].copy()

    # numeric conversion
    for c in FEATURE_COLS:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    # drop incomplete rows
    keep_mask = X.notna().all(axis=1)
    meta_df = meta_df.loc[keep_mask].reset_index(drop=True)
    X = X.loc[keep_mask].reset_index(drop=True)

    # defensive cleanup for occupancy features
    for c in OCC_COLS:
        X[c] = X[c].clip(lower=0.0, upper=1.0)

    # log1p for nonnegative skew-prone features
    for c in LOG1P_COLS:
        if (X[c] < 0).any():
            n_neg = int((X[c] < 0).sum())
            print(f"[WARN] {c}: found {n_neg} negative values before log1p; clipping to 0.")
            X[c] = X[c].clip(lower=0.0)
        X[c] = np.log1p(X[c])

    # asinh for signed feature
    for c in ASINH_COLS:
        X[c] = np.arcsinh(X[c])

    # robust scaling on all features
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X[FEATURE_COLS])

    X_transformed_df = X.copy()
    X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURE_COLS)

    return meta_df, X_transformed_df, X_scaled_df


def maybe_sample(meta_df: pd.DataFrame, X_scaled_df: pd.DataFrame, sample_n=None, random_state=42):
    """
    Keeps metadata aligned with sampled rows.
    """
    if sample_n is None or len(X_scaled_df) <= sample_n:
        return meta_df.reset_index(drop=True), X_scaled_df.reset_index(drop=True)

    idx = X_scaled_df.sample(n=sample_n, random_state=random_state).index
    return (
        meta_df.loc[idx].reset_index(drop=True),
        X_scaled_df.loc[idx].reset_index(drop=True),
    )


def safe_internal_scores(Xm, labels):
    """
    Internal validity only on non-noise points.
    Secondary diagnostics only.
    """
    mask = labels != -1
    unique = np.unique(labels[mask])

    if mask.sum() < 3 or len(unique) < 2:
        return np.nan, np.nan, np.nan

    X_non = Xm[mask]
    y_non = labels[mask]

    try:
        sil = silhouette_score(X_non, y_non)
    except Exception:
        sil = np.nan

    try:
        db = davies_bouldin_score(X_non, y_non)
    except Exception:
        db = np.nan

    try:
        ch = calinski_harabasz_score(X_non, y_non)
    except Exception:
        ch = np.nan

    return sil, db, ch


def cluster_size_stats(labels, small_share_threshold=0.01):
    """
    Counts over non-noise clusters.
    Shares are computed relative to all modeled windows.
    """
    s = pd.Series(labels)
    s = s[s != -1]

    if s.empty:
        return {
            "largest_cluster_share": np.nan,
            "median_cluster_share": np.nan,
            "min_cluster_share": np.nan,
            "n_small_clusters": 0,
            "smallest_cluster_size": 0,
            "largest_cluster_size": 0,
        }

    counts = s.value_counts().sort_values(ascending=False)
    shares = counts / len(labels)

    return {
        "largest_cluster_share": shares.iloc[0],
        "median_cluster_share": shares.median(),
        "min_cluster_share": shares.min(),
        "n_small_clusters": int((shares < small_share_threshold).sum()),
        "smallest_cluster_size": int(counts.min()),
        "largest_cluster_size": int(counts.max()),
    }


# =========================================================
# SCREENING LOOP
# =========================================================

all_results = []
summary_rows = []

for w in WINDOW_SIZES:
    input_csv = DATA_DIR / f"dataset_1Hz_windowed_{w}s.csv"
    df = pd.read_csv(input_csv, encoding="utf-8-sig")
    df.columns = [str(c).strip() for c in df.columns]

    print(f"\n=========================================================")
    print(f"WINDOW {w}s")
    print(f"Loaded: {input_csv}")
    print(f"Raw shape: {df.shape}")

    meta_df, X_transformed_df, X_scaled_df = preprocess_window_features(df)
    meta_model_df, X_model_df = maybe_sample(
        meta_df, X_scaled_df,
        sample_n=SAMPLE_N,
        random_state=RANDOM_STATE
    )

    X_model = X_model_df.to_numpy(dtype=float)

    print(f"Usable rows after preprocessing: {len(X_scaled_df):,}")
    print(f"Rows used for screening:         {len(X_model_df):,}")
    print(f"Feature count:                  {len(FEATURE_COLS)}")
    print("Feature order:")
    print(FEATURE_COLS)

    if SAVE_PREPROCESSED_SAMPLE:
        sample_out = OUT_DIR / f"window_{w}s_preprocessed_features.csv"
        pd.concat([meta_model_df, X_model_df], axis=1).to_csv(sample_out, index=False, encoding="utf-8-sig")
        print(f"Saved preprocessed feature matrix to: {sample_out}")

    window_results = []

    for min_cluster_size, min_samples in PARAM_GRID:
        print(f"\nRunning HDBSCAN | window={w}s | min_cluster_size={min_cluster_size} | min_samples={min_samples}")

        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            cluster_selection_method=CLUSTER_SELECTION_METHOD,
            prediction_data=False
        )

        labels = clusterer.fit_predict(X_model)

        n_noise = int((labels == -1).sum())
        noise_share = n_noise / len(labels)

        non_noise_labels = np.unique(labels[labels != -1])
        n_clusters = len(non_noise_labels)

        if hasattr(clusterer, "cluster_persistence_") and len(clusterer.cluster_persistence_) > 0:
            mean_persistence = float(np.mean(clusterer.cluster_persistence_))
            median_persistence = float(np.median(clusterer.cluster_persistence_))
            min_persistence = float(np.min(clusterer.cluster_persistence_))
            max_persistence = float(np.max(clusterer.cluster_persistence_))
        else:
            mean_persistence = np.nan
            median_persistence = np.nan
            min_persistence = np.nan
            max_persistence = np.nan

        if hasattr(clusterer, "probabilities_") and len(clusterer.probabilities_) == len(labels):
            probs = clusterer.probabilities_
            mean_membership_prob = float(np.mean(probs[labels != -1])) if n_clusters > 0 else np.nan
            median_membership_prob = float(np.median(probs[labels != -1])) if n_clusters > 0 else np.nan
        else:
            probs = np.full(len(labels), np.nan)
            mean_membership_prob = np.nan
            median_membership_prob = np.nan

        sil, dbi, ch = safe_internal_scores(X_model, labels)
        size_stats = cluster_size_stats(labels, SMALL_CLUSTER_SHARE_THRESHOLD)

        row = {
            "window_s": w,
            "n_rows_modeled": len(labels),
            "min_cluster_size": min_cluster_size,
            "min_samples": min_samples,
            "n_clusters": n_clusters,
            "n_noise_windows": n_noise,
            "noise_share": noise_share,
            "mean_persistence": mean_persistence,
            "median_persistence": median_persistence,
            "min_persistence": min_persistence,
            "max_persistence": max_persistence,
            "mean_membership_prob_non_noise": mean_membership_prob,
            "median_membership_prob_non_noise": median_membership_prob,
            "silhouette_non_noise": sil,
            "davies_bouldin_non_noise": dbi,
            "calinski_harabasz_non_noise": ch,
            **size_stats,
        }

        window_results.append(row)
        all_results.append(row)

        if SAVE_LABEL_FILES:
            out_labels = meta_model_df.copy()
            out_labels["hdbscan_label"] = labels
            out_labels["membership_probability"] = probs
            out_file = OUT_DIR / f"hdbscan_labels_window_{w}s_mcs{min_cluster_size}_ms{min_samples}.csv"
            out_labels.to_csv(out_file, index=False, encoding="utf-8-sig")

    results_df = pd.DataFrame(window_results)

    # practical ordering for quick inspection
    results_df = results_df.sort_values(
        by=["noise_share", "n_small_clusters", "n_clusters", "davies_bouldin_non_noise"],
        ascending=[True, True, True, True]
    ).reset_index(drop=True)

    per_window_file = OUT_DIR / f"hdbscan_screening_window_{w}s.csv"
    results_df.to_csv(per_window_file, index=False, encoding="utf-8-sig")

    print("\n=== HDBSCAN results (sorted shortlist) ===")
    display(results_df)

    print(f"\nSaved per-window results to: {per_window_file}")

    # summary row: purely heuristic, not a final model selection
    best_row = results_df.iloc[0]
    summary_rows.append({
        "window_s": w,
        "n_rows_modeled": int(best_row["n_rows_modeled"]),
        "heuristic_best_min_cluster_size": int(best_row["min_cluster_size"]),
        "heuristic_best_min_samples": int(best_row["min_samples"]),
        "heuristic_best_n_clusters": int(best_row["n_clusters"]),
        "heuristic_best_noise_share": float(best_row["noise_share"]),
        "results_file": per_window_file.name,
    })

# =========================================================
# SAVE COMBINED OUTPUTS
# =========================================================

all_results_df = pd.DataFrame(all_results)
combined_file = OUT_DIR / "hdbscan_stage2_screening_results_preprocessed.csv"
all_results_df.to_csv(combined_file, index=False, encoding="utf-8-sig")

summary_df = pd.DataFrame(summary_rows)
summary_file = OUT_DIR / "hdbscan_stage2_screening_summary_preprocessed.csv"
summary_df.to_csv(summary_file, index=False, encoding="utf-8-sig")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

print("\n=========================================================")
print(f"Saved combined results to: {combined_file}")
print(f"Saved summary to:          {summary_file}")
print(f"All output files are in:   {OUT_DIR.resolve()}")

print("\n=== Combined summary ===")
display(summary_df)

print(
    "\nInterpretation hint:\n"
    "- Prefer configurations that are not trivial (not 1-2 clusters, not almost everything as noise).\n"
    "- For HDBSCAN, noise share + cluster size distribution + number of small clusters matter more than a single index.\n"
    "- Internal scores here are secondary diagnostics computed only on non-noise windows.\n"
    "- Do NOT do noise reassignment, collapse, or minimum-duration merging here; that belongs to Stage 3.\n"
)


WINDOW 15s
Loaded: data\dataset_1Hz_windowed_15s.csv
Raw shape: (327438, 17)
Usable rows after preprocessing: 327,438
Rows used for screening:         327,438
Feature count:                  10
Feature order:
['DriveStatus_occ', 'DoorStatusMain_occ', 'DoorStatusTooling_occ', 'CoolantFlow_occ', 'OverrideFeed_mean', 'OverrideFeed_std', 'FeedRate_mean', 'FeedRate_std', 'SpindleSpeed_mean', 'SpindleSpeed_std']
Saved preprocessed feature matrix to: data\hdbscan_screening_preprocessed\window_15s_preprocessed_features.csv

Running HDBSCAN | window=15s | min_cluster_size=1500 | min_samples=50

Running HDBSCAN | window=15s | min_cluster_size=2000 | min_samples=50

Running HDBSCAN | window=15s | min_cluster_size=3000 | min_samples=50

Running HDBSCAN | window=15s | min_cluster_size=5000 | min_samples=50

Running HDBSCAN | window=15s | min_cluster_size=1500 | min_samples=100

Running HDBSCAN | window=15s | min_cluster_size=2000 | min_samples=100

Running HDBSCAN | window=15s | min_cluster_size=3

,window_s,n_rows_modeled,min_cluster_size,min_samples,n_clusters,n_noise_windows,noise_share,mean_persistence,median_persistence,min_persistence,...,median_membership_prob_non_noise,silhouette_non_noise,davies_bouldin_non_noise,calinski_harabasz_non_noise,largest_cluster_share,median_cluster_share,min_cluster_share,n_small_clusters,smallest_cluster_size,largest_cluster_size
0,15,327438,1500,50,43,51351,0.156827,1.0,1.0,1.0,...,1.0,0.589025,1.047220,8.720506e+05,0.145762,0.010139,0.004801,21,1572,47728
1,15,327438,1500,200,33,57075,0.174308,1.0,1.0,1.0,...,1.0,0.697712,0.659594,5.372332e+06,0.184859,0.010668,0.004679,15,1532,60530
2,15,327438,2000,50,35,58950,0.180034,1.0,1.0,1.0,...,1.0,0.577289,1.184023,1.050357e+06,0.145762,0.011559,0.006569,13,2151,47728
3,15,327438,2000,200,26,59640,0.182141,1.0,1.0,1.0,...,1.0,0.718408,0.515203,8.677640e+06,0.184859,0.015751,0.006578,7,2154,60530
4,15,327438,3000,200,18,60554,0.184933,1.0,1.0,1.0,...,1.0,0.687292,0.518479,4.946245e+06,0.184859,0.023522,0.010524,0,3446,60530
5,15,327438,3000,50,22,64258,0.196245,1.0,1.0,1.0,...,1.0,0.557914,1.008051,3.267552e+06,0.145762,0.023731,0.010139,0,3320,47728
6,15,327438,1500,100,42,69174,0.211258,1.0,1.0,1.0,...,1.0,0.642793,0.972509,1.166216e+06,0.145762,0.008676,0.004618,22,1512,47728
7,15,327438,5000,200,13,72804,0.222344,1.0,1.0,1.0,...,1.0,0.694497,0.507484,6.669107e+06,0.184859,0.035078,0.019482,0,6379,60530
8,15,327438,2000,100,32,74844,0.228575,1.0,1.0,1.0,...,1.0,0.644853,0.988303,2.080220e+06,0.145762,0.011656,0.006569,11,2151,47728
9,15,327438,5000,50,14,79902,0.244022,1.0,1.0,1.0,...,1.0,0.618615,0.799384,4.942844e+06,0.145762,0.039149,0.019515,0,6390,47728



Saved per-window results to: data\hdbscan_screening_preprocessed\hdbscan_screening_window_15s.csv

Saved combined results to: data\hdbscan_screening_preprocessed\hdbscan_stage2_screening_results_preprocessed.csv
Saved summary to:          data\hdbscan_screening_preprocessed\hdbscan_stage2_screening_summary_preprocessed.csv
All output files are in:   D:\zero-to-mastery\sle\data\hdbscan_screening_preprocessed

=== Combined summary ===


,window_s,n_rows_modeled,heuristic_best_min_cluster_size,heuristic_best_min_samples,heuristic_best_n_clusters,heuristic_best_noise_share,results_file
0,15,327438,1500,50,43,0.1568,hdbscan_screening_window_15s.csv



Interpretation hint:
- Prefer configurations that are not trivial (not 1-2 clusters, not almost everything as noise).
- For HDBSCAN, noise share + cluster size distribution + number of small clusters matter more than a single index.
- Internal scores here are secondary diagnostics computed only on non-noise windows.
- Do NOT do noise reassignment, collapse, or minimum-duration merging here; that belongs to Stage 3.



### LEGACY — 2.7 Legacy HMM exploratory fit (archived)

This block is retained for traceability only. The final HMM branch used in the paper is defined in the later screening cells.

In [1]:
# LEGACY
import numpy as np
import pandas as pd
from pathlib import Path

from hmmlearn.hmm import GaussianHMM

# ============================================================
# HMM over 1 Hz harmonized core dataset
# - global fit across all valid contiguous runs using lengths
# - decode state sequence
# - derive episodes
# ============================================================

# -----------------------------
# Settings
# -----------------------------
DATA_DIR = Path("data")
INPUT_FILE = DATA_DIR / "dataset_1Hz_core_harmonized.csv"

HMM_LEGACY_DIR = DATA_DIR / "hmm_legacy_exploratory"
HMM_LEGACY_DIR.mkdir(parents=True, exist_ok=True)

STATE_SEQ_OUT = HMM_LEGACY_DIR / "hmm_state_sequence_legacy.csv"
EPISODES_OUT = HMM_LEGACY_DIR / "hmm_episodes_legacy.csv"
STATE_PROFILES_OUT = HMM_LEGACY_DIR / "hmm_state_profiles_legacy.csv"

FEATURES = [
    "DriveStatus",
    "DoorStatusMain",
    "DoorStatusTooling",
    "CoolantFlow",
    "OverrideFeed",
    "FeedRate",
    "SpindleSpeed",
]

GROUP_COLS = ["session_id", "source_file", "valid_run_id"]
TIME_COL = "DateTime"

# HMM config
N_STATES = 12              # try 12 first, based on your k-means screening
COVARIANCE_TYPE = "diag"   # safer baseline for mixed/bounded features
N_ITER = 100
TOL = 1e-3
RANDOM_STATE = 42

# very short runs are skipped
MIN_RUN_LENGTH = max(30, 2 * N_STATES)

# post-processing: merge very short state fragments
MIN_EP_LEN = 3

# -----------------------------
# Helpers
# -----------------------------
def merge_short_segments(labels: np.ndarray, tau: int) -> np.ndarray:
    """
    Merge segments shorter than tau into a neighboring segment.
    This is useful for HMM because duration is not modeled explicitly.
    """
    labels = labels.copy()
    if len(labels) == 0 or tau <= 1:
        return labels

    segments = []
    start = 0
    for i in range(1, len(labels)):
        if labels[i] != labels[i - 1]:
            segments.append([start, i - 1, int(labels[i - 1])])
            start = i
    segments.append([start, len(labels) - 1, int(labels[-1])])

    i = 0
    while i < len(segments):
        s, e, lab = segments[i]
        L = e - s + 1

        if L < tau and len(segments) > 1:
            if i == 0:
                segments[i + 1][0] = s
                segments.pop(i)
                continue

            if i == len(segments) - 1:
                segments[i - 1][1] = e
                segments.pop(i)
                i -= 1
                continue

            Lprev = segments[i - 1][1] - segments[i - 1][0] + 1
            Lnext = segments[i + 1][1] - segments[i + 1][0] + 1

            if Lprev >= Lnext:
                segments[i - 1][1] = e
                segments.pop(i)
                i -= 1
                continue
            else:
                segments[i + 1][0] = s
                segments.pop(i)
                continue
        i += 1

    out = labels.copy()
    for s, e, lab in segments:
        out[s:e + 1] = lab
    return out


def extract_episodes(df_run, state_col="state"):
    """
    Convert tick-level decoded states to episodes.
    """
    x = df_run.copy().sort_values(TIME_COL).reset_index(drop=True)

    x["episode_break"] = x[state_col].ne(x[state_col].shift(1)).astype(int)
    x["episode_id_local"] = x["episode_break"].cumsum()

    ep = (
        x.groupby("episode_id_local")
         .agg(
             session_id=("session_id", "first"),
             source_file=("source_file", "first"),
             valid_run_id=("valid_run_id", "first"),
             state=(state_col, "first"),
             start_time=(TIME_COL, "first"),
             end_time=(TIME_COL, "last"),
             n_ticks=(TIME_COL, "size"),
             DriveStatus_mean=("DriveStatus", "mean"),
             DoorStatusMain_mean=("DoorStatusMain", "mean"),
             DoorStatusTooling_mean=("DoorStatusTooling", "mean"),
             CoolantFlow_mean=("CoolantFlow", "mean"),
             OverrideFeed_mean=("OverrideFeed", "mean"),
             FeedRate_mean=("FeedRate", "mean"),
             SpindleSpeed_mean=("SpindleSpeed", "mean"),
         )
         .reset_index(drop=True)
    )

    ep["duration_sec"] = ep["n_ticks"].astype(int)
    return ep


def relabel_states_by_activity(df_seq, state_col="state_raw"):
    """
    Optional relabeling for interpretability:
    order states by increasing activity score.
    """
    summary = (
        df_seq.groupby(state_col)[["FeedRate", "SpindleSpeed", "OverrideFeed"]]
        .mean()
        .fillna(0.0)
    )

    score = (
        summary["FeedRate"].rank(method="dense") +
        summary["SpindleSpeed"].rank(method="dense") +
        summary["OverrideFeed"].rank(method="dense")
    )

    ordered_states = score.sort_values().index.tolist()
    return {old: new for new, old in enumerate(ordered_states)}


# -----------------------------
# Load dataset
# -----------------------------
df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
df.columns = [str(c).strip() for c in df.columns]
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")

required = GROUP_COLS + [TIME_COL] + FEATURES
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.dropna(subset=[TIME_COL]).copy()

for c in FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# -----------------------------
# Build pooled training matrix + lengths
# -----------------------------
run_dfs = []
lengths = []

for keys, run_df in df.groupby(GROUP_COLS, sort=False):
    run_df = run_df.sort_values(TIME_COL).reset_index(drop=True).copy()
    run_df = run_df.dropna(subset=FEATURES)

    if len(run_df) < MIN_RUN_LENGTH:
        print(f"[SKIP] {keys}: too short ({len(run_df)} rows)")
        continue

    run_dfs.append(run_df)
    lengths.append(len(run_df))

if not run_dfs:
    raise RuntimeError("No runs available for HMM fitting.")

train_df = pd.concat(run_dfs, ignore_index=True)
X = train_df[FEATURES].to_numpy(dtype=float)

print("Training matrix shape:", X.shape)
print("Number of runs:", len(lengths))
print("Total length check:", sum(lengths), "==", len(X))

# -----------------------------
# Fit global HMM
# -----------------------------
hmm = GaussianHMM(
    n_components=N_STATES,
    covariance_type=COVARIANCE_TYPE,
    n_iter=N_ITER,
    tol=TOL,
    random_state=RANDOM_STATE,
    verbose=True
)

hmm.fit(X, lengths=lengths)

train_loglik = hmm.score(X, lengths=lengths)
print("Train log-likelihood:", train_loglik)

# -----------------------------
# Decode per run
# -----------------------------
all_state_sequences = []
all_episodes = []
all_profiles = []

for keys, run_df in df.groupby(GROUP_COLS, sort=False):
    session_id, source_file, valid_run_id = keys
    run_df = run_df.sort_values(TIME_COL).reset_index(drop=True).copy()
    run_df = run_df.dropna(subset=FEATURES)

    if len(run_df) < MIN_RUN_LENGTH:
        continue

    X_run = run_df[FEATURES].to_numpy(dtype=float)

    raw_states = hmm.predict(X_run)
    smooth_states = merge_short_segments(raw_states, MIN_EP_LEN)

    decoded_df = run_df.copy()
    decoded_df["state_raw"] = raw_states
    decoded_df["state_smooth_raw"] = smooth_states

    # relabel for readability
    mapping = relabel_states_by_activity(decoded_df, state_col="state_smooth_raw")
    decoded_df["state"] = decoded_df["state_smooth_raw"].map(mapping).astype(int)

    # state profiles
    profile = (
        decoded_df.groupby("state")[FEATURES]
        .mean()
        .reset_index()
    )
    profile.insert(0, "valid_run_id", valid_run_id)
    profile.insert(0, "source_file", source_file)
    profile.insert(0, "session_id", session_id)
    all_profiles.append(profile)

    # tick-level sequence
    all_state_sequences.append(decoded_df[
        GROUP_COLS + [TIME_COL] + FEATURES + ["state_raw", "state_smooth_raw", "state"]
    ].copy())

    # episodes
    ep = extract_episodes(decoded_df, state_col="state")
    all_episodes.append(ep)

# -----------------------------
# Save outputs
# -----------------------------
state_seq_df = pd.concat(all_state_sequences, ignore_index=True)
episodes_df = pd.concat(all_episodes, ignore_index=True)
profiles_df = pd.concat(all_profiles, ignore_index=True)

state_seq_df.to_csv(STATE_SEQ_OUT, index=False, encoding="utf-8-sig")
episodes_df.to_csv(EPISODES_OUT, index=False, encoding="utf-8-sig")
profiles_df.to_csv(STATE_PROFILES_OUT, index=False, encoding="utf-8-sig")

print("\nSaved:")
print(" -", STATE_SEQ_OUT)
print(" -", EPISODES_OUT)
print(" -", STATE_PROFILES_OUT)

print("\nShapes:")
print("state sequence:", state_seq_df.shape)
print("episodes:", episodes_df.shape)
print("state profiles:", profiles_df.shape)

print("\nEpisode head:")
print(episodes_df.head())

[SKIP] (1, 'Data_from_2021-11-26.csv', 15): too short (17 rows)
[SKIP] (1, 'Data_from_2021-11-26.csv', 40): too short (3 rows)
[SKIP] (2, 'Data_from_2021-12-07.csv', 19): too short (17 rows)
[SKIP] (3, 'Data_from_2021-12-17.csv', 4): too short (3 rows)
[SKIP] (5, 'Data_from_2022-01-21.csv', 16): too short (3 rows)
[SKIP] (5, 'Data_from_2022-01-21.csv', 30): too short (9 rows)
[SKIP] (6, 'Data_from_2022-02-14.csv', 2): too short (21 rows)
[SKIP] (6, 'Data_from_2022-02-14.csv', 3): too short (21 rows)
[SKIP] (6, 'Data_from_2022-02-14.csv', 10): too short (21 rows)
[SKIP] (6, 'Data_from_2022-02-14.csv', 11): too short (21 rows)
[SKIP] (7, 'Data_from_2022-02-16.csv', 2): too short (3 rows)
[SKIP] (8, 'Data_from_2022-02-18.csv', 8): too short (3 rows)
[SKIP] (9, 'Data_from_2022-02-22.csv', 30): too short (8 rows)
[SKIP] (9, 'Data_from_2022-02-22.csv', 31): too short (20 rows)
[SKIP] (9, 'Data_from_2022-02-22.csv', 32): too short (2 rows)
[SKIP] (9, 'Data_from_2022-02-22.csv', 33): too short

         1 -1785918.03459977             +nan
         2 2030322.62543147 +3816240.66003125
         3 7291582.36571237 +5261259.74028090
         4 9339321.69452432 +2047739.32881194
         5 9764493.35085002 +425171.65632571
         6 9897164.74194396 +132671.39109394
         7 9972803.77269100  +75639.03074704
         8 10079117.13355013 +106313.36085913
         9 10207392.57069931 +128275.43714918
        10 10232171.39087612  +24778.82017680
        11 10247919.52741238  +15748.13653627
        12 10248984.48582375   +1064.95841137
        13 10249543.57461880    +559.08879505
        14 10249686.59360196    +143.01898316
        15 10249769.47894868     +82.88534672
        16 10249919.93481710    +150.45586842
        17 10250084.23839113    +164.30357403
        18 10250158.15271292     +73.91432178
        19 10250185.32237937     +27.16966645
        20 10250199.90314044     +14.58076107
        21 10250208.31413716      +8.41099672
        22 10250213.70670195      +5.

Train log-likelihood: 10250230.495485323

Saved:
 - data\hmm_state_sequence.csv
 - data\hmm_episodes.csv
 - data\hmm_state_profiles.csv

Shapes:
state sequence: (331254, 14)
episodes: (7325, 15)
state profiles: (1122, 11)

Episode head:
   session_id               source_file  valid_run_id  state  \
0           1  Data_from_2021-11-26.csv             1      0   
1           1  Data_from_2021-11-26.csv             2      0   
2           1  Data_from_2021-11-26.csv             3      0   
3           1  Data_from_2021-11-26.csv             4      1   
4           1  Data_from_2021-11-26.csv             4      0   

           start_time            end_time  n_ticks  DriveStatus_mean  \
0 2021-11-26 07:15:36 2021-11-26 07:18:38      183               1.0   
1 2021-11-26 07:18:42 2021-11-26 07:20:46      125               1.0   
2 2021-11-26 07:20:50 2021-11-26 07:21:40       51               1.0   
3 2021-11-26 07:21:44 2021-11-26 07:23:52      129               1.0   
4 2021-11-26 07:23

### LEGACY — 2.8 Legacy HMM diagnostic check (archived)

In [4]:
# LEGACY
import pandas as pd

from pathlib import Path

LEGACY_DIR = Path("data") / "hmm_legacy_exploratory"

ep = pd.read_csv(LEGACY_DIR / "hmm_episodes_legacy.csv")
seq = pd.read_csv(LEGACY_DIR / "hmm_state_sequence_legacy.csv")
prof = pd.read_csv(LEGACY_DIR / "hmm_state_profiles_legacy.csv")

print("Median episode duration:", ep["duration_sec"].median())
print("Mean episode duration:", ep["duration_sec"].mean())
print("Share of episodes < 5 s:", (ep["duration_sec"] < 5).mean())
print("Share of episodes < 10 s:", (ep["duration_sec"] < 10).mean())

print("\nEpisodes per state:")
print(ep["state"].value_counts().sort_index())

print("\nTicks per state:")
print(seq["state"].value_counts(normalize=True).sort_index())

print("\nEpisodes per run:")
run_counts = ep.groupby(["session_id", "source_file", "valid_run_id"]).size()
print(run_counts.describe())

print("Converged:", hmm.monitor_.converged)
print("Iterations:", hmm.monitor_.iter)
print("Final loglik:", hmm.monitor_.history[-1])

Median episode duration: 11.0
Mean episode duration: 45.222389078498296
Share of episodes < 5 s: 0.22880546075085323
Share of episodes < 10 s: 0.46034129692832765

Episodes per state:
state
0    1284
1    1706
2    1331
3     908
4     946
5     582
6     430
7     111
8      20
9       7
Name: count, dtype: int64

Ticks per state:
state
0    0.325572
1    0.155144
2    0.140167
3    0.114067
4    0.128488
5    0.075045
6    0.046556
7    0.004293
8    0.007746
9    0.002922
Name: proportion, dtype: float64

Episodes per run:
count    277.000000
mean      26.444043
std       38.039757
min        1.000000
25%        4.000000
50%       12.000000
75%       33.000000
max      259.000000
dtype: float64
Converged: True
Iterations: 56
Final loglik: 10250230.494767286


In [5]:
# LEGACY
import pandas as pd
import numpy as np

# ============================================================
# HMM diagnostics from saved outputs
# - checks convergence metadata if hmm object still exists
# - assesses fragmentation, state usage, and run-level behavior
# ============================================================

from pathlib import Path

LEGACY_DIR = Path("data") / "hmm_legacy_exploratory"

EP_FILE = LEGACY_DIR / "hmm_episodes_legacy.csv"
SEQ_FILE = LEGACY_DIR / "hmm_state_sequence_legacy.csv"
PROF_FILE = LEGACY_DIR / "hmm_state_profiles_legacy.csv"

ep = pd.read_csv(EP_FILE, encoding="utf-8-sig")
seq = pd.read_csv(SEQ_FILE, encoding="utf-8-sig")
prof = pd.read_csv(PROF_FILE, encoding="utf-8-sig")

# --------------------------------------------------
# Basic cleanup
# --------------------------------------------------
for df in [ep, seq, prof]:
    df.columns = [str(c).strip() for c in df.columns]

if "duration_sec" in ep.columns:
    ep["duration_sec"] = pd.to_numeric(ep["duration_sec"], errors="coerce")

# --------------------------------------------------
# 1) Convergence info, if hmm object is still in memory
# --------------------------------------------------
print("=== HMM convergence info ===")
try:
    print("Converged:", hmm.monitor_.converged)
    print("Iterations:", hmm.monitor_.iter)
    print("Final log-likelihood:", hmm.monitor_.history[-1])
except Exception:
    print("HMM object not available in memory. Convergence metadata not accessible from saved CSVs.")

# --------------------------------------------------
# 2) Global episode diagnostics
# --------------------------------------------------
print("\n=== Global episode diagnostics ===")
print("Number of decoded ticks:", len(seq))
print("Number of episodes:", len(ep))

if len(ep) > 0:
    print("Mean episode duration (s):", round(ep["duration_sec"].mean(), 3))
    print("Median episode duration (s):", round(ep["duration_sec"].median(), 3))
    print("Min episode duration (s):", round(ep["duration_sec"].min(), 3))
    print("Max episode duration (s):", round(ep["duration_sec"].max(), 3))

    print("\nShare of short episodes:")
    print("< 3 s :", round((ep["duration_sec"] < 3).mean(), 4))
    print("< 5 s :", round((ep["duration_sec"] < 5).mean(), 4))
    print("< 10 s:", round((ep["duration_sec"] < 10).mean(), 4))
    print("< 30 s:", round((ep["duration_sec"] < 30).mean(), 4))

# --------------------------------------------------
# 3) Run-level fragmentation diagnostics
# --------------------------------------------------
print("\n=== Run-level fragmentation diagnostics ===")
run_group_cols = ["session_id", "source_file", "valid_run_id"]

run_ep_counts = ep.groupby(run_group_cols).size().rename("n_episodes")
run_tick_counts = seq.groupby(run_group_cols).size().rename("n_ticks")

run_diag = pd.concat([run_tick_counts, run_ep_counts], axis=1).reset_index()
run_diag["episodes_per_1000_ticks"] = 1000 * run_diag["n_episodes"] / run_diag["n_ticks"]
run_diag["mean_ticks_per_episode"] = run_diag["n_ticks"] / run_diag["n_episodes"]

print(run_diag[["n_ticks", "n_episodes", "episodes_per_1000_ticks", "mean_ticks_per_episode"]].describe())

# optional flags
run_diag["flag_high_fragmentation"] = run_diag["episodes_per_1000_ticks"] > run_diag["episodes_per_1000_ticks"].median() * 2
run_diag["flag_short_avg_episode"] = run_diag["mean_ticks_per_episode"] < 10

print("\nRuns flagged as potentially fragmented:")
flagged = run_diag[run_diag["flag_high_fragmentation"] | run_diag["flag_short_avg_episode"]]
print(flagged.sort_values(["episodes_per_1000_ticks", "mean_ticks_per_episode"], ascending=[False, True]).head(20))

# --------------------------------------------------
# 4) State usage diagnostics
# --------------------------------------------------
print("\n=== State usage diagnostics ===")

state_tick_share = (
    seq["state"]
    .value_counts(normalize=True)
    .sort_index()
    .rename("tick_share")
)

state_ep_share = (
    ep["state"]
    .value_counts(normalize=True)
    .sort_index()
    .rename("episode_share")
)

state_ep_count = (
    ep["state"]
    .value_counts()
    .sort_index()
    .rename("episode_count")
)

state_tick_count = (
    seq["state"]
    .value_counts()
    .sort_index()
    .rename("tick_count")
)

state_usage = pd.concat(
    [state_tick_count, state_tick_share, state_ep_count, state_ep_share],
    axis=1
).fillna(0)

print(state_usage)

print("\nPotentially underused states (tick share < 1% OR fewer than 20 episodes):")
underused = state_usage[(state_usage["tick_share"] < 0.01) | (state_usage["episode_count"] < 20)]
print(underused)

# --------------------------------------------------
# 5) State duration diagnostics
# --------------------------------------------------
print("\n=== Episode duration by state ===")
state_duration = ep.groupby("state")["duration_sec"].agg(
    n_episodes="count",
    mean_duration="mean",
    median_duration="median",
    min_duration="min",
    max_duration="max"
).sort_index()

print(state_duration)

# --------------------------------------------------
# 6) Transition diagnostics from decoded sequence
# --------------------------------------------------
print("\n=== Transition diagnostics ===")

seq = seq.sort_values(run_group_cols + ["DateTime"]).reset_index(drop=True)

def transitions_per_run(g):
    s = g["state"].to_numpy()
    n_transitions = int(np.sum(s[1:] != s[:-1]))
    return pd.Series({
        "n_ticks": len(g),
        "n_transitions": n_transitions,
        "transitions_per_1000_ticks": 1000 * n_transitions / len(g) if len(g) > 0 else np.nan
    })

trans_diag = seq.groupby(run_group_cols).apply(transitions_per_run).reset_index()
print(trans_diag[["n_ticks", "n_transitions", "transitions_per_1000_ticks"]].describe())

# global transition matrix
states_sorted = sorted(seq["state"].dropna().unique().tolist())
state_to_idx = {s: i for i, s in enumerate(states_sorted)}
M = np.zeros((len(states_sorted), len(states_sorted)), dtype=int)

for _, g in seq.groupby(run_group_cols, sort=False):
    s = g["state"].to_numpy()
    for i in range(len(s) - 1):
        M[state_to_idx[s[i]], state_to_idx[s[i + 1]]] += 1

trans_mat = pd.DataFrame(
    M,
    index=[f"from_{s}" for s in states_sorted],
    columns=[f"to_{s}" for s in states_sorted]
)

print("\nTransition count matrix:")
print(trans_mat)

# --------------------------------------------------
# 7) State profile diagnostics
# --------------------------------------------------
print("\n=== State profile diagnostics ===")

profile_features = [c for c in prof.columns if c not in run_group_cols + ["state"]]
profile_summary = prof.groupby("state")[profile_features].mean().sort_index()

print(profile_summary)

# quick separation heuristic
print("\nState centroid distances (rough check):")
centroids = profile_summary.to_numpy(dtype=float)
labels = profile_summary.index.to_list()

if len(centroids) >= 2:
    dist_rows = []
    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            d = np.linalg.norm(centroids[i] - centroids[j])
            dist_rows.append({"state_a": labels[i], "state_b": labels[j], "euclid_dist": d})
    dist_df = pd.DataFrame(dist_rows).sort_values("euclid_dist")
    print(dist_df.head(15))
else:
    print("Not enough states to compute centroid distances.")

# --------------------------------------------------
# 8) Compact verbal interpretation
# --------------------------------------------------
print("\n=== Quick interpretation ===")

median_dur = ep["duration_sec"].median() if len(ep) else np.nan
share_lt5 = (ep["duration_sec"] < 5).mean() if len(ep) else np.nan
n_states_used = seq["state"].nunique()

if pd.notna(median_dur):
    if median_dur < 5:
        print("- Strong warning: median episode duration is very short; the solution is likely over-fragmented.")
    elif median_dur < 15:
        print("- Caution: median episode duration is still relatively short; inspect fragmentation carefully.")
    else:
        print("- Median episode duration looks reasonably stable for a first baseline.")

if pd.notna(share_lt5):
    if share_lt5 > 0.30:
        print("- Strong warning: a large share of episodes is shorter than 5 s.")
    elif share_lt5 > 0.15:
        print("- Moderate warning: short episodes are not negligible.")
    else:
        print("- Share of very short episodes is not excessive.")

if n_states_used < 3:
    print("- Warning: too few states are effectively used.")
else:
    print(f"- {n_states_used} states are used in decoded sequences.")

if len(underused) > 0:
    print("- Some states appear weakly used; inspect whether they are meaningful or redundant.")
else:
    print("- No obviously degenerate state is underused by the simple threshold rule.")

=== HMM convergence info ===
Converged: True
Iterations: 56
Final log-likelihood: 10250230.494767286

=== Global episode diagnostics ===
Number of decoded ticks: 331254
Number of episodes: 7325
Mean episode duration (s): 45.222
Median episode duration (s): 11.0
Min episode duration (s): 3
Max episode duration (s): 3044

Share of short episodes:
< 3 s : 0.0
< 5 s : 0.2288
< 10 s: 0.4603
< 30 s: 0.7257

=== Run-level fragmentation diagnostics ===
           n_ticks  n_episodes  episodes_per_1000_ticks  \
count   277.000000  277.000000               277.000000   
mean   1195.862816   26.444043                24.292969   
std    1182.053143   38.039757                20.502805   
min      31.000000    1.000000                 0.328515   
25%     187.000000    4.000000                 8.880995   
50%    1114.000000   12.000000                18.518519   
75%    1507.000000   33.000000                34.090909   
max    7498.000000  259.000000               121.212121   

       mean_ticks_p

C:\Users\Sinus\AppData\Local\Temp\ipykernel_7672\3677206381.py:154: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  trans_diag = seq.groupby(run_group_cols).apply(transitions_per_run).reset_index()


           n_ticks  n_transitions  transitions_per_1000_ticks
count   277.000000     277.000000                  277.000000
mean   1195.862816      25.444043                   20.621516
std    1182.053143      38.039757                   19.873209
min      31.000000       0.000000                    0.000000
25%     187.000000       3.000000                    4.496403
50%    1114.000000      11.000000                   16.291161
75%    1507.000000      32.000000                   29.953917
max    7498.000000     258.000000                   96.385542

Transition count matrix:
          to_0   to_1   to_2   to_3   to_4   to_5   to_6  to_7  to_8  to_9
from_0  106563    564    266    151     67     19     74    14     0     0
from_1     485  49686    283    303    417    153     17     7     2     0
from_2     232    262  45100    197    254    222     98    27     4     0
from_3     146    304    145  36877     85     66    122    17     0     0
from_4     118    378    241     80  4161

### 2.9 HMM preprocessing for the final screening branch

In [7]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.preprocessing import RobustScaler

# ============================================================
# Create preprocessed + scaled 1 Hz dataset for HMM screening
# - input: dataset_1Hz_raw.csv
# - philosophy aligned with clustering branches:
#     * status/bounded features: unchanged (defensive clipping to [0, 1])
#     * selected nonnegative continuous features: log1p
#     * signed FeedRate: asinh
#     * all features: RobustScaler
# ============================================================

# -----------------------------
# Settings
# -----------------------------
DATA_DIR = Path("data")
INPUT_FILE = DATA_DIR / "dataset_1Hz_raw.csv"

OUTPUT_FILE = DATA_DIR / "dataset_1Hz_raw_preprocessed_scaled.csv"

META_COLS = [
    "session_id",
    "source_file",
    "valid_run_id",
    "DateTime",
]

FEATURE_COLS = [
    "DriveStatus",
    "DoorStatusMain",
    "DoorStatusTooling",
    "CoolantFlow",
    "OverrideFeed",
    "FeedRate",
    "SpindleSpeed",
]

STATUS_COLS = [
    "DriveStatus",
    "DoorStatusMain",
    "DoorStatusTooling",
    "CoolantFlow",
]

LOG1P_COLS = [
    "OverrideFeed",
    "SpindleSpeed",
]

ASINH_COLS = [
    "FeedRate",
]

# -----------------------------
# Load
# -----------------------------
df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
df.columns = [str(c).strip() for c in df.columns]

required = META_COLS + FEATURE_COLS
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[required].copy()
df["DateTime"] = pd.to_datetime(df["DateTime"], errors="coerce")

# numeric conversion
for c in FEATURE_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# drop rows with missing metadata/time/features
before_n = len(df)
df = df.dropna(subset=["DateTime"] + FEATURE_COLS).reset_index(drop=True)
after_dropna_n = len(df)

# -----------------------------
# Transformations
# -----------------------------
X = df[FEATURE_COLS].copy()

# defensive clipping for status-like features
for c in STATUS_COLS:
    x_before = X[c].copy()
    X[c] = X[c].clip(lower=0.0, upper=1.0)
    n_changed = int((x_before != X[c]).sum())
    if n_changed > 0:
        print(f"[WARN] {c}: clipped {n_changed} values outside [0, 1].")

# log1p for nonnegative skew-prone features
for c in LOG1P_COLS:
    n_neg = int((X[c] < 0).sum())
    if n_neg > 0:
        print(f"[WARN] {c}: found {n_neg} negative values before log1p; clipping to 0.")
        X[c] = X[c].clip(lower=0.0)
    X[c] = np.log1p(X[c])

# asinh for signed feature
for c in ASINH_COLS:
    X[c] = np.arcsinh(X[c])

# -----------------------------
# Robust scaling
# -----------------------------
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X[FEATURE_COLS])

X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURE_COLS)

# final output
out_df = pd.concat([df[META_COLS].reset_index(drop=True), X_scaled_df.reset_index(drop=True)], axis=1)
out_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

# -----------------------------
# Console summary
# -----------------------------
print("============================================================")
print(f"Input file:              {INPUT_FILE}")
print(f"Rows before cleaning:    {before_n:,}")
print(f"Rows after dropna:       {after_dropna_n:,}")
print(f"Rows saved:              {len(out_df):,}")
print(f"Feature count:           {len(FEATURE_COLS)}")
print("Feature order:")
print(FEATURE_COLS)
print(f"Saved file:              {OUTPUT_FILE}")
print("============================================================")

print("\nPreview:")
print(out_df.head())

Input file:              data\dataset_1Hz_raw.csv
Rows before cleaning:    331,630
Rows after dropna:       331,630
Rows saved:              331,630
Feature count:           7
Feature order:
['DriveStatus', 'DoorStatusMain', 'DoorStatusTooling', 'CoolantFlow', 'OverrideFeed', 'FeedRate', 'SpindleSpeed']
Saved file:              data\dataset_1Hz_raw_preprocessed_scaled.csv

Preview:
   session_id               source_file  valid_run_id            DateTime  \
0           1  Data_from_2021-11-26.csv             1 2021-11-26 07:15:36   
1           1  Data_from_2021-11-26.csv             1 2021-11-26 07:15:37   
2           1  Data_from_2021-11-26.csv             1 2021-11-26 07:15:38   
3           1  Data_from_2021-11-26.csv             1 2021-11-26 07:15:39   
4           1  Data_from_2021-11-26.csv             1 2021-11-26 07:15:40   

   DriveStatus  DoorStatusMain  DoorStatusTooling  CoolantFlow  OverrideFeed  \
0          0.0             0.0                0.0          0.0     -7.20

### 2.10 HMM screening

In [9]:
import numpy as np
import pandas as pd
from pathlib import Path

from hmmlearn.hmm import GaussianHMM

# ============================================================
# HMM screening over preprocessed + scaled 1 Hz dataset
# - global fit across all valid contiguous runs using lengths
# - decode tick-level state sequence
# - save sequence + global state profiles for each configuration
# - NO episodes yet
# - NO post-hoc merging of short fragments yet
# ============================================================

# -----------------------------
# Settings
# -----------------------------
DATA_DIR = Path("data")
INPUT_FILE = DATA_DIR / "dataset_1Hz_raw_preprocessed_scaled.csv"

OUT_DIR = DATA_DIR / "hmm_screening_preprocessed"
SEQ_DIR = OUT_DIR / "sequences"
PROF_DIR = OUT_DIR / "profiles"
SEQ_DIR.mkdir(parents=True, exist_ok=True)
PROF_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_OUT = OUT_DIR / "hmm_screening_summary.csv"

FEATURES = [
    "DriveStatus",
    "DoorStatusMain",
    "DoorStatusTooling",
    "CoolantFlow",
    "OverrideFeed",
    "FeedRate",
    "SpindleSpeed",
]

GROUP_COLS = ["session_id", "source_file", "valid_run_id"]
TIME_COL = "DateTime"

# screening grid
N_STATES_GRID = [8, 10, 12, 14, 16, 18]
COVARIANCE_TYPES = ["diag", "full"]

# fixed HMM settings
N_ITER = 200
TOL = 1e-2
RANDOM_STATE = 42
BASE_MIN_RUN_LENGTH = 30

SMALL_STATE_SHARE_THRESHOLD = 0.01  # 1%

# -----------------------------
# Helpers
# -----------------------------
def flatten_columns(cols):
    out = []
    for c in cols:
        if isinstance(c, tuple):
            out.append("_".join([str(x) for x in c if str(x) != ""]))
        else:
            out.append(str(c))
    return out


def relabel_states_by_activity_global(df_seq: pd.DataFrame, state_col="state_raw"):
    """
    Global relabeling for interpretability within one configuration.
    Order states by increasing activity score based on mean
    FeedRate, SpindleSpeed, OverrideFeed.
    """
    summary = (
        df_seq.groupby(state_col)[["FeedRate", "SpindleSpeed", "OverrideFeed"]]
        .mean()
        .fillna(0.0)
    )

    score = (
        summary["FeedRate"].rank(method="dense") +
        summary["SpindleSpeed"].rank(method="dense") +
        summary["OverrideFeed"].rank(method="dense")
    )

    ordered_states = score.sort_values().index.tolist()
    return {old: new for new, old in enumerate(ordered_states)}


def build_profiles(decoded_seq_df: pd.DataFrame):
    """
    Global state profiles for one configuration.
    """
    counts = decoded_seq_df["state"].value_counts().sort_index()
    shares = counts / len(decoded_seq_df)

    prof = (
        decoded_seq_df.groupby("state")[FEATURES]
        .agg(["mean", "std", "median"])
        .reset_index()
    )
    prof.columns = flatten_columns(prof.columns)

    prof["n_ticks"] = prof["state"].map(counts).astype(int)
    prof["share"] = prof["state"].map(shares).astype(float)

    return prof.sort_values("state").reset_index(drop=True)


def state_share_stats(decoded_seq_df: pd.DataFrame, threshold=0.01):
    counts = decoded_seq_df["state"].value_counts()
    shares = counts / len(decoded_seq_df)

    return {
        "largest_state_share": float(shares.max()),
        "median_state_share": float(shares.median()),
        "smallest_state_share": float(shares.min()),
        "n_states_below_1pct": int((shares < threshold).sum()),
        "smallest_state_n_ticks": int(counts.min()),
        "largest_state_n_ticks": int(counts.max()),
    }


# -----------------------------
# Load dataset
# -----------------------------
df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
df.columns = [str(c).strip() for c in df.columns]
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")

required = GROUP_COLS + [TIME_COL] + FEATURES
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.dropna(subset=[TIME_COL]).copy()

for c in FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# -----------------------------
# Screening loop
# -----------------------------
summary_rows = []

for n_states in N_STATES_GRID:
    min_run_length = max(BASE_MIN_RUN_LENGTH, 2 * n_states)

    # prepare valid runs once per n_states
    valid_runs = []
    lengths = []

    for keys, run_df in df.groupby(GROUP_COLS, sort=False):
        run_df = run_df.sort_values(TIME_COL).reset_index(drop=True).copy()
        run_df = run_df.dropna(subset=FEATURES)

        if len(run_df) < min_run_length:
            continue

        valid_runs.append((keys, run_df))
        lengths.append(len(run_df))

    if not valid_runs:
        print(f"[SKIP] n_states={n_states}: no valid runs after length filtering.")
        continue

    train_df = pd.concat([x[1] for x in valid_runs], ignore_index=True)
    X = train_df[FEATURES].to_numpy(dtype=float)

    print("\n============================================================")
    print(f"n_states = {n_states}")
    print(f"valid runs = {len(valid_runs)}")
    print(f"ticks used = {len(X):,}")
    print(f"min_run_length = {min_run_length}")

    for cov_type in COVARIANCE_TYPES:
        config_tag = f"ns{n_states:02d}_cov{cov_type}"
        print(f"\nRunning config: {config_tag}")

        try:
            hmm = GaussianHMM(
                n_components=n_states,
                covariance_type=cov_type,
                n_iter=N_ITER,
                tol=TOL,
                random_state=RANDOM_STATE,
                verbose=False
            )

            hmm.fit(X, lengths=lengths)
            train_loglik = float(hmm.score(X, lengths=lengths))
            loglik_per_tick = train_loglik / len(X)

            # decode per run
            decoded_runs = []

            for keys, run_df in valid_runs:
                X_run = run_df[FEATURES].to_numpy(dtype=float)
                raw_states = hmm.predict(X_run)

                tmp = run_df.copy()
                tmp["state_raw"] = raw_states
                decoded_runs.append(tmp)

            decoded_seq_df = pd.concat(decoded_runs, ignore_index=True)

            # global relabeling within this configuration
            mapping = relabel_states_by_activity_global(decoded_seq_df, state_col="state_raw")
            decoded_seq_df["state"] = decoded_seq_df["state_raw"].map(mapping).astype(int)

            # keep only relevant columns in final sequence output
            sequence_out_df = decoded_seq_df[
                GROUP_COLS + [TIME_COL] + FEATURES + ["state_raw", "state"]
            ].copy()

            # global profiles
            profiles_df = build_profiles(sequence_out_df)

            # save outputs
            seq_file = SEQ_DIR / f"hmm_sequence_{config_tag}.csv"
            prof_file = PROF_DIR / f"hmm_profiles_{config_tag}.csv"

            sequence_out_df.to_csv(seq_file, index=False, encoding="utf-8-sig")
            profiles_df.to_csv(prof_file, index=False, encoding="utf-8-sig")

            # summary diagnostics
            used_states_raw = int(sequence_out_df["state_raw"].nunique())
            used_states_final = int(sequence_out_df["state"].nunique())

            share_stats = state_share_stats(sequence_out_df, SMALL_STATE_SHARE_THRESHOLD)

            summary_rows.append({
                "config_tag": config_tag,
                "n_states_requested": n_states,
                "covariance_type": cov_type,
                "random_state": RANDOM_STATE,
                "n_iter": N_ITER,
                "tol": TOL,
                "min_run_length": min_run_length,
                "n_runs_used": len(valid_runs),
                "n_ticks_modeled": len(sequence_out_df),
                "train_loglik": train_loglik,
                "train_loglik_per_tick": loglik_per_tick,
                "n_states_used_raw": used_states_raw,
                "n_states_used_final": used_states_final,
                "mean_self_transition": float(np.mean(np.diag(hmm.transmat_))),
                "min_self_transition": float(np.min(np.diag(hmm.transmat_))),
                "max_self_transition": float(np.max(np.diag(hmm.transmat_))),
                **share_stats,
                "sequence_file": seq_file.name,
                "profiles_file": prof_file.name,
                "status": "ok",
                "error_message": ""
            })

            print(f"Saved: {seq_file.name}")
            print(f"Saved: {prof_file.name}")
            print(f"loglik/tick = {loglik_per_tick:.6f}")
            print(f"used states = {used_states_final}")

        except Exception as e:
            summary_rows.append({
                "config_tag": config_tag,
                "n_states_requested": n_states,
                "covariance_type": cov_type,
                "random_state": RANDOM_STATE,
                "n_iter": N_ITER,
                "tol": TOL,
                "min_run_length": min_run_length,
                "n_runs_used": len(valid_runs),
                "n_ticks_modeled": len(X),
                "train_loglik": np.nan,
                "train_loglik_per_tick": np.nan,
                "n_states_used_raw": np.nan,
                "n_states_used_final": np.nan,
                "mean_self_transition": np.nan,
                "min_self_transition": np.nan,
                "max_self_transition": np.nan,
                "largest_state_share": np.nan,
                "median_state_share": np.nan,
                "smallest_state_share": np.nan,
                "n_states_below_1pct": np.nan,
                "smallest_state_n_ticks": np.nan,
                "largest_state_n_ticks": np.nan,
                "sequence_file": "",
                "profiles_file": "",
                "status": "error",
                "error_message": str(e)
            })

            print(f"[ERROR] {config_tag}: {e}")

# -----------------------------
# Save combined summary
# -----------------------------
summary_df = pd.DataFrame(summary_rows)

# practical sorting
if not summary_df.empty:
    ok_mask = summary_df["status"].eq("ok")
    summary_df_ok = summary_df.loc[ok_mask].copy()
    summary_df_err = summary_df.loc[~ok_mask].copy()

    if not summary_df_ok.empty:
        summary_df_ok = summary_df_ok.sort_values(
            by=[
                "train_loglik_per_tick",
                "n_states_below_1pct",
                "smallest_state_share"
            ],
            ascending=[False, True, False]
        )

    summary_df = pd.concat([summary_df_ok, summary_df_err], ignore_index=True)

summary_df.to_csv(SUMMARY_OUT, index=False, encoding="utf-8-sig")

print("\n============================================================")
print(f"Saved summary to: {SUMMARY_OUT}")
print(f"Sequence files:   {SEQ_DIR.resolve()}")
print(f"Profile files:    {PROF_DIR.resolve()}")
print("============================================================")

print("\nSummary preview:")
print(summary_df.head(12))


n_states = 8
valid runs = 277
ticks used = 331,254
min_run_length = 30

Running config: ns08_covdiag
Saved: hmm_sequence_ns08_covdiag.csv
Saved: hmm_profiles_ns08_covdiag.csv
loglik/tick = 25.856115
used states = 8

Running config: ns08_covfull


Model is not converging.  Current: 9326094.754015218 is not greater than 9896601.494739138. Delta is -570506.7407239191


Saved: hmm_sequence_ns08_covfull.csv
Saved: hmm_profiles_ns08_covfull.csv
loglik/tick = 28.666548
used states = 8

n_states = 10
valid runs = 277
ticks used = 331,254
min_run_length = 30

Running config: ns10_covdiag
Saved: hmm_sequence_ns10_covdiag.csv
Saved: hmm_profiles_ns10_covdiag.csv
loglik/tick = 29.616372
used states = 10

Running config: ns10_covfull


Model is not converging.  Current: 8830602.079954855 is not greater than 9122562.9797128. Delta is -291960.89975794405


Saved: hmm_sequence_ns10_covfull.csv
Saved: hmm_profiles_ns10_covfull.csv
loglik/tick = 28.488529
used states = 10

n_states = 12
valid runs = 277
ticks used = 331,254
min_run_length = 30

Running config: ns12_covdiag


...

Saved: hmm_sequence_ns16_covfull.csv
Saved: hmm_profiles_ns16_covfull.csv
loglik/tick = 28.627966
used states = 15

n_states = 18
valid runs = 275
ticks used = 331,190
min_run_length = 36

Running config: ns18_covdiag


Model is not converging.  Current: 11490022.655681942 is not greater than 11490022.666874334. Delta is -0.01119239255785942


Saved: hmm_sequence_ns18_covdiag.csv
Saved: hmm_profiles_ns18_covdiag.csv
loglik/tick = 34.693145
used states = 18

Running config: ns18_covfull


Model is not converging.  Current: 7873911.048365461 is not greater than 11787390.37965935. Delta is -3913479.3312938884


Saved: hmm_sequence_ns18_covfull.csv
Saved: hmm_profiles_ns18_covfull.csv
loglik/tick = 30.709549
used states = 17

Saved summary to: data\hmm_screening_preprocessed\hmm_screening_summary.csv
Sequence files:   D:\zero-to-mastery\sle\data\hmm_screening_preprocessed\sequences
Profile files:    D:\zero-to-mastery\sle\data\hmm_screening_preprocessed\profiles

Summary preview:
      config_tag  n_states_requested covariance_type  random_state  n_iter  \
0   ns18_covdiag                  18            diag            42     200   
1   ns16_covdiag                  16            diag            42     200   
2   ns14_covdiag                  14            diag            42     200   
3   ns18_covfull                  18            full            42     200   
4   ns12_covdiag                  12            diag            42     200   
5   ns12_covfull                  12            full            42     200   
6   ns10_covdiag                  10            diag            42     200   
7

### 2.11 HMM raw-fragmentation diagnostics

In [10]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# Raw fragmentation proxy from saved HMM sequence files
# - reads all saved sequence files
# - computes config-level and run-level fragmentation summaries
# - no episodes, no smoothing
# ============================================================

# -----------------------------
# Settings
# -----------------------------
DATA_DIR = Path("data")
SEQ_DIR = DATA_DIR / "hmm_screening_preprocessed" / "sequences"
OUT_DIR = DATA_DIR / "hmm_screening_preprocessed"

CONFIG_OUT = OUT_DIR / "hmm_raw_fragmentation_by_config.csv"
RUN_OUT = OUT_DIR / "hmm_raw_fragmentation_by_run.csv"

GROUP_COLS = ["session_id", "source_file", "valid_run_id"]
TIME_COL = "DateTime"
STATE_COL = "state"   # boundaries are identical to state_raw after relabeling

# thresholds for quick proxy summaries
SHORT_THRESHOLDS = [1, 2, 3, 5, 10]

# -----------------------------
# Helpers
# -----------------------------
def extract_segments_from_run(g: pd.DataFrame, state_col="state") -> pd.DataFrame:
    """
    Convert one decoded run into raw contiguous state segments.
    """
    x = g.sort_values(TIME_COL).reset_index(drop=True).copy()
    s = x[state_col].to_numpy()

    if len(x) == 0:
        return pd.DataFrame(columns=GROUP_COLS + ["state", "segment_id_local", "start_time", "end_time", "n_ticks"])

    starts = [0]
    for i in range(1, len(s)):
        if s[i] != s[i - 1]:
            starts.append(i)

    ends = starts[1:] + [len(s)]
    rows = []

    for seg_id, (a, b) in enumerate(zip(starts, ends), start=1):
        rows.append({
            "session_id": x["session_id"].iloc[0],
            "source_file": x["source_file"].iloc[0],
            "valid_run_id": x["valid_run_id"].iloc[0],
            "state": int(s[a]),
            "segment_id_local": seg_id,
            "start_time": x[TIME_COL].iloc[a],
            "end_time": x[TIME_COL].iloc[b - 1],
            "n_ticks": int(b - a),
        })

    return pd.DataFrame(rows)


def summarize_one_sequence_file(seq_file: Path):
    """
    Returns:
        config_row: one-row config summary
        run_rows: per-run summary rows
    """
    df = pd.read_csv(seq_file, encoding="utf-8-sig")
    df.columns = [str(c).strip() for c in df.columns]
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")

    required = GROUP_COLS + [TIME_COL, STATE_COL]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{seq_file.name}: missing required columns: {missing}")

    df = df.dropna(subset=[TIME_COL, STATE_COL]).copy()
    df[STATE_COL] = pd.to_numeric(df[STATE_COL], errors="coerce")
    df = df.dropna(subset=[STATE_COL]).copy()
    df[STATE_COL] = df[STATE_COL].astype(int)

    config_tag = seq_file.stem.replace("hmm_sequence_", "")

    # segment extraction per run
    seg_list = []
    run_rows = []

    for keys, g in df.groupby(GROUP_COLS, sort=False):
        g = g.sort_values(TIME_COL).reset_index(drop=True).copy()
        seg = extract_segments_from_run(g, state_col=STATE_COL)
        seg_list.append(seg)

        n_ticks = len(g)
        n_segments = len(seg)
        n_transitions = max(0, n_segments - 1)

        row = {
            "config_tag": config_tag,
            "session_id": keys[0],
            "source_file": keys[1],
            "valid_run_id": keys[2],
            "n_ticks": n_ticks,
            "n_segments": n_segments,
            "n_transitions": n_transitions,
            "segments_per_1000_ticks": 1000 * n_segments / n_ticks if n_ticks > 0 else np.nan,
            "transitions_per_1000_ticks": 1000 * n_transitions / n_ticks if n_ticks > 0 else np.nan,
            "mean_ticks_per_segment": float(seg["n_ticks"].mean()) if n_segments > 0 else np.nan,
            "median_ticks_per_segment": float(seg["n_ticks"].median()) if n_segments > 0 else np.nan,
            "min_ticks_per_segment": int(seg["n_ticks"].min()) if n_segments > 0 else np.nan,
            "max_ticks_per_segment": int(seg["n_ticks"].max()) if n_segments > 0 else np.nan,
        }

        for t in SHORT_THRESHOLDS:
            if n_segments > 0:
                row[f"share_segments_le_{t}"] = float((seg["n_ticks"] <= t).mean())
                row[f"share_segments_lt_{t}"] = float((seg["n_ticks"] < t).mean())
            else:
                row[f"share_segments_le_{t}"] = np.nan
                row[f"share_segments_lt_{t}"] = np.nan

        run_rows.append(row)

    seg_df = pd.concat(seg_list, ignore_index=True) if seg_list else pd.DataFrame(columns=GROUP_COLS + ["state", "segment_id_local", "start_time", "end_time", "n_ticks"])
    run_df = pd.DataFrame(run_rows)

    # config-level summary
    n_ticks_total = len(df)
    n_runs = df[GROUP_COLS].drop_duplicates().shape[0]
    n_states_used = df[STATE_COL].nunique()
    n_segments_total = len(seg_df)
    n_transitions_total = max(0, n_segments_total - n_runs)

    config_row = {
        "config_tag": config_tag,
        "n_runs": n_runs,
        "n_ticks_total": n_ticks_total,
        "n_states_used": int(n_states_used),
        "n_segments_total": int(n_segments_total),
        "n_transitions_total": int(n_transitions_total),
        "segments_per_1000_ticks_global": 1000 * n_segments_total / n_ticks_total if n_ticks_total > 0 else np.nan,
        "transitions_per_1000_ticks_global": 1000 * n_transitions_total / n_ticks_total if n_ticks_total > 0 else np.nan,
        "mean_segment_len_ticks_global": float(seg_df["n_ticks"].mean()) if n_segments_total > 0 else np.nan,
        "median_segment_len_ticks_global": float(seg_df["n_ticks"].median()) if n_segments_total > 0 else np.nan,
        "min_segment_len_ticks_global": int(seg_df["n_ticks"].min()) if n_segments_total > 0 else np.nan,
        "max_segment_len_ticks_global": int(seg_df["n_ticks"].max()) if n_segments_total > 0 else np.nan,
        "run_mean_segments_per_1000_ticks": float(run_df["segments_per_1000_ticks"].mean()) if len(run_df) > 0 else np.nan,
        "run_median_segments_per_1000_ticks": float(run_df["segments_per_1000_ticks"].median()) if len(run_df) > 0 else np.nan,
        "run_mean_transitions_per_1000_ticks": float(run_df["transitions_per_1000_ticks"].mean()) if len(run_df) > 0 else np.nan,
        "run_median_transitions_per_1000_ticks": float(run_df["transitions_per_1000_ticks"].median()) if len(run_df) > 0 else np.nan,
        "run_mean_median_segment_len": float(run_df["median_ticks_per_segment"].mean()) if len(run_df) > 0 else np.nan,
        "run_median_median_segment_len": float(run_df["median_ticks_per_segment"].median()) if len(run_df) > 0 else np.nan,
        "runs_with_median_segment_len_lt_3": int((run_df["median_ticks_per_segment"] < 3).sum()) if len(run_df) > 0 else 0,
        "runs_with_median_segment_len_lt_5": int((run_df["median_ticks_per_segment"] < 5).sum()) if len(run_df) > 0 else 0,
    }

    for t in SHORT_THRESHOLDS:
        if n_segments_total > 0:
            config_row[f"share_segments_le_{t}_global"] = float((seg_df["n_ticks"] <= t).mean())
            config_row[f"share_segments_lt_{t}_global"] = float((seg_df["n_ticks"] < t).mean())
        else:
            config_row[f"share_segments_le_{t}_global"] = np.nan
            config_row[f"share_segments_lt_{t}_global"] = np.nan

    return pd.DataFrame([config_row]), run_df


# -----------------------------
# Main loop
# -----------------------------
seq_files = sorted(SEQ_DIR.glob("hmm_sequence_*.csv"))

if not seq_files:
    raise FileNotFoundError(f"No sequence files found in: {SEQ_DIR.resolve()}")

config_rows = []
run_rows = []

for seq_file in seq_files:
    print(f"Processing: {seq_file.name}")
    cfg_df, run_df = summarize_one_sequence_file(seq_file)
    config_rows.append(cfg_df)
    run_rows.append(run_df)

config_summary = pd.concat(config_rows, ignore_index=True)
run_summary = pd.concat(run_rows, ignore_index=True)

# practical sorting:
# prefer longer raw segments and lower fragmentation
config_summary = config_summary.sort_values(
    by=[
        "median_segment_len_ticks_global",
        "share_segments_le_2_global",
        "share_segments_lt_5_global",
        "transitions_per_1000_ticks_global"
    ],
    ascending=[False, True, True, True]
).reset_index(drop=True)

config_summary.to_csv(CONFIG_OUT, index=False, encoding="utf-8-sig")
run_summary.to_csv(RUN_OUT, index=False, encoding="utf-8-sig")

print("\n============================================================")
print(f"Saved config summary to: {CONFIG_OUT}")
print(f"Saved run summary to:    {RUN_OUT}")
print("============================================================")

print("\n=== Config-level raw fragmentation summary ===")
show_cols = [
    "config_tag",
    "n_states_used",
    "n_segments_total",
    "segments_per_1000_ticks_global",
    "transitions_per_1000_ticks_global",
    "mean_segment_len_ticks_global",
    "median_segment_len_ticks_global",
    "share_segments_le_1_global",
    "share_segments_le_2_global",
    "share_segments_lt_5_global",
    "runs_with_median_segment_len_lt_3",
    "runs_with_median_segment_len_lt_5",
]
print(config_summary[show_cols].head(20))

print("\n=== Most fragmented runs (top 20) ===")
run_show_cols = [
    "config_tag",
    "session_id",
    "valid_run_id",
    "n_ticks",
    "n_segments",
    "segments_per_1000_ticks",
    "transitions_per_1000_ticks",
    "mean_ticks_per_segment",
    "median_ticks_per_segment",
    "share_segments_le_1",
    "share_segments_le_2",
    "share_segments_lt_5",
]
print(
    run_summary.sort_values(
        ["segments_per_1000_ticks", "share_segments_le_1", "median_ticks_per_segment"],
        ascending=[False, False, True]
    )[run_show_cols].head(20)
)

Processing: hmm_sequence_ns08_covdiag.csv
Processing: hmm_sequence_ns08_covfull.csv
Processing: hmm_sequence_ns10_covdiag.csv
Processing: hmm_sequence_ns10_covfull.csv
Processing: hmm_sequence_ns12_covdiag.csv
Processing: hmm_sequence_ns12_covfull.csv
Processing: hmm_sequence_ns14_covdiag.csv
Processing: hmm_sequence_ns14_covfull.csv
Processing: hmm_sequence_ns16_covdiag.csv
Processing: hmm_sequence_ns16_covfull.csv
Processing: hmm_sequence_ns18_covdiag.csv
Processing: hmm_sequence_ns18_covfull.csv

Saved config summary to: data\hmm_screening_preprocessed\hmm_raw_fragmentation_by_config.csv
Saved run summary to:    data\hmm_screening_preprocessed\hmm_raw_fragmentation_by_run.csv

=== Config-level raw fragmentation summary ===
      config_tag  n_states_used  n_segments_total  \
0   ns10_covdiag             10              8257   
1   ns08_covfull              8              9391   
2   ns08_covdiag              8             11665   
3   ns12_covfull             12             11347   

## Stage 3 — Episodic standardization and meta-mode alignment

### 3.1 Shared 1 Hz scaling for Stage 3 episode descriptors

In [23]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.preprocessing import RobustScaler

# ============================================================
# Create unified 1 Hz dataset for clustering-style branches
# and descriptor recomputation
#
# Philosophy:
# - occupancy/state-like features:
#     * already coded as 0/1 in dataset_1Hz_raw.csv
#     * kept unchanged and unscaled
# - nonnegative continuous features:
#     * log1p
# - signed continuous features:
#     * asinh
# - continuous features only:
#     * RobustScaler
#
# Output:
#   dataset_1Hz_raw_uni_scaled.csv
# ============================================================

# -----------------------------
# Settings
# -----------------------------
DATA_DIR = Path("data")
INPUT_FILE = DATA_DIR / "dataset_1Hz_raw.csv"
OUTPUT_FILE = DATA_DIR / "dataset_1Hz_raw_uni_scaled.csv"

META_COLS = [
    "session_id",
    "source_file",
    "valid_run_id",
    "DateTime",
]

OCC_COLS = [
    "DriveStatus",
    "DoorStatusMain",
    "DoorStatusTooling",
    "CoolantFlow",
]

LOG1P_COLS = [
    "OverrideFeed",
    "SpindleSpeed",
]

ASINH_COLS = [
    "FeedRate",
]

FEATURE_COLS = OCC_COLS + LOG1P_COLS + ASINH_COLS


# -----------------------------
# Preprocess
# -----------------------------
def preprocess_1hz_features(df_raw: pd.DataFrame):
    """
    Returns:
        meta_df: metadata rows aligned with modeled rows
        X_transformed_df: features after signal-specific transforms, before scaling of continuous vars
        X_final_df: final feature dataframe used downstream
                    (occupancy cols unchanged in 0/1, continuous cols scaled)
    """
    missing = [c for c in META_COLS + FEATURE_COLS if c not in df_raw.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    meta_df = df_raw[META_COLS].copy()
    X = df_raw[FEATURE_COLS].copy()

    for c in FEATURE_COLS:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    keep_mask = meta_df["DateTime"].notna() & X.notna().all(axis=1)
    meta_df = meta_df.loc[keep_mask].reset_index(drop=True)
    X = X.loc[keep_mask].reset_index(drop=True)

    # occupancy/state-like features: keep in 0/1, just defensive clipping
    for c in OCC_COLS:
        X[c] = X[c].clip(lower=0.0, upper=1.0)

    # log1p for nonnegative skew-prone continuous features
    for c in LOG1P_COLS:
        n_neg = int((X[c] < 0).sum())
        if n_neg > 0:
            print(f"[WARN] {c}: found {n_neg} negative values before log1p; clipping to 0.")
            X[c] = X[c].clip(lower=0.0)
        X[c] = np.log1p(X[c])

    # asinh for signed continuous feature
    for c in ASINH_COLS:
        X[c] = np.arcsinh(X[c])

    X_transformed_df = X.copy()

    # scale continuous features only
    CONT_COLS = LOG1P_COLS + ASINH_COLS
    scaler = RobustScaler()
    X_cont_scaled = scaler.fit_transform(X[CONT_COLS])

    X_final_df = X.copy()
    X_final_df.loc[:, CONT_COLS] = X_cont_scaled

    return meta_df, X_transformed_df, X_final_df


# -----------------------------
# Load
# -----------------------------
df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
df.columns = [str(c).strip() for c in df.columns]
df["DateTime"] = pd.to_datetime(df["DateTime"], errors="coerce")

before_n = len(df)

meta_df, X_transformed_df, X_final_df = preprocess_1hz_features(df)

out_df = pd.concat(
    [
        meta_df.reset_index(drop=True),
        X_final_df.reset_index(drop=True),
    ],
    axis=1,
)

out_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

# -----------------------------
# Console summary
# -----------------------------
print("============================================================")
print(f"Input file:              {INPUT_FILE}")
print(f"Rows before cleaning:    {before_n:,}")
print(f"Rows saved:              {len(out_df):,}")
print(f"Feature count:           {len(FEATURE_COLS)}")
print("Occupancy/state-like cols (kept in 0/1, unscaled):")
print(OCC_COLS)
print("Continuous cols (transformed + robust-scaled):")
print(LOG1P_COLS + ASINH_COLS)
print(f"Saved file:              {OUTPUT_FILE}")
print("============================================================")

print("\nPreview:")
print(out_df.head())

Input file:              data\dataset_1Hz_raw.csv
Rows before cleaning:    331,630
Rows saved:              331,630
Feature count:           7
Occupancy/state-like cols (kept in 0/1, unscaled):
['DriveStatus', 'DoorStatusMain', 'DoorStatusTooling', 'CoolantFlow']
Continuous cols (transformed + robust-scaled):
['OverrideFeed', 'SpindleSpeed', 'FeedRate']
Saved file:              data\dataset_1Hz_raw_uni_scaled.csv

Preview:
   session_id               source_file  valid_run_id            DateTime  \
0           1  Data_from_2021-11-26.csv             1 2021-11-26 07:15:36   
1           1  Data_from_2021-11-26.csv             1 2021-11-26 07:15:37   
2           1  Data_from_2021-11-26.csv             1 2021-11-26 07:15:38   
3           1  Data_from_2021-11-26.csv             1 2021-11-26 07:15:39   
4           1  Data_from_2021-11-26.csv             1 2021-11-26 07:15:40   

   DriveStatus  DoorStatusMain  DoorStatusTooling  CoolantFlow  OverrideFeed  \
0            1               0

### 3.2 Project raw labels to the shared 1 Hz timeline and form preliminary episodes

In [28]:
from pathlib import Path
from collections import defaultdict
import json
import numpy as np
import pandas as pd

# ==========================================
# Configuration
# ==========================================
STAGE3_DIR = Path("Stage3")
SHARED_1HZ_PATH = Path("data") / "dataset_1Hz_raw_uni_scaled.csv"
OUTPUT_DIR = STAGE3_DIR / "stage3_331_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_ROOTS = {
    "kmeans": Path("data"),
    "hdbscan": Path("data") / "hdbscan_screening_preprocessed",
    "hmm": Path("data") / "hmm_screening_preprocessed" / "sequences",
}

SAVE_MICRO_AUDIT = True
SAVE_SUMMARY = True

KEY_COLS = ["session_id", "source_file", "valid_run_id", "DateTime"]
RUN_KEY_COLS = ["session_id", "source_file", "valid_run_id"]

# Shared 1 Hz state-vector x_t used in v_k^(m)
BINARY_COLS = [
    "DriveStatus",
    "DoorStatusMain",
    "DoorStatusTooling",
    "CoolantFlow",
]
CONT_COLS = [
    "OverrideFeed",
    "FeedRate",
    "SpindleSpeed",
]
SIGNAL_COLS = BINARY_COLS + CONT_COLS

PROJECTION_RULE = "majority_vote__nearest_window_center__latest_window_end"

INPUT_SPECS = [
    # -------------------------
    # K-means
    # -------------------------
    {
        "input_file": "dataset_1Hz_windowed_5s_kmeans_selected.csv",
        "family": "kmeans",
        "designs": [
            {"design_id": "kmeans_5s_k4", "label_col": "kmeans_raw_k4", "used_col": "kmeans_used_k4", "window_size_s": 5},
            {"design_id": "kmeans_5s_k7", "label_col": "kmeans_raw_k7", "used_col": "kmeans_used_k7", "window_size_s": 5},
        ],
    },
    {
        "input_file": "dataset_1Hz_windowed_10s_kmeans_selected.csv",
        "family": "kmeans",
        "designs": [
            {"design_id": "kmeans_10s_k4", "label_col": "kmeans_raw_k4", "used_col": "kmeans_used_k4", "window_size_s": 10},
            {"design_id": "kmeans_10s_k10", "label_col": "kmeans_raw_k10", "used_col": "kmeans_used_k10", "window_size_s": 10},
        ],
    },
    {
        "input_file": "dataset_1Hz_windowed_15s_kmeans_selected.csv",
        "family": "kmeans",
        "designs": [
            {"design_id": "kmeans_15s_k4", "label_col": "kmeans_raw_k4", "used_col": "kmeans_used_k4", "window_size_s": 15},
            {"design_id": "kmeans_15s_k9", "label_col": "kmeans_raw_k9", "used_col": "kmeans_used_k9", "window_size_s": 15},
        ],
    },

    # -------------------------
    # HDBSCAN
    # -------------------------
    {
        "input_file": "hdbscan_labels_window_5s_mcs3000_ms200.csv",
        "family": "hdbscan",
        "designs": [
            {"design_id": "hdbscan_5s_mcs3000_ms200", "label_col": "hdbscan_label", "used_col": None, "window_size_s": 5},
        ],
    },
    {
        "input_file": "hdbscan_labels_window_10s_mcs5000_ms50.csv",
        "family": "hdbscan",
        "designs": [
            {"design_id": "hdbscan_10s_mcs5000_ms50", "label_col": "hdbscan_label", "used_col": None, "window_size_s": 10},
        ],
    },
    {
        "input_file": "hdbscan_labels_window_15s_mcs3000_ms200.csv",
        "family": "hdbscan",
        "designs": [
            {"design_id": "hdbscan_15s_mcs3000_ms200", "label_col": "hdbscan_label", "used_col": None, "window_size_s": 15},
        ],
    },

    # -------------------------
    # HMM
    # -------------------------
    {
        "input_file": "hmm_sequence_ns08_covdiag.csv",
        "family": "hmm",
        "designs": [
            {"design_id": "hmm_ns08_covdiag", "label_col": "state_raw", "fallback_label_col": "state", "used_col": None, "window_size_s": np.nan},
        ],
    },
    {
        "input_file": "hmm_sequence_ns10_covdiag.csv",
        "family": "hmm",
        "designs": [
            {"design_id": "hmm_ns10_covdiag", "label_col": "state_raw", "fallback_label_col": "state", "used_col": None, "window_size_s": np.nan},
        ],
    },
    {
        "input_file": "hmm_sequence_ns12_covdiag.csv",
        "family": "hmm",
        "designs": [
            {"design_id": "hmm_ns12_covdiag", "label_col": "state_raw", "fallback_label_col": "state", "used_col": None, "window_size_s": np.nan},
        ],
    },
]


# ==========================================
# Helpers
# ==========================================
def parse_datetime_cols(df: pd.DataFrame) -> pd.DataFrame:
    for col in ["DateTime", "window_start", "window_end"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df


def coerce_used_mask(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s.fillna(False)
    s_str = s.astype(str).str.strip().str.lower()
    return s_str.isin(["1", "true", "t", "yes", "y"])


def label_sort_key(x):
    return str(x)


def json_safe_value(x):
    if pd.isna(x):
        return None
    if isinstance(x, (np.floating, float)):
        return float(x)
    if isinstance(x, (np.integer, int)):
        return int(x)
    return x


def iqr(series: pd.Series) -> float | None:
    vals = pd.to_numeric(series, errors="coerce").dropna()
    if len(vals) == 0:
        return None
    q75, q25 = np.percentile(vals, [75, 25])
    return float(q75 - q25)


def build_shared_tick_frame(shared_1hz_path: Path) -> pd.DataFrame:
    """
    Load the authoritative shared 1 Hz timeline + x_t from the unified 1 Hz file.
    """
    if not shared_1hz_path.exists():
        raise FileNotFoundError(f"Shared 1 Hz file not found: {shared_1hz_path}")

    df = pd.read_csv(shared_1hz_path)
    df = parse_datetime_cols(df)

    required = KEY_COLS + SIGNAL_COLS
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{shared_1hz_path.name} is missing required columns: {missing}")

    df = df[required].drop_duplicates(subset=KEY_COLS).copy()
    df = df.sort_values(RUN_KEY_COLS + ["DateTime"]).reset_index(drop=True)

    # Tick index within each run on the shared timeline
    df["t_idx"] = df.groupby(RUN_KEY_COLS).cumcount() + 1
    return df


def build_x_t_json(df: pd.DataFrame) -> pd.Series:
    return pd.Series([
        json.dumps({c: json_safe_value(row[c]) for c in SIGNAL_COLS}, ensure_ascii=False)
        for _, row in df[SIGNAL_COLS].iterrows()
    ], index=df.index)


def build_episode_descriptor(g: pd.DataFrame) -> dict:
    """
    v_k^(m) = psi(X_{s_k:f_k})
    Contract used here:
    - binary/state-like signals -> occupancy
    - continuous signals -> median and IQR
    """
    out = {}

    # Binary / state-like: occupancy from 0/1 unified source
    for col in BINARY_COLS:
        vals = pd.to_numeric(g[col], errors="coerce").dropna()
        out[f"{col}_occ"] = None if len(vals) == 0 else float(vals.mean())

    # Continuous
    for col in CONT_COLS:
        vals = pd.to_numeric(g[col], errors="coerce").dropna()
        if len(vals) == 0:
            out[f"{col}_median"] = None
            out[f"{col}_iqr"] = None
        else:
            out[f"{col}_median"] = float(vals.median())
            out[f"{col}_iqr"] = iqr(vals)

    return out


def derive_window_bounds(df: pd.DataFrame, default_window_size_s: int | float | None) -> pd.DataFrame:
    """
    Ensure window_start and window_end exist.
    Uses explicit columns if present; otherwise falls back to DateTime + window_size_s.
    """
    df = df.copy()

    has_start = "window_start" in df.columns
    has_end = "window_end" in df.columns

    if has_start and has_end:
        return df

    if "DateTime" not in df.columns:
        raise ValueError("Window input has no DateTime column and no explicit window_start/window_end.")

    if "window_size_s" in df.columns:
        size = pd.to_numeric(df["window_size_s"], errors="coerce")
    else:
        size = pd.Series(default_window_size_s, index=df.index)

    if size.isna().any():
        raise ValueError("Cannot derive window bounds because window_size_s is missing.")

    # Assume DateTime marks the window end if explicit bounds are absent.
    df["window_end"] = df["DateTime"]
    df["window_start"] = df["window_end"] - pd.to_timedelta(size.astype(int) - 1, unit="s")
    return df


def project_windows_to_ticks(
    windows_df: pd.DataFrame,
    shared_ticks: pd.DataFrame,
    family: str,
    design_cfg: dict,
    input_file: str,
) -> pd.DataFrame:
    """
    Convert window-level labels into a 1 Hz raw label sequence z_t^(m) by projecting
    window labels onto the shared timeline.

    Projection rule:
    1) majority vote across covering windows
    2) tie-break by nearest window center
    3) tie-break by latest window_end
    4) final tie-break lexicographically by label string
    """
    df = windows_df.copy()
    df = derive_window_bounds(df, design_cfg.get("window_size_s"))
    label_col = design_cfg["label_col"]

    if label_col not in df.columns:
        raise ValueError(f"{input_file}: missing label column {label_col}")

    used_col = design_cfg.get("used_col")
    if used_col is not None:
        if used_col not in df.columns:
            raise ValueError(f"{input_file}: missing used flag {used_col}")
        df = df.loc[coerce_used_mask(df[used_col])].copy()

    df = df.loc[df[label_col].notna()].copy()
    df = df[RUN_KEY_COLS + ["window_start", "window_end", label_col]].copy()
    df = df.rename(columns={label_col: "raw_label_window"})
    df = df.sort_values(RUN_KEY_COLS + ["window_start", "window_end"]).reset_index(drop=True)

    out_parts = []

    # Only runs that appear in this window file are processed for this design
    run_keys_present = df[RUN_KEY_COLS].drop_duplicates()

    merged_runs = shared_ticks.merge(run_keys_present, on=RUN_KEY_COLS, how="inner")
    if merged_runs.empty:
        print(f"[WARN] {input_file} / {design_cfg['design_id']}: no matching shared 1 Hz ticks for any run.")
        return pd.DataFrame()

    for run_key, ticks_run in merged_runs.groupby(RUN_KEY_COLS, sort=False):
        ticks_run = ticks_run.sort_values("DateTime").reset_index(drop=True)
        windows_run = df[
            (df["session_id"] == run_key[0]) &
            (df["source_file"] == run_key[1]) &
            (df["valid_run_id"] == run_key[2])
        ].copy()

        if windows_run.empty:
            continue

        times = ticks_run["DateTime"].to_numpy(dtype="datetime64[ns]")
        n_ticks = len(ticks_run)

        vote_counts = [defaultdict(int) for _ in range(n_ticks)]
        best_dist = [dict() for _ in range(n_ticks)]
        best_end = [dict() for _ in range(n_ticks)]
        n_covering_windows = np.zeros(n_ticks, dtype=int)

        for row in windows_run.itertuples(index=False):
            w_start = row.window_start
            w_end = row.window_end
            label = row.raw_label_window

            if pd.isna(w_start) or pd.isna(w_end):
                continue

            left = np.searchsorted(times, np.datetime64(w_start), side="left")
            right = np.searchsorted(times, np.datetime64(w_end), side="right") - 1

            if right < left:
                continue

            center = w_start + (w_end - w_start) / 2

            for i in range(left, right + 1):
                n_covering_windows[i] += 1
                vote_counts[i][label] += 1

                tick_time = pd.Timestamp(times[i])
                dist = abs((tick_time - center).total_seconds())

                prev_dist = best_dist[i].get(label, None)
                prev_end = best_end[i].get(label, None)

                if (
                    prev_dist is None
                    or dist < prev_dist
                    or (dist == prev_dist and (prev_end is None or w_end > prev_end))
                ):
                    best_dist[i][label] = dist
                    best_end[i][label] = w_end

        assigned_labels = []
        for i in range(n_ticks):
            if not vote_counts[i]:
                assigned_labels.append(np.nan)
                continue

            max_votes = max(vote_counts[i].values())
            candidate_labels = [lab for lab, cnt in vote_counts[i].items() if cnt == max_votes]

            if len(candidate_labels) == 1:
                assigned = candidate_labels[0]
            else:
                min_dist = min(best_dist[i][lab] for lab in candidate_labels)
                candidate_labels_2 = [lab for lab in candidate_labels if best_dist[i][lab] == min_dist]

                if len(candidate_labels_2) == 1:
                    assigned = candidate_labels_2[0]
                else:
                    latest_end = max(best_end[i][lab] for lab in candidate_labels_2)
                    candidate_labels_3 = [lab for lab in candidate_labels_2 if best_end[i][lab] == latest_end]

                    if len(candidate_labels_3) == 1:
                        assigned = candidate_labels_3[0]
                    else:
                        assigned = sorted(candidate_labels_3, key=label_sort_key)[0]

            assigned_labels.append(assigned)

        run_out = ticks_run.copy()
        run_out["raw_label"] = assigned_labels
        run_out["n_covering_windows"] = n_covering_windows
        run_out["projection_rule"] = PROJECTION_RULE
        run_out["design_id"] = design_cfg["design_id"]
        run_out["design_family"] = family
        run_out["input_file"] = input_file
        run_out["window_size_s"] = design_cfg.get("window_size_s", np.nan)

        out_parts.append(run_out)

    if not out_parts:
        return pd.DataFrame()

    tick_df = pd.concat(out_parts, ignore_index=True)
    tick_df["x_t_json"] = build_x_t_json(tick_df)
    tick_df["micro_start_time"] = tick_df["DateTime"]
    tick_df["micro_end_time"] = tick_df["DateTime"]
    tick_df["micro_duration_ticks"] = 1
    return tick_df


def prepare_hmm_ticks(
    hmm_df: pd.DataFrame,
    shared_ticks: pd.DataFrame,
    family: str,
    design_cfg: dict,
    input_file: str,
) -> pd.DataFrame:
    """
    HMM provides tick-wise labels, but x_t and SIGNAL_COLS come from shared_ticks.
    """
    df = hmm_df.copy()
    label_col = design_cfg["label_col"]
    if label_col not in df.columns:
        fallback = design_cfg.get("fallback_label_col")
        if fallback and fallback in df.columns:
            label_col = fallback
        else:
            raise ValueError(f"{input_file}: missing HMM label column {label_col}")

    required = KEY_COLS + [label_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{input_file}: missing required columns for HMM tick processing: {missing}")

    df = parse_datetime_cols(df)
    df = df[required].copy()
    df = df.rename(columns={label_col: "raw_label"})
    df = df.loc[df["raw_label"].notna()].copy()

    tick_df = shared_ticks.merge(df, on=KEY_COLS, how="inner")
    if tick_df.empty:
        print(f"[WARN] {input_file} / {design_cfg['design_id']}: no matching shared 1 Hz ticks for HMM labels.")
        return pd.DataFrame()

    tick_df = tick_df.sort_values(RUN_KEY_COLS + ["DateTime"]).reset_index(drop=True)
    tick_df["design_id"] = design_cfg["design_id"]
    tick_df["design_family"] = family
    tick_df["input_file"] = input_file
    tick_df["window_size_s"] = design_cfg.get("window_size_s", np.nan)
    tick_df["projection_rule"] = "direct_tick_labels"
    tick_df["n_covering_windows"] = 1
    tick_df["x_t_json"] = build_x_t_json(tick_df)
    tick_df["micro_start_time"] = tick_df["DateTime"]
    tick_df["micro_end_time"] = tick_df["DateTime"]
    tick_df["micro_duration_ticks"] = 1

    return tick_df


def aggregate_preliminary_episodes(tick_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build E^(m) by aggregating temporally adjacent micro-episodes with the same raw label.
    Only labeled ticks are used for episode construction.
    Unlabeled ticks are kept in the micro audit and diagnostics, but not converted into episodes.
    """
    if tick_df.empty:
        return pd.DataFrame()

    df = tick_df.loc[tick_df["raw_label"].notna()].copy()
    if df.empty:
        return pd.DataFrame()

    df = df.sort_values(["design_id"] + RUN_KEY_COLS + ["DateTime"]).reset_index(drop=True)

    group_change = (
        df["design_id"].ne(df["design_id"].shift()) |
        df["session_id"].ne(df["session_id"].shift()) |
        df["source_file"].ne(df["source_file"].shift()) |
        df["valid_run_id"].ne(df["valid_run_id"].shift())
    )
    dt_diff = df["DateTime"].diff()
    time_gap = dt_diff.ne(pd.Timedelta(seconds=1))
    label_change = df["raw_label"].ne(df["raw_label"].shift())

    new_episode = group_change | time_gap | label_change
    df["episode_group"] = new_episode.cumsum()

    rows = []
    for _, g in df.groupby("episode_group", sort=True):
        g = g.sort_values("DateTime")
        descriptor = build_episode_descriptor(g)

        row = {
            "design_id": g["design_id"].iloc[0],
            "design_family": g["design_family"].iloc[0],
            "input_file": g["input_file"].iloc[0],
            "window_size_s": g["window_size_s"].iloc[0],
            "session_id": g["session_id"].iloc[0],
            "source_file": g["source_file"].iloc[0],
            "valid_run_id": g["valid_run_id"].iloc[0],
            "start_time": g["DateTime"].iloc[0],
            "end_time": g["DateTime"].iloc[-1],
            "s_k": int(g["t_idx"].iloc[0]),
            "f_k": int(g["t_idx"].iloc[-1]),
            "d_k": int(len(g)),
            "l_k": g["raw_label"].iloc[0],
            "n_micro_episodes": int(len(g)),
        }

        row["v_json"] = json.dumps({k: json_safe_value(v) for k, v in descriptor.items()}, ensure_ascii=False)
        for k, v in descriptor.items():
            row[f"v_{k}"] = v

        rows.append(row)

    ep = pd.DataFrame(rows)
    ep = ep.sort_values(["design_id"] + RUN_KEY_COLS + ["start_time"]).reset_index(drop=True)
    ep["episode_id"] = ep.groupby(["design_id"] + RUN_KEY_COLS).cumcount() + 1

    descriptor_cols = [c for c in ep.columns if c.startswith("v_") and c != "v_json"]
    ordered_cols = [
        "design_id",
        "design_family",
        "input_file",
        "window_size_s",
        "session_id",
        "source_file",
        "valid_run_id",
        "episode_id",
        "start_time",
        "end_time",
        "s_k",
        "f_k",
        "d_k",
        "n_micro_episodes",
        "l_k",
        "v_json",
    ] + descriptor_cols

    return ep[ordered_cols]


def build_summary(tick_df: pd.DataFrame, episodes_df: pd.DataFrame) -> pd.DataFrame:
    if tick_df.empty:
        return pd.DataFrame()

    rows = []
    design_groups = tick_df.groupby(["design_id", "design_family", "input_file", "window_size_s"], dropna=False)

    for (design_id, design_family, input_file, window_size_s), tg in design_groups:
        eg = episodes_df.loc[episodes_df["design_id"] == design_id].copy() if not episodes_df.empty else pd.DataFrame()

        n_ticks_total = len(tg)
        n_labeled_ticks = int(tg["raw_label"].notna().sum())
        n_unlabeled_ticks = int(n_ticks_total - n_labeled_ticks)
        coverage_rate = float(n_labeled_ticks / n_ticks_total) if n_ticks_total > 0 else np.nan

        n_runs = tg[RUN_KEY_COLS].drop_duplicates().shape[0]
        n_unique_labels = int(tg.loc[tg["raw_label"].notna(), "raw_label"].nunique())

        if eg.empty:
            n_episodes = 0
            median_d = np.nan
            q10_d = np.nan
            q25_d = np.nan
            n_single_tick = 0
            prop_single_tick = np.nan
        else:
            d = pd.to_numeric(eg["d_k"], errors="coerce").dropna()
            n_episodes = int(len(eg))
            median_d = float(d.median()) if len(d) else np.nan
            q10_d = float(np.percentile(d, 10)) if len(d) else np.nan
            q25_d = float(np.percentile(d, 25)) if len(d) else np.nan
            n_single_tick = int((d == 1).sum()) if len(d) else 0
            prop_single_tick = float(n_single_tick / len(d)) if len(d) else np.nan

        rows.append({
            "design_id": design_id,
            "design_family": design_family,
            "input_file": input_file,
            "window_size_s": window_size_s,
            "projection_rule": tg["projection_rule"].dropna().iloc[0] if "projection_rule" in tg.columns and tg["projection_rule"].notna().any() else None,
            "n_runs": n_runs,
            "n_ticks_total": n_ticks_total,
            "n_labeled_ticks": n_labeled_ticks,
            "n_unlabeled_ticks": n_unlabeled_ticks,
            "label_coverage_rate": coverage_rate,
            "n_unique_raw_labels": n_unique_labels,
            "n_preliminary_episodes": n_episodes,
            "median_episode_duration_ticks": median_d,
            "q10_episode_duration_ticks": q10_d,
            "q25_episode_duration_ticks": q25_d,
            "n_single_tick_episodes": n_single_tick,
            "prop_single_tick_episodes": prop_single_tick,
        })

    return pd.DataFrame(rows).sort_values(["design_family", "design_id"]).reset_index(drop=True)


def process_input_spec(spec: dict, shared_ticks: pd.DataFrame):
    in_path = INPUT_ROOTS[spec["family"]] / spec["input_file"]
    if not in_path.exists():
        print(f"[WARN] Missing input file: {in_path.name}")
        return

    print(f"[INFO] Processing {in_path.name}")
    raw_df = pd.read_csv(in_path)
    raw_df = parse_datetime_cols(raw_df)

    for design_cfg in spec["designs"]:
        design_id = design_cfg["design_id"]

        if spec["family"] == "hmm":
            tick_df = prepare_hmm_ticks(
                raw_df,
                shared_ticks,
                spec["family"],
                design_cfg,
                spec["input_file"],
            )
        else:
            tick_df = project_windows_to_ticks(
                raw_df,
                shared_ticks,
                spec["family"],
                design_cfg,
                spec["input_file"],
            )

        if tick_df.empty:
            print(f"[WARN] {spec['input_file']} / {design_id}: no tick-level sequence produced.")
            continue

        tick_df = tick_df.sort_values(["design_id"] + RUN_KEY_COLS + ["DateTime"]).reset_index(drop=True)

        episodes = aggregate_preliminary_episodes(tick_df)
        summary = build_summary(tick_df, episodes)

        # duplicated intervals inside one design/run should never happen
        if not episodes.empty:
            dup_mask = episodes.duplicated(
                ["design_id"] + RUN_KEY_COLS + ["s_k", "f_k"],
                keep=False
            )
            if dup_mask.any():
                dup_view = episodes.loc[
                    dup_mask,
                    ["design_id"] + RUN_KEY_COLS + ["episode_id", "s_k", "f_k", "l_k"]
                ].sort_values(["design_id"] + RUN_KEY_COLS + ["s_k", "f_k", "episode_id"])
                raise ValueError(
                    f"Duplicated episode intervals detected for design {design_id}:\n"
                    f"{dup_view.to_string(index=False)}"
                )

        # -------------------------
        # Save outputs per design
        # -------------------------
        out_prefix = design_id

        if SAVE_MICRO_AUDIT:
            micro_cols = [
                "design_id",
                "design_family",
                "input_file",
                "window_size_s",
                "projection_rule",
                "session_id",
                "source_file",
                "valid_run_id",
                "DateTime",
                "t_idx",
                "micro_start_time",
                "micro_end_time",
                "micro_duration_ticks",
                "raw_label",
                "n_covering_windows",
                "x_t_json",
            ] + SIGNAL_COLS
            out_micro = OUTPUT_DIR / f"{out_prefix}__micro_331.csv"
            tick_df[micro_cols].to_csv(out_micro, index=False)
        else:
            out_micro = None

        out_ep = OUTPUT_DIR / f"{out_prefix}__episodes_331.csv"
        episodes.to_csv(out_ep, index=False)

        if SAVE_SUMMARY:
            out_sum = OUTPUT_DIR / f"{out_prefix}__summary_331.csv"
            summary.to_csv(out_sum, index=False)
        else:
            out_sum = None

        msg = f"[OK] {spec['input_file']} / {design_id} -> {out_ep.name}"
        if out_micro is not None:
            msg += f", {out_micro.name}"
        if out_sum is not None:
            msg += f", {out_sum.name}"
        print(msg)


# ==========================================
# Run all inputs
# ==========================================
shared_ticks = build_shared_tick_frame(SHARED_1HZ_PATH)

for spec in INPUT_SPECS:
    process_input_spec(spec, shared_ticks)

print(f"\nDone. Outputs saved to: {OUTPUT_DIR.resolve()}")

[INFO] Processing dataset_1Hz_windowed_5s_kmeans_selected.csv
[OK] dataset_1Hz_windowed_5s_kmeans_selected.csv / kmeans_5s_k4 -> kmeans_5s_k4__episodes_331.csv, kmeans_5s_k4__micro_331.csv, kmeans_5s_k4__summary_331.csv
[OK] dataset_1Hz_windowed_5s_kmeans_selected.csv / kmeans_5s_k7 -> kmeans_5s_k7__episodes_331.csv, kmeans_5s_k7__micro_331.csv, kmeans_5s_k7__summary_331.csv
[INFO] Processing dataset_1Hz_windowed_10s_kmeans_selected.csv
[OK] dataset_1Hz_windowed_10s_kmeans_selected.csv / kmeans_10s_k4 -> kmeans_10s_k4__episodes_331.csv, kmeans_10s_k4__micro_331.csv, kmeans_10s_k4__summary_331.csv
[OK] dataset_1Hz_windowed_10s_kmeans_selected.csv / kmeans_10s_k10 -> kmeans_10s_k10__episodes_331.csv, kmeans_10s_k10__micro_331.csv, kmeans_10s_k10__summary_331.csv
[INFO] Processing dataset_1Hz_windowed_15s_kmeans_selected.csv
[OK] dataset_1Hz_windowed_15s_kmeans_selected.csv / kmeans_15s_k4 -> kmeans_15s_k4__episodes_331.csv, kmeans_15s_k4__micro_331.csv, kmeans_15s_k4__summary_331.csv
[OK

### 3.3 Boundary stabilization

In [29]:
from pathlib import Path
from collections import Counter
import copy
import json
import math
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
STAGE3_DIR = Path("Stage3/stage3_331_outputs")
RAW_PATH = Path("data/dataset_1Hz_raw_uni_scaled.csv")
OUTPUT_DIR = STAGE3_DIR / "boundary_stabilized_scaled"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_BASENAMES = [
    "kmeans_5s_k4__episodes_331",
    "kmeans_5s_k7__episodes_331",
    "kmeans_10s_k4__episodes_331",
    "kmeans_10s_k10__episodes_331",
    "kmeans_15s_k4__episodes_331",
    "kmeans_15s_k9__episodes_331",
    "hdbscan_5s_mcs3000_ms200__episodes_331",
    "hdbscan_10s_mcs5000_ms50__episodes_331",
    "hdbscan_15s_mcs3000_ms200__episodes_331",
    "hmm_ns08_covdiag__episodes_331",
    "hmm_ns10_covdiag__episodes_331",
    "hmm_ns12_covdiag__episodes_331",
]

D_MIN_VALUES = [5, 10]
MAX_PASSES = 10
REASSIGN_RATIO_THRESHOLD = 0.90
DURATION_WEIGHT = 0.20
ALLOW_EDGE_REASSIGN = False
STRICT_INTERVAL_CHECK = True
CSV_SEP = ","

RAW_KEY_COLS = ["source_file", "valid_run_id", "session_id"]
SEQ_KEY_COLS = ["design_id", "source_file", "valid_run_id", "session_id"]
KEY_NA_TOKEN = "__NA__"

# ============================================================
# BASIC HELPERS
# ============================================================
def read_table_auto(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig")

def find_stage3_file(base_name: str) -> Path:
    for p in [STAGE3_DIR / f"{base_name}.csv", STAGE3_DIR / base_name]:
        if p.exists():
            return p
    raise FileNotFoundError(f"Stage3 file not found for: {base_name}")

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def normalize_label(x):
    return "__NA__" if pd.isna(x) else str(x)

def same_label(a, b):
    return normalize_label(a) == normalize_label(b)

def normalize_key_value(x):
    if pd.isna(x):
        return KEY_NA_TOKEN

    if isinstance(x, str):
        s = x.strip()
        if s == "":
            return KEY_NA_TOKEN
        try:
            f = float(s)
            if math.isfinite(f) and f.is_integer():
                return str(int(f))
        except Exception:
            pass
        return s

    if isinstance(x, (int, np.integer)):
        return str(int(x))

    if isinstance(x, (float, np.floating)):
        if not np.isfinite(x):
            return KEY_NA_TOKEN
        if float(x).is_integer():
            return str(int(x))
        return format(float(x), ".15g")

    return str(x)

def standardize_keys(df: pd.DataFrame, key_cols):
    out = df.copy()
    for c in key_cols:
        if c not in out.columns:
            raise ValueError(f"Missing key column: {c}")
        out[c] = out[c].map(normalize_key_value)
    return out

# ============================================================
# RAW SCALED 1 Hz DATA
# ============================================================
if not RAW_PATH.exists():
    raise FileNotFoundError(f"Raw scaled file not found: {RAW_PATH}")

RAW_DF = read_table_auto(RAW_PATH).copy()
RAW_DF.columns = [str(c).strip() for c in RAW_DF.columns]

required_raw = [
    "session_id", "source_file", "valid_run_id", "DateTime",
    "DriveStatus", "DoorStatusMain", "DoorStatusTooling", "CoolantFlow",
    "OverrideFeed", "FeedRate", "SpindleSpeed",
]
missing_raw = [c for c in required_raw if c not in RAW_DF.columns]
if missing_raw:
    raise ValueError(f"Missing raw columns in {RAW_PATH}: {missing_raw}")

RAW_DF = standardize_keys(RAW_DF, RAW_KEY_COLS)
RAW_DF["DateTime"] = pd.to_datetime(RAW_DF["DateTime"], errors="coerce")

if RAW_DF["DateTime"].isna().any():
    bad_n = int(RAW_DF["DateTime"].isna().sum())
    raise ValueError(f"{RAW_PATH} contains {bad_n} rows with unparsable DateTime.")

dup_mask = RAW_DF.duplicated(RAW_KEY_COLS + ["DateTime"], keep=False)
if dup_mask.any():
    dup_n = int(dup_mask.sum())
    raise ValueError(
       f"{RAW_PATH} contains {dup_n} duplicated timestamps within "
       f"{RAW_KEY_COLS} groups. This breaks deterministic tick mapping."
    )

RAW_DF = RAW_DF.sort_values(RAW_KEY_COLS + ["DateTime"]).reset_index(drop=True)
RAW_DF["__tick_in_group"] = RAW_DF.groupby(RAW_KEY_COLS, dropna=False).cumcount() + 1

RAW_GROUPS = {}
RAW_GROUP_LENGTHS = {}
for key, g in RAW_DF.groupby(RAW_KEY_COLS, sort=False, dropna=False):
    if not isinstance(key, tuple):
        key = (key,)
    g = g.reset_index(drop=True)
    RAW_GROUPS[key] = g
    RAW_GROUP_LENGTHS[key] = len(g)

def make_raw_group_key(obj):
    return tuple(obj[c] for c in RAW_KEY_COLS)

def make_seq_group_key(obj):
    return tuple(obj[c] for c in SEQ_KEY_COLS)

def get_raw_interval(ep_like, s_k, f_k):
    gkey = make_raw_group_key(ep_like)
    if gkey not in RAW_GROUPS:
        raise KeyError(f"Raw group not found: {gkey}")

    gdf = RAW_GROUPS[gkey]
    s_k = int(s_k)
    f_k = int(f_k)
    out = gdf[(gdf["__tick_in_group"] >= s_k) & (gdf["__tick_in_group"] <= f_k)].copy()

    expected = f_k - s_k + 1
    if STRICT_INTERVAL_CHECK and len(out) != expected:
        raise ValueError(
            f"Interval mismatch for group={gkey}, s_k={s_k}, f_k={f_k}: "
            f"expected {expected}, got {len(out)}"
        )
    return out

# ============================================================
# DESCRIPTOR SPEC
# ============================================================
def infer_summary_spec(df: pd.DataFrame):
    occ_cols = [c for c in df.columns if c.startswith("v_") and c.endswith("_occ")]
    med_cols = [c for c in df.columns if c.startswith("v_") and c.endswith("_median")]
    iqr_cols = [c for c in df.columns if c.startswith("v_") and c.endswith("_iqr")]

    occ_map = {c: c[len("v_"):-len("_occ")] for c in occ_cols}

    cont_map = {}
    for c in med_cols:
        raw_col = c[len("v_"):-len("_median")]
        cont_map.setdefault(raw_col, {})["median_col"] = c
    for c in iqr_cols:
        raw_col = c[len("v_"):-len("_iqr")]
        cont_map.setdefault(raw_col, {})["iqr_col"] = c

    return {
        "descriptor_cols": occ_cols + med_cols + iqr_cols,
        "occ_map": occ_map,
        "cont_map": cont_map,
    }

def coerce_episode_numeric(df: pd.DataFrame, descriptor_cols):
    num_cols = ["window_size_s", "s_k", "f_k", "d_k", "n_micro_episodes"] + descriptor_cols
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def to_binary_indicator(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(float)

    if pd.api.types.is_numeric_dtype(series):
        return (pd.to_numeric(series, errors="coerce") != 0).astype(float)

    s = series.astype(str).str.strip().str.lower()
    truthy = {"1", "true", "t", "yes", "y", "on", "open"}
    falsy = {"0", "false", "f", "no", "n", "off", "closed", "nan", "none", ""}
    out = pd.Series(np.nan, index=series.index, dtype=float)
    out[s.isin(truthy)] = 1.0
    out[s.isin(falsy)] = 0.0
    return out

def compute_exact_descriptor(raw_interval: pd.DataFrame, spec: dict) -> dict:
    out = {}

    for out_col, raw_col in spec["occ_map"].items():
        if raw_col not in raw_interval.columns:
            raise KeyError(f"Missing raw column '{raw_col}' required for '{out_col}'.")
        x = to_binary_indicator(raw_interval[raw_col])
        out[out_col] = float(np.nanmean(x)) if x.notna().any() else np.nan

    for raw_col, colmap in spec["cont_map"].items():
        if raw_col not in raw_interval.columns:
            raise KeyError(f"Missing raw column '{raw_col}' required for descriptor recomputation.")
        x = pd.to_numeric(raw_interval[raw_col], errors="coerce").to_numpy(dtype=float)

        if "median_col" in colmap:
            out[colmap["median_col"]] = float(np.nanmedian(x)) if np.isfinite(x).any() else np.nan
        if "iqr_col" in colmap:
            out[colmap["iqr_col"]] = (
                float(np.nanpercentile(x, 75) - np.nanpercentile(x, 25))
                if np.isfinite(x).any() else np.nan
            )

    return out

def build_v_json(ep, descriptor_cols):
    payload = {}
    for c in descriptor_cols:
        x = ep.get(c, np.nan)
        payload[c] = None if pd.isna(x) else float(x)
    return json.dumps(payload, ensure_ascii=False)

# ============================================================
# FEATURE SPACE FOR LOCAL COMPARISON
# ============================================================
def build_scaler(df: pd.DataFrame, descriptor_cols):
    feat = df[descriptor_cols].copy()
    med = feat.median(numeric_only=True)
    q1 = feat.quantile(0.25, numeric_only=True)
    q3 = feat.quantile(0.75, numeric_only=True)
    iqr = (q3 - q1).replace(0, 1.0).fillna(1.0)

    dur_log = np.log1p(df["d_k"].clip(lower=1).astype(float))
    dur_med = float(np.nanmedian(dur_log))
    dur_iqr = float(np.nanpercentile(dur_log, 75) - np.nanpercentile(dur_log, 25))
    if (not np.isfinite(dur_iqr)) or (dur_iqr == 0):
        dur_iqr = 1.0

    return {"med": med, "iqr": iqr, "dur_med": dur_med, "dur_iqr": dur_iqr}

def feature_vector(ep, descriptor_cols, scaler):
    vals = []
    for c in descriptor_cols:
        x = safe_float(ep.get(c, np.nan))
        m = safe_float(scaler["med"].get(c, 0.0))
        s = safe_float(scaler["iqr"].get(c, 1.0))
        if (not np.isfinite(x)) or (not np.isfinite(m)) or (not np.isfinite(s)) or s == 0:
            vals.append(np.nan)
        else:
            vals.append((x - m) / s)

    d = max(safe_float(ep.get("d_k", 1)), 1.0)
    dur_z = (math.log1p(d) - scaler["dur_med"]) / scaler["dur_iqr"]
    vals.append(DURATION_WEIGHT * dur_z)
    return np.array(vals, dtype=float)

def l2_dist(a, b):
    mask = np.isfinite(a) & np.isfinite(b)
    return np.inf if mask.sum() == 0 else float(np.linalg.norm(a[mask] - b[mask]))

# ============================================================
# EPISODE OBJECTS + CACHE
# ============================================================
def episode_from_row(row):
    ep = row.to_dict()
    ep["_members"] = [int(row.name)]
    ep["_source_episode_ids"] = [row["episode_id"]]
    return ep

def are_adjacent_in_time(left_ep, right_ep) -> bool:
    return int(left_ep["f_k"]) + 1 == int(right_ep["s_k"])

def attach_descriptor(ep, desc, descriptor_cols):
    for c in descriptor_cols:
        ep[c] = desc.get(c, np.nan)
    return ep

def make_descriptor_cache_key(base, s_k, f_k):
    return make_raw_group_key(base) + (int(s_k), int(f_k))

def rebuild_episode(member_ids, new_label, lookup, spec, descriptor_cache):
    sub = lookup.iloc[list(member_ids)].copy().sort_values(["s_k", "f_k"])
    base = sub.iloc[0].to_dict()

    for c in SEQ_KEY_COLS:
      if sub[c].nunique(dropna=False) != 1:
        raise ValueError(f"Inconsistent '{c}' inside merged episode.")
        
    s_k = int(sub["s_k"].min())
    f_k = int(sub["f_k"].max())
    cache_key = make_descriptor_cache_key(base, s_k, f_k)

    if cache_key in descriptor_cache:
        desc, start_time, end_time = descriptor_cache[cache_key]
    else:
        raw_interval = get_raw_interval(base, s_k, f_k)
        desc = compute_exact_descriptor(raw_interval, spec)
        start_time = raw_interval["DateTime"].iloc[0]
        end_time = raw_interval["DateTime"].iloc[-1]
        descriptor_cache[cache_key] = (desc, start_time, end_time)

    out = {}
    for c in ["design_id", "design_family", "input_file", "window_size_s", "session_id", "source_file", "valid_run_id"]:
        if c in base:
            out[c] = base[c]

    out["start_time"] = start_time
    out["end_time"] = end_time
    out["s_k"] = s_k
    out["f_k"] = f_k
    out["d_k"] = int(f_k - s_k + 1)
    out["n_micro_episodes"] = int(np.nansum(pd.to_numeric(sub["n_micro_episodes"], errors="coerce").fillna(0))) \
        if "n_micro_episodes" in sub.columns else len(member_ids)
    out["l_k"] = new_label
    out = attach_descriptor(out, desc, spec["descriptor_cols"])
    out["_members"] = list(member_ids)
    out["_source_episode_ids"] = sub["episode_id"].tolist()
    return out

def merge_pair(eps, idx_left, lookup, spec, descriptor_cache):
    left_ep = eps[idx_left]
    right_ep = eps[idx_left + 1]
    if not are_adjacent_in_time(left_ep, right_ep):
        raise ValueError(
            f"Attempted to merge non-adjacent episodes: "
            f"[{left_ep['s_k']}, {left_ep['f_k']}] and [{right_ep['s_k']}, {right_ep['f_k']}]"
        )
    member_ids = left_ep["_members"] + right_ep["_members"]
    new_ep = rebuild_episode(member_ids, left_ep["l_k"], lookup, spec, descriptor_cache)
    return eps[:idx_left] + [new_ep] + eps[idx_left + 2:]

def collapse_around_index(eps, idx, lookup, spec, descriptor_cache):
    if not eps:
        return eps, idx

    idx = max(0, min(idx, len(eps) - 1))
    changed = True

    while changed and len(eps) > 1:
        changed = False

        if idx > 0 and same_label(eps[idx - 1]["l_k"], eps[idx]["l_k"]):
            eps = merge_pair(eps, idx - 1, lookup, spec, descriptor_cache)
            idx = idx - 1
            changed = True
            continue

        if idx < len(eps) - 1 and same_label(eps[idx]["l_k"], eps[idx + 1]["l_k"]):
            eps = merge_pair(eps, idx, lookup, spec, descriptor_cache)
            changed = True
            continue

    return eps, idx

def collapse_all_initial(eps, lookup, spec, descriptor_cache):
    if not eps:
        return eps

    out = [eps[0]]
    for ep in eps[1:]:
        out.append(ep)
        out, _ = collapse_around_index(out, len(out) - 1, lookup, spec, descriptor_cache)
    return out

# ============================================================
# ONE PASS
# ============================================================
def apply_one_pass(eps, d_min, lookup, spec, scaler, descriptor_cache):
    eps = copy.deepcopy(eps)
    counts = Counter()
    changed = False

    i = 0
    while i < len(eps):
        ep = eps[i]
        d_k = safe_float(ep["d_k"])

        if (not np.isfinite(d_k)) or d_k >= d_min:
            i += 1
            continue

        left = eps[i - 1] if i > 0 else None
        right = eps[i + 1] if i + 1 < len(eps) else None

        # 1) Bridge rule
        if (
            left is not None and right is not None
            and are_adjacent_in_time(left, ep)
            and are_adjacent_in_time(ep, right)
            and same_label(left["l_k"], right["l_k"])
        ):
            new_ep = rebuild_episode(
                left["_members"] + ep["_members"] + right["_members"],
                left["l_k"],
                lookup,
                spec,
                descriptor_cache,
            )
            eps = eps[:i - 1] + [new_ep] + eps[i + 2:]
            i = max(i - 1, 0)
            eps, i = collapse_around_index(eps, i, lookup, spec, descriptor_cache)
            counts["bridge_merge"] += 1
            changed = True
            continue

        # 2) Local similarity rule
        if (
            left is not None and right is not None
            and are_adjacent_in_time(left, ep)
            and are_adjacent_in_time(ep, right)
        ):
            vk = feature_vector(ep, spec["descriptor_cols"], scaler)
            vl = feature_vector(left, spec["descriptor_cols"], scaler)
            vr = feature_vector(right, spec["descriptor_cols"], scaler)

            dist_left = l2_dist(vk, vl)
            dist_right = l2_dist(vk, vr)

            if dist_left <= dist_right:
                best_dist, other_dist, target_label, target_side = dist_left, dist_right, left["l_k"], "left"
            else:
                best_dist, other_dist, target_label, target_side = dist_right, dist_left, right["l_k"], "right"

            decisive = np.isfinite(best_dist) and (
                other_dist == np.inf or other_dist == 0 or (best_dist / other_dist) <= REASSIGN_RATIO_THRESHOLD
            )

            if decisive:
                eps[i]["l_k"] = target_label
                eps, i = collapse_around_index(eps, i, lookup, spec, descriptor_cache)
                counts[f"{target_side}_reassign"] += 1
                changed = True
                continue

            counts["retained_uncertain"] += 1
            i += 1
            continue

        # 3) Edge episode
        if ALLOW_EDGE_REASSIGN:
            target = left if left is not None else right
            if target is not None:
                eps[i]["l_k"] = target["l_k"]
                eps, i = collapse_around_index(eps, i, lookup, spec, descriptor_cache)
                counts["edge_reassign"] += 1
                changed = True
                continue

        counts["edge_retained"] += 1
        i += 1

    return eps, counts, changed

# ============================================================
# METRICS / EXPORT
# ============================================================
def compute_metrics(current_by_group, d_min):
    durations = []
    total_span = 0
    total_transitions = 0

    for eps in current_by_group.values():
        if not eps:
            continue
        eps = sorted(eps, key=lambda x: (safe_float(x["s_k"]), safe_float(x["f_k"])))
        total_span += int(max(e["f_k"] for e in eps) - min(e["s_k"] for e in eps) + 1)
        total_transitions += max(len(eps) - 1, 0)
        durations.extend([safe_float(e["d_k"]) for e in eps])

    if not durations:
        return {
            "n_groups": 0, "n_episodes": 0, "total_ticks": 0, "coverage": np.nan,
            "segments_per_1000_ticks": np.nan, "transitions_per_1000_ticks": np.nan,
            "median_d_k": np.nan, "mean_d_k": np.nan, "single_tick_share": np.nan,
            "short_share_lt_5": np.nan, "short_share_lt_10": np.nan, "short_share_lt_current_dmin": np.nan,
        }

    d = np.array(durations, dtype=float)
    total_ticks = float(np.nansum(d))

    return {
        "n_groups": int(len(current_by_group)),
        "n_episodes": int(len(d)),
        "total_ticks": total_ticks,
        "coverage": (total_ticks / total_span) if total_span > 0 else np.nan,
        "segments_per_1000_ticks": (1000.0 * len(d) / total_ticks) if total_ticks > 0 else np.nan,
        "transitions_per_1000_ticks": (1000.0 * total_transitions / total_ticks) if total_ticks > 0 else np.nan,
        "median_d_k": float(np.nanmedian(d)),
        "mean_d_k": float(np.nanmean(d)),
        "single_tick_share": float(np.mean(d == 1)),
        "short_share_lt_5": float(np.mean(d < 5)),
        "short_share_lt_10": float(np.mean(d < 10)),
        "short_share_lt_current_dmin": float(np.mean(d < d_min)),
    }

def flatten_final(current_by_group, spec, d_min, passes_used, dataset_name):
    rows = []
    global_id = 1

    for group_key in sorted(current_by_group.keys()):
        eps = sorted(current_by_group[group_key], key=lambda x: (safe_float(x["s_k"]), safe_float(x["f_k"])))
        for ep in eps:
            row = {k: v for k, v in ep.items() if not k.startswith("_")}
            row["episode_id"] = f"E{global_id:07d}"
            global_id += 1
            row["n_source_preliminary_episodes"] = len(ep["_members"])
            row["source_episode_ids_json"] = json.dumps(ep["_source_episode_ids"], ensure_ascii=False)
            row["stabilization_dmin"] = d_min
            row["stabilization_passes_used"] = passes_used
            row["dataset_name"] = dataset_name
            row["v_json"] = build_v_json(row, spec["descriptor_cols"])
            rows.append(row)

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    out = out.sort_values(SEQ_KEY_COLS + ["s_k", "f_k"]).reset_index(drop=True)

    ordered = []
    for c in [
        "design_id", "design_family", "input_file", "window_size_s",
        "session_id", "source_file", "valid_run_id", "episode_id",
        "start_time", "end_time", "s_k", "f_k", "d_k", "n_micro_episodes",
        "l_k", "v_json"
    ]:
        if c in out.columns and c not in ordered:
            ordered.append(c)

    for c in spec["descriptor_cols"]:
        if c in out.columns and c not in ordered:
            ordered.append(c)

    for c in out.columns:
        if c not in ordered:
            ordered.append(c)

    return out[ordered]

# ============================================================
# SANITY CHECKS AGAINST RAW
# ============================================================
def validate_stage3_against_raw(df, dataset_name):
    stage3_groups = 0

    for seq_key, g in df.groupby(SEQ_KEY_COLS, sort=False, dropna=False):
        if not isinstance(seq_key, tuple):
            seq_key = (seq_key,)
        stage3_groups += 1

        raw_key = tuple(g.iloc[0][c] for c in RAW_KEY_COLS)

        if raw_key not in RAW_GROUP_LENGTHS:
            raise KeyError(f"{dataset_name}: raw group {raw_key} not found in raw data.")

        g = g.sort_values(["s_k", "f_k"]).reset_index(drop=True)
        s = pd.to_numeric(g["s_k"], errors="coerce").to_numpy(dtype=float)
        f = pd.to_numeric(g["f_k"], errors="coerce").to_numpy(dtype=float)

        if np.isnan(s).any() or np.isnan(f).any():
            raise ValueError(f"{dataset_name}: group {seq_key} contains non-numeric s_k/f_k.")

        s = s.astype(int)
        f = f.astype(int)

        if s.min() < 1:
            raise ValueError(f"{dataset_name}: found s_k < 1 for group {seq_key}.")

        if f.max() > RAW_GROUP_LENGTHS[raw_key]:
            raise ValueError(
                f"{dataset_name}: max f_k={f.max()} exceeds raw length={RAW_GROUP_LENGTHS[raw_key]} "
                f"for group {seq_key}."
            )

        if np.any(f < s):
            raise ValueError(f"{dataset_name}: found episode with f_k < s_k in group {seq_key}.")

        if len(g) > 1:
            next_s = s[1:]
            prev_f = f[:-1]

            overlaps = np.where(next_s <= prev_f)[0]
            if len(overlaps) > 0:
                i = int(overlaps[0])
                raise ValueError(
                    f"{dataset_name}: overlapping episodes in group {seq_key}: "
                    f"[{s[i]}, {f[i]}] and [{s[i+1]}, {f[i+1]}]"
                )

            gaps = np.where(next_s != prev_f + 1)[0]
            if len(gaps) > 0:
                i = int(gaps[0])
                raise ValueError(
                    f"{dataset_name}: non-contiguous Stage3 sequence in group {seq_key}: "
                    f"[{s[i]}, {f[i]}] followed by [{s[i+1]}, {f[i+1]}]"
                )

    return stage3_groups
    
# ============================================================
# PER-DATASET RUNNER
# ============================================================
def process_dataset(input_path, d_min):
    dataset_name = input_path.stem

    df = read_table_auto(input_path).copy()
    df.columns = [str(c).strip() for c in df.columns]
    df = standardize_keys(df, RAW_KEY_COLS + ["design_id"])

    spec = infer_summary_spec(df)
    df = coerce_episode_numeric(df, spec["descriptor_cols"])

    required_stage3 = SEQ_KEY_COLS + ["episode_id", "s_k", "f_k", "d_k", "l_k"]
    missing_stage3 = [c for c in required_stage3 if c not in df.columns]
    if missing_stage3:
        raise ValueError(f"{dataset_name}: missing columns {missing_stage3}")

    df = df.sort_values(SEQ_KEY_COLS + ["s_k", "f_k"]).reset_index(drop=True)

    n_groups = validate_stage3_against_raw(df, dataset_name)

    lookup = df.copy()
    scaler = build_scaler(df, spec["descriptor_cols"])
    descriptor_cache = {}

    current_by_group = {}
    for group_key, g in df.groupby(SEQ_KEY_COLS, sort=False, dropna=False):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        eps = [episode_from_row(row) for _, row in g.iterrows()]
        eps = collapse_all_initial(eps, lookup, spec, descriptor_cache)
        current_by_group[group_key] = eps

    metrics_before = compute_metrics(current_by_group, d_min)
    action_totals = Counter()
    passes_used = 0

    for _ in range(MAX_PASSES):
        changed_any = False
        new_current = {}

        for group_key, eps in current_by_group.items():
            new_eps, counts, changed = apply_one_pass(
                eps=eps,
                d_min=d_min,
                lookup=lookup,
                spec=spec,
                scaler=scaler,
                descriptor_cache=descriptor_cache,
            )
            new_current[group_key] = new_eps
            action_totals.update(counts)
            changed_any = changed_any or changed

        if changed_any:
            passes_used += 1

        current_by_group = new_current

        if not changed_any:
            break

    metrics_after = compute_metrics(current_by_group, d_min)
    final_df = flatten_final(current_by_group, spec, d_min, passes_used, dataset_name)

    stem = f"{dataset_name}__stabilized_scaled_dmin{d_min}"
    final_df.to_csv(OUTPUT_DIR / f"{stem}.csv", index=False, sep=CSV_SEP)

    return pd.DataFrame([{
        "dataset_name": dataset_name,
        "raw_file_used": str(RAW_PATH),
        "d_min": d_min,
        "n_groups": n_groups,
        "passes_used": passes_used,
        "ratio_threshold": REASSIGN_RATIO_THRESHOLD,
        "duration_weight": DURATION_WEIGHT,
        "episodes_before": metrics_before["n_episodes"],
        "episodes_after": metrics_after["n_episodes"],
        "episodes_delta": metrics_after["n_episodes"] - metrics_before["n_episodes"],
        "median_d_before": metrics_before["median_d_k"],
        "median_d_after": metrics_after["median_d_k"],
        "single_tick_before": metrics_before["single_tick_share"],
        "single_tick_after": metrics_after["single_tick_share"],
        "short_lt_5_before": metrics_before["short_share_lt_5"],
        "short_lt_5_after": metrics_after["short_share_lt_5"],
        "short_lt_10_before": metrics_before["short_share_lt_10"],
        "short_lt_10_after": metrics_after["short_share_lt_10"],
        "segments_per_1000_before": metrics_before["segments_per_1000_ticks"],
        "segments_per_1000_after": metrics_after["segments_per_1000_ticks"],
        "transitions_per_1000_before": metrics_before["transitions_per_1000_ticks"],
        "transitions_per_1000_after": metrics_after["transitions_per_1000_ticks"],
        "coverage_before": metrics_before["coverage"],
        "coverage_after": metrics_after["coverage"],
        "bridge_merges": int(action_totals.get("bridge_merge", 0)),
        "left_reassigns": int(action_totals.get("left_reassign", 0)),
        "right_reassigns": int(action_totals.get("right_reassign", 0)),
        "retained_uncertain": int(action_totals.get("retained_uncertain", 0)),
        "edge_retained": int(action_totals.get("edge_retained", 0)),
    }])

# ============================================================
# RUN EVERYTHING
# ============================================================
def run_all():
    print(f"RAW FILE USED: {RAW_PATH}")

    all_comparisons = []

    for base_name in INPUT_BASENAMES:
        input_path = find_stage3_file(base_name)
        for d_min in D_MIN_VALUES:
            cmp_df = process_dataset(input_path, d_min)
            all_comparisons.append(cmp_df)

    comparison_df = pd.concat(all_comparisons, ignore_index=True)
    comparison_df.to_csv(
        OUTPUT_DIR / "ALL_DATASETS__stabilization_comparison.csv",
        index=False,
        sep=CSV_SEP,
    )

    print("DONE")

run_all()

RAW FILE USED: data\dataset_1Hz_raw_uni_scaled.csv
DONE


### 3.4 Raw-label prototypes

In [30]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
STABILIZED_DIR = Path("Stage3") / "stage3_331_outputs/boundary_stabilized_scaled" 
OUTPUT_DIR = Path("Stage3") / "raw_label_prototypes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_BASENAMES = [
    "kmeans_5s_k4__episodes_331",
    "kmeans_5s_k7__episodes_331",
    "kmeans_10s_k4__episodes_331",
    "kmeans_10s_k10__episodes_331",
    "kmeans_15s_k4__episodes_331",
    "kmeans_15s_k9__episodes_331",
    "hdbscan_5s_mcs3000_ms200__episodes_331",
    "hdbscan_10s_mcs5000_ms50__episodes_331",
    "hdbscan_15s_mcs3000_ms200__episodes_331",
    "hmm_ns08_covdiag__episodes_331",
    "hmm_ns10_covdiag__episodes_331",
    "hmm_ns12_covdiag__episodes_331",
]

D_MIN_VALUES = [5, 10]
CSV_SEP = ","

META_COLS = [
    "design_id",
    "design_family",
    "input_file",
    "window_size_s",
]

REQUIRED_COLS = [
    "design_id",
    "l_k",
    "d_k",
]


# ============================================================
# HELPERS
# ============================================================
def find_stabilized_file(base_name: str, d_min: int) -> Path:
    filename = f"{base_name}__stabilized_scaled_dmin{d_min}.csv"
    path = STABILIZED_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing stabilized file: {path}")
    return path


def infer_descriptor_cols(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if c.startswith("v_") and c != "v_json"]


def iqr_series(s: pd.Series) -> float:
    vals = pd.to_numeric(s, errors="coerce").dropna().to_numpy(dtype=float)
    if len(vals) == 0:
        return np.nan
    return float(np.percentile(vals, 75) - np.percentile(vals, 25))


def json_safe(x):
    if pd.isna(x):
        return None
    if isinstance(x, (np.floating, float)):
        return float(x)
    if isinstance(x, (np.integer, int)):
        return int(x)
    return x


# ============================================================
# PROTOTYPE BUILDING
# ============================================================
def build_prototypes(df: pd.DataFrame, descriptor_cols: list[str], d_min: int) -> pd.DataFrame:
    rows = []

    for label, g in df.groupby("l_k", dropna=False, sort=True):
        row = {}

        # metadata: each file should contain exactly one design
        for c in META_COLS:
            row[c] = g[c].iloc[0] if c in g.columns else np.nan

        row["stabilization_dmin"] = d_min
        row["raw_label"] = label
        row["n_episodes"] = int(len(g))

        d = pd.to_numeric(g["d_k"], errors="coerce")
        row["median_duration_ticks"] = float(d.median()) if d.notna().any() else np.nan
        row["iqr_duration_ticks"] = iqr_series(d)

        proto_payload = {}

        for c in descriptor_cols:
            vals = pd.to_numeric(g[c], errors="coerce")
            med = float(vals.median()) if vals.notna().any() else np.nan
            iqr = iqr_series(vals)

            row[f"p_{c}_median"] = med
            row[f"p_{c}_iqr"] = iqr

            proto_payload[c] = {
                "median": json_safe(med),
                "iqr": json_safe(iqr),
            }

        row["prototype_json"] = json.dumps(proto_payload, ensure_ascii=False)
        rows.append(row)

    out = pd.DataFrame(rows)

    if out.empty:
        return out

    ordered = [
        "design_id",
        "design_family",
        "input_file",
        "window_size_s",
        "stabilization_dmin",
        "raw_label",
        "n_episodes",
        "median_duration_ticks",
        "iqr_duration_ticks",
        "prototype_json",
    ]

    extra_cols = [c for c in out.columns if c not in ordered]
    return out[ordered + extra_cols]


def build_label_summary(df: pd.DataFrame, d_min: int) -> pd.DataFrame:
    counts = (
        df.groupby("l_k", dropna=False)
          .size()
          .reset_index(name="n_episodes")
          .rename(columns={"l_k": "raw_label"})
    )

    row = {
        "design_id": df["design_id"].iloc[0],
        "design_family": df["design_family"].iloc[0] if "design_family" in df.columns else np.nan,
        "input_file": df["input_file"].iloc[0] if "input_file" in df.columns else np.nan,
        "window_size_s": df["window_size_s"].iloc[0] if "window_size_s" in df.columns else np.nan,
        "stabilization_dmin": d_min,
        "n_labels": int(len(counts)),
        "n_episodes_total": int(len(df)),
        "median_episodes_per_label": float(counts["n_episodes"].median()) if not counts.empty else np.nan,
        "min_episodes_per_label": int(counts["n_episodes"].min()) if not counts.empty else np.nan,
        "max_episodes_per_label": int(counts["n_episodes"].max()) if not counts.empty else np.nan,
    }

    return pd.DataFrame([row])


# ============================================================
# PER-FILE PROCESSING
# ============================================================
def process_file(base_name: str, d_min: int):
    path = find_stabilized_file(base_name, d_min)
    df = pd.read_csv(path, sep=CSV_SEP)
    df.columns = [str(c).strip() for c in df.columns]

    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name}: missing required columns {missing}")

    unique_designs = sorted(df["design_id"].dropna().astype(str).unique().tolist())
    if len(unique_designs) != 1:
        raise ValueError(
            f"{path.name}: expected exactly one design_id, found {len(unique_designs)}: {unique_designs}"
        )

    descriptor_cols = infer_descriptor_cols(df)
    if not descriptor_cols:
        raise ValueError(f"{path.name}: no descriptor columns 'v_*' found")

    num_cols = ["d_k", "window_size_s"] + descriptor_cols
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    prototypes = build_prototypes(df, descriptor_cols, d_min)
    label_summary = build_label_summary(df, d_min)

    out_prefix = f"{base_name}__dmin{d_min}"

    out_proto = OUTPUT_DIR / f"{out_prefix}__prototypes.csv"
    out_sum = OUTPUT_DIR / f"{out_prefix}__label_summary.csv"

    prototypes.to_csv(out_proto, index=False, sep=CSV_SEP)
    label_summary.to_csv(out_sum, index=False, sep=CSV_SEP)

    print(f"[OK] {path.name} -> {out_proto.name}, {out_sum.name}")

    return prototypes, label_summary


# ============================================================
# RUN ALL
# ============================================================
def run_all():
    all_proto = []
    all_sum = []

    for base_name in INPUT_BASENAMES:
        for d_min in D_MIN_VALUES:
            proto_df, sum_df = process_file(base_name, d_min)
            all_proto.append(proto_df)
            all_sum.append(sum_df)

    all_proto_df = pd.concat(all_proto, ignore_index=True)
    all_sum_df = pd.concat(all_sum, ignore_index=True)

    all_proto_df.to_csv(
        OUTPUT_DIR / "ALL_DESIGNS__raw_label_prototypes.csv",
        index=False,
        sep=CSV_SEP,
    )
    all_sum_df.to_csv(
        OUTPUT_DIR / "ALL_DESIGNS__label_summary.csv",
        index=False,
        sep=CSV_SEP,
    )

    print(f"DONE. Outputs saved to: {OUTPUT_DIR.resolve()}")


run_all()

[OK] kmeans_5s_k4__episodes_331__stabilized_scaled_dmin5.csv -> kmeans_5s_k4__episodes_331__dmin5__prototypes.csv, kmeans_5s_k4__episodes_331__dmin5__label_summary.csv
[OK] kmeans_5s_k4__episodes_331__stabilized_scaled_dmin10.csv -> kmeans_5s_k4__episodes_331__dmin10__prototypes.csv, kmeans_5s_k4__episodes_331__dmin10__label_summary.csv
[OK] kmeans_5s_k7__episodes_331__stabilized_scaled_dmin5.csv -> kmeans_5s_k7__episodes_331__dmin5__prototypes.csv, kmeans_5s_k7__episodes_331__dmin5__label_summary.csv
[OK] kmeans_5s_k7__episodes_331__stabilized_scaled_dmin10.csv -> kmeans_5s_k7__episodes_331__dmin10__prototypes.csv, kmeans_5s_k7__episodes_331__dmin10__label_summary.csv
[OK] kmeans_10s_k4__episodes_331__stabilized_scaled_dmin5.csv -> kmeans_10s_k4__episodes_331__dmin5__prototypes.csv, kmeans_10s_k4__episodes_331__dmin5__label_summary.csv
[OK] kmeans_10s_k4__episodes_331__stabilized_scaled_dmin10.csv -> kmeans_10s_k4__episodes_331__dmin10__prototypes.csv, kmeans_10s_k4__episodes_331__dmi

### 3.5 Ward alignment in shared prototype space

In [38]:
# ============================================
# STAGE 3A: WARD OVER RAW-LABEL PROTOTYPES
# pre jednu vetvu stabilization_dmin
# ============================================

from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler


# =========================================================
# CONFIG
# =========================================================

PROTOTYPES_PATH = Path("Stage3") / "raw_label_prototypes" / "ALL_DESIGNS__raw_label_prototypes.csv"
OUT_DIR = Path("Stage3") / "stage3_ward"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DMIN_VALUES = [5, 10]

KMEANS_DESIGNS = [
    "kmeans_5s_k4",
    "kmeans_5s_k7",
    "kmeans_10s_k4",
    "kmeans_10s_k10",
    "kmeans_15s_k4",
    "kmeans_15s_k9",
]

HDBSCAN_DESIGNS = [
    "hdbscan_5s_mcs3000_ms200",
    "hdbscan_10s_mcs5000_ms50",
    "hdbscan_15s_mcs3000_ms200",
]

HMM_DESIGNS = [
    "hmm_ns08_covdiag",
    "hmm_ns10_covdiag",
    "hmm_ns12_covdiag",
]

Q_GRID = [4, 5, 6, 7, 8]
RE_STANDARDIZE_FOR_WARD = True

DESIGN_COL = "design_id"
FAMILY_COL = "design_family"
LABEL_COL = "raw_label"
DMIN_COL = "stabilization_dmin"


def derive_family(design_id):
    s = str(design_id).lower()
    if s.startswith("kmeans"):
        return "kmeans"
    if s.startswith("hdbscan"):
        return "hdbscan"
    if s.startswith("hmm"):
        return "hmm"
    return "other"

def make_triplet_id(km, hdb, hmm):
    return f"{km}__{hdb}__{hmm}"

def run_ward(X, n_clusters):
    model = AgglomerativeClustering(n_clusters=n_clusters, linkage="ward")
    return model.fit_predict(X)

def cluster_summary(map_df):
    fam_per_meta = map_df.groupby("meta_mode")[FAMILY_COL].nunique()
    size_per_meta = map_df.groupby("meta_mode").size()
    return {
        "min_cluster_size": int(size_per_meta.min()),
        "max_cluster_size": int(size_per_meta.max()),
        "mean_cluster_size": float(size_per_meta.mean()),
        "meta_with_1_family": int((fam_per_meta == 1).sum()),
        "meta_with_2_families": int((fam_per_meta == 2).sum()),
        "meta_with_3_families": int((fam_per_meta >= 3).sum()),
        "singleton_meta": int((size_per_meta == 1).sum()),
    }


proto = pd.read_csv(PROTOTYPES_PATH)

required = [DESIGN_COL, LABEL_COL, DMIN_COL]
missing = [c for c in required if c not in proto.columns]
if missing:
    raise ValueError("Missing required columns: " + str(missing))

if FAMILY_COL not in proto.columns:
    proto[FAMILY_COL] = proto[DESIGN_COL].map(derive_family)

FEATURE_COLS = [c for c in proto.columns if c.startswith("p_v_")]
if not FEATURE_COLS:
    raise ValueError("No prototype feature columns starting with 'p_v_' were found.")

zero_var = [c for c in FEATURE_COLS if proto[c].nunique(dropna=False) <= 1]
FEATURE_COLS = [c for c in FEATURE_COLS if c not in zero_var]

if not FEATURE_COLS:
    raise ValueError("No usable p_v_ feature columns remain.")

if proto[FEATURE_COLS].isna().any().any():
    bad = proto[FEATURE_COLS].isna().sum()
    bad = bad[bad > 0]
    raise ValueError("Missing values in p_v_ feature columns:\n" + str(bad))


triplets = []
for km, hdb, hmm in product(KMEANS_DESIGNS, HDBSCAN_DESIGNS, HMM_DESIGNS):
    triplets.append({
        "triplet_id": make_triplet_id(km, hdb, hmm),
        "triplet_kmeans": km,
        "triplet_hdbscan": hdb,
        "triplet_hmm": hmm,
        "designs": [km, hdb, hmm],
    })

all_runs = []
all_maps = []

for dmin in DMIN_VALUES:
    proto_d = proto.loc[proto[DMIN_COL] == dmin].copy()

    retained_designs = KMEANS_DESIGNS + HDBSCAN_DESIGNS + HMM_DESIGNS
    proto_d = proto_d.loc[proto_d[DESIGN_COL].isin(retained_designs)].copy()

    print(f"[INFO] d_min={dmin}, rows after filters: {len(proto_d)}")

    dup_mask = proto_d.duplicated(subset=[DESIGN_COL, LABEL_COL], keep=False)
    if dup_mask.any():
        print(proto_d.loc[dup_mask, [DESIGN_COL, LABEL_COL, DMIN_COL]].sort_values([DESIGN_COL, LABEL_COL]).head(30))
        raise ValueError(f"Duplicate rows remain for d_min={dmin}")

    for trip in triplets:
        triplet_id = trip["triplet_id"]
        chosen_designs = trip["designs"]

        sub = proto_d.loc[proto_d[DESIGN_COL].isin(chosen_designs)].copy()
        sub["prototype_id"] = sub[DESIGN_COL].astype(str) + "::" + sub[LABEL_COL].astype(str)

        n_proto = len(sub)
        if n_proto < 3:
            continue

        valid_q = [q for q in Q_GRID if 2 <= q < n_proto]
        if not valid_q:
            continue

        X = sub[FEATURE_COLS].to_numpy(dtype=float)
        if RE_STANDARDIZE_FOR_WARD:
            X = StandardScaler().fit_transform(X)

        for q in valid_q:
            ward_labels = run_ward(X, n_clusters=q)

            map_df = sub[[DESIGN_COL, FAMILY_COL, LABEL_COL, "prototype_id"]].copy()
            map_df["triplet_id"] = triplet_id
            map_df["triplet_kmeans"] = trip["triplet_kmeans"]
            map_df["triplet_hdbscan"] = trip["triplet_hdbscan"]
            map_df["triplet_hmm"] = trip["triplet_hmm"]
            map_df["stabilization_dmin"] = dmin
            map_df["q"] = q
            map_df["meta_mode"] = ward_labels.astype(int)

            summary = cluster_summary(map_df)

            all_runs.append({
                "triplet_id": triplet_id,
                "triplet_kmeans": trip["triplet_kmeans"],
                "triplet_hdbscan": trip["triplet_hdbscan"],
                "triplet_hmm": trip["triplet_hmm"],
                "stabilization_dmin": dmin,
                "q": q,
                "n_prototypes": int(n_proto),
                "n_meta_modes": int(q),
                **summary,
            })

            all_maps.append(map_df)

runs_df = pd.DataFrame(all_runs).sort_values(
    ["stabilization_dmin", "triplet_id", "q"]
).reset_index(drop=True)

maps_df = pd.concat(all_maps, ignore_index=True).rename(columns={
    DESIGN_COL: "design_id",
    FAMILY_COL: "design_family",
    LABEL_COL: "raw_label",
})

maps_df = maps_df.sort_values(
    ["stabilization_dmin", "triplet_id", "q", "design_family", "design_id", "raw_label"]
).reset_index(drop=True)

runs_df.to_csv(OUT_DIR / "stage3_ward_runs_all_dmin.csv", index=False)
maps_df.to_csv(OUT_DIR / "stage3_label_to_meta_map_all_dmin.csv", index=False)

runs_df.loc[runs_df["stabilization_dmin"] == 5].to_csv(
    OUT_DIR / "stage3_ward_runs_dmin5.csv", index=False
)
runs_df.loc[runs_df["stabilization_dmin"] == 10].to_csv(
    OUT_DIR / "stage3_ward_runs_dmin10.csv", index=False
)

maps_df.loc[maps_df["stabilization_dmin"] == 5].to_csv(
    OUT_DIR / "stage3_label_to_meta_map_dmin5.csv", index=False
)
maps_df.loc[maps_df["stabilization_dmin"] == 10].to_csv(
    OUT_DIR / "stage3_label_to_meta_map_dmin10.csv", index=False
)

print("[OK] Saved all outputs to:", OUT_DIR.resolve())

[INFO] d_min=5, rows after filters: 123
[INFO] d_min=10, rows after filters: 123
[OK] Saved all outputs to: D:\zero-to-mastery\sle\Stage3\stage3_ward


## Stage 4 — Cross-design episodic evaluation

### 4.1 Map stabilized episode files to shared meta-modes

In [11]:
from pathlib import Path
import pandas as pd
import re

# =========================================================
# SETTINGS
# =========================================================
MAP_DIR = Path("Stage4")
EPISODE_DIR = Path("Stage3") / "stage3_331_outputs" / "boundary_stabilized_scaled"
MAP_FILE = Path("Stage3") / "stage3_ward" / "stage3_label_to_meta_map_all_dmin.csv"
OUT_DIR = MAP_DIR / "stage4_metamode_mapped"
OUT_DIR.mkdir(parents=True, exist_ok=True)

"""

TARGET = {
    "episode_files": [
        "kmeans_10s_k10__episodes_331__stabilized_scaled_dmin5.csv",
        "hdbscan_15s_mcs3000_ms200__episodes_331__stabilized_scaled_dmin5.csv",
        "hmm_ns12_covdiag__episodes_331__stabilized_scaled_dmin5.csv",
    ],
    "q": 5,
    "stabilization_dmin": 5,    
}

meta_id = 1
##-------------------------------------------------------
TARGET = {
    "episode_files": [
        "kmeans_5s_k7__episodes_331__stabilized_scaled_dmin5.csv",
        "hdbscan_5s_mcs3000_ms200__episodes_331__stabilized_scaled_dmin5.csv",
        "hmm_ns12_covdiag__episodes_331__stabilized_scaled_dmin5.csv",
    ],
    "q": 6,
    "stabilization_dmin": 5,    
}

meta_id = 2

##-------------------------------------------------------
TARGET = {
    "episode_files": [
        "kmeans_15s_k4__episodes_331__stabilized_scaled_dmin5.csv",
        "hdbscan_5s_mcs3000_ms200__episodes_331__stabilized_scaled_dmin5.csv",
        "hmm_ns12_covdiag__episodes_331__stabilized_scaled_dmin5.csv",
    ],
    "q": 5,
    "stabilization_dmin": 5,    
}

meta_id = 3

##-------------------------------------------------------
TARGET = {
    "episode_files": [
        "kmeans_10s_k10__episodes_331__stabilized_scaled_dmin10.csv",
        "hdbscan_5s_mcs3000_ms200__episodes_331__stabilized_scaled_dmin10.csv",
        "hmm_ns12_covdiag__episodes_331__stabilized_scaled_dmin10.csv",
    ],
    "q": 5,
    "stabilization_dmin": 10,    
}

meta_id = 4
"""
##-------------------------------------------------------
TARGET = {
    "episode_files": [
        "kmeans_15s_k9__episodes_331__stabilized_scaled_dmin10.csv",
        "hdbscan_5s_mcs3000_ms200__episodes_331__stabilized_scaled_dmin10.csv",
        "hmm_ns12_covdiag__episodes_331__stabilized_scaled_dmin10.csv",
    ],
    "q": 5,
    "stabilization_dmin": 10,    
}

meta_id = 5



# =========================================================
# HELPERS
# =========================================================
def normalize_label(x):
    if pd.isna(x):
        return None
    s = str(x).strip().replace(",", ".")
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
        return format(f, "g")
    except Exception:
        return s

# =========================================================
# 1) LOAD META-MODE MAP
# =========================================================
map_df = pd.read_csv(MAP_FILE)
map_df["q"] = pd.to_numeric(map_df["q"], errors="coerce").astype("Int64")
map_df["stabilization_dmin"] = pd.to_numeric(map_df["stabilization_dmin"], errors="coerce").astype("Int64")
map_df["raw_label_key"] = map_df["raw_label"].map(normalize_label)

# =========================================================
# 2) LOAD TARGET EPISODE FILES AND EXTRACT DESIGN IDS
# =========================================================
episode_paths = [EPISODE_DIR / fname for fname in TARGET["episode_files"]]

episode_dfs = {}
design_ids = []

for path in episode_paths:
    df = pd.read_csv(path)
    if "design_id" not in df.columns:
        raise ValueError(f"Missing 'design_id' column in {path.name}")
    design_id = str(df["design_id"].iloc[0])

    episode_dfs[design_id] = {
        "path": path,
        "df": df
    }
    design_ids.append(design_id)

if len(design_ids) != 3:
    raise ValueError("Exactly three episode files are required.")

# Identify the three design roles from design_id prefix
kmeans_design = next((d for d in design_ids if d.startswith("kmeans_")), None)
hdbscan_design = next((d for d in design_ids if d.startswith("hdbscan_")), None)
hmm_design = next((d for d in design_ids if d.startswith("hmm_")), None)

if not all([kmeans_design, hdbscan_design, hmm_design]):
    raise ValueError(
        f"Could not identify one kmeans, one hdbscan, and one hmm design from: {design_ids}"
    )

# =========================================================
# 3) SELECT THE CORRECT TRIPLET FROM THE MAP FILE
# =========================================================
triplet_map = map_df.loc[
    (map_df["triplet_kmeans"] == kmeans_design) &
    (map_df["triplet_hdbscan"] == hdbscan_design) &
    (map_df["triplet_hmm"] == hmm_design) &
    (map_df["q"] == int(TARGET["q"])) &
    (map_df["stabilization_dmin"] == int(TARGET["stabilization_dmin"]))
].copy()

if triplet_map.empty:
    raise ValueError(
        "No matching rows found in the meta-mode map for the selected three files, q, and dmin."
    )

triplet_id = str(triplet_map["triplet_id"].iloc[0])
q_val = int(triplet_map["q"].iloc[0])
dmin_val = int(triplet_map["stabilization_dmin"].iloc[0])

print("Selected triplet:", triplet_id)
print("kmeans :", kmeans_design)
print("hdbscan:", hdbscan_design)
print("hmm    :", hmm_design)
print("q      :", q_val)
print("dmin   :", dmin_val)

# =========================================================
# 4) MAP META-MODES TO EACH EPISODE FILE
# =========================================================
all_mapped = []
saved_files = []

for design_id, item in episode_dfs.items():
    epi_df = item["df"].copy()
    episode_file = item["path"]

    epi_df["raw_label_key"] = epi_df["l_k"].map(normalize_label)

    design_map = triplet_map.loc[
        triplet_map["design_id"] == design_id,
        ["design_id", "raw_label", "raw_label_key", "meta_mode", "prototype_id", "triplet_id", "q", "stabilization_dmin"]
    ].drop_duplicates(subset=["design_id", "raw_label_key"]).copy()

    merged = epi_df.merge(
        design_map,
        on=["design_id", "raw_label_key"],
        how="left",
        validate="many_to_one"
    )

    missing_labels = sorted(
        merged.loc[merged["meta_mode"].isna(), "raw_label_key"]
        .dropna()
        .unique()
        .tolist()
    )
    if missing_labels:
        raise ValueError(
            f"Missing meta-mode mapping for design '{design_id}' and labels: {missing_labels}"
        )

    merged = merged.rename(columns={"raw_label": "mapped_raw_label"})
    merged["meta_mode"] = pd.to_numeric(merged["meta_mode"], errors="coerce").astype("Int64")

    preferred_cols = [
        "design_id", "design_family", "input_file", "window_size_s",
        "session_id", "source_file", "valid_run_id", "episode_id",
        "start_time", "end_time", "s_k", "f_k", "d_k",
        "n_micro_episodes", "l_k", "mapped_raw_label", "meta_mode",
        "prototype_id", "triplet_id", "q", "stabilization_dmin"
    ]
    other_cols = [c for c in merged.columns if c not in preferred_cols and c != "raw_label_key"]
    merged = merged[[c for c in preferred_cols if c in merged.columns] + other_cols]

    out_name = f"{meta_id}_{episode_file.stem}__metamode_q{q_val}_dmin{dmin_val}.csv"
    out_path = OUT_DIR / out_name
    merged.to_csv(out_path, index=False)

    saved_files.append(out_path)
    all_mapped.append(merged)

    print(f"Saved: {out_path.name}")


Selected triplet: kmeans_15s_k9__hdbscan_5s_mcs3000_ms200__hmm_ns12_covdiag
kmeans : kmeans_15s_k9
hdbscan: hdbscan_5s_mcs3000_ms200
hmm    : hmm_ns12_covdiag
q      : 5
dmin   : 10
Saved: 5_kmeans_15s_k9__episodes_331_dmin10__metamode_q5_dmin10.csv
Saved: 5_hdbscan_5s_mcs3000_ms200_dmin10__metamode_q5_dmin10.csv
Saved: 5_hmm_ns12_covdiag__episodes_331_dmin10__metamode_q5_dmin10.csv


### 4.2 Build time-wise meta-mode sequences

In [14]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

# =========================================================
# SETTINGS
# =========================================================
INPUT_DIR = Path("Stage4/stage4_metamode_mapped")
OUTPUT_DIR = Path("Stage4/stage4_timewise_sequences")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# HELPERS
# =========================================================
def get_set_id(filename):
    m = re.match(r"^(\d+)_", filename)
    if not m:
        raise ValueError(f"Cannot extract set id from file name: {filename}")
    return m.group(1)

def get_family(df, filename):
    if "design_family" in df.columns and df["design_family"].notna().any():
        return str(df["design_family"].dropna().iloc[0]).strip().lower()
    name = filename.lower()
    if "kmeans" in name:
        return "kmeans"
    if "hdbscan" in name:
        return "hdbscan"
    if "hmm" in name:
        return "hmm"
    raise ValueError(f"Cannot determine design family for file: {filename}")

def parse_datetime_column(s):
    # First try ISO format
    dt = pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S", errors="coerce")
    # Fallback to day-first format if needed
    mask = dt.isna()
    if mask.any():
        dt.loc[mask] = pd.to_datetime(s.loc[mask].astype(str), format="%d/%m/%Y %H:%M", errors="coerce")
    return dt

def expand_episodes_to_ticks(df, family, set_id):
    required_cols = [
        "session_id", "source_file", "valid_run_id",
        "start_time", "s_k", "f_k", "meta_mode"
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns for {family}: {missing}")

    tmp = df.copy()

    tmp["start_time"] = parse_datetime_column(tmp["start_time"])
    if tmp["start_time"].isna().any():
        bad_n = int(tmp["start_time"].isna().sum())
        raise ValueError(f"{family}: {bad_n} rows have invalid start_time.")

    tmp["s_k"] = pd.to_numeric(tmp["s_k"], errors="coerce").astype(int)
    tmp["f_k"] = pd.to_numeric(tmp["f_k"], errors="coerce").astype(int)
    tmp["meta_mode"] = pd.to_numeric(tmp["meta_mode"], errors="coerce").astype("Int64")

    if (tmp["f_k"] < tmp["s_k"]).any():
        bad = tmp.loc[tmp["f_k"] < tmp["s_k"], ["session_id", "source_file", "valid_run_id", "s_k", "f_k"]]
        raise ValueError(f"{family}: found rows with f_k < s_k.\n{bad.head()}")

    expanded_parts = []

    for _, row in tmp.iterrows():
        ticks = np.arange(row["s_k"], row["f_k"] + 1, dtype=int)
        n = len(ticks)

        part = pd.DataFrame({
            "set_id": set_id,
            "session_id": row["session_id"],
            "source_file": row["source_file"],
            "valid_run_id": row["valid_run_id"],
            "tick": ticks,
            f"time_{family}": row["start_time"] + pd.to_timedelta(np.arange(n), unit="s"),
            family: row["meta_mode"]
        })
        expanded_parts.append(part)

    out = pd.concat(expanded_parts, ignore_index=True)

    # Safety check: no duplicate tick within one family/run
    dup_mask = out.duplicated(subset=["session_id", "source_file", "valid_run_id", "tick"], keep=False)
    if dup_mask.any():
        bad = out.loc[dup_mask, ["session_id", "source_file", "valid_run_id", "tick"]].drop_duplicates()
        raise ValueError(
            f"{family}: duplicate tick rows detected after expansion. Example:\n{bad.head(10)}"
        )

    return out

def coalesce_times(df):
    time_cols = [c for c in ["time_kmeans", "time_hdbscan", "time_hmm"] if c in df.columns]

    # Optional consistency check: if multiple time columns are present, they should match
    for i in range(len(time_cols)):
        for j in range(i + 1, len(time_cols)):
            c1, c2 = time_cols[i], time_cols[j]
            mask = df[c1].notna() & df[c2].notna() & (df[c1] != df[c2])
            if mask.any():
                bad = df.loc[mask, ["session_id", "source_file", "valid_run_id", "tick", c1, c2]].head(10)
                raise ValueError(f"Time mismatch between {c1} and {c2}. Example rows:\n{bad}")

    # Coalesce to one time column
    df["time"] = pd.NaT
    for c in time_cols:
        df["time"] = df["time"].fillna(df[c])

    return df

# =========================================================
# LOAD FILES
# =========================================================
files = sorted(INPUT_DIR.glob("*.csv"))
if not files:
    raise FileNotFoundError(f"No CSV files found in {INPUT_DIR}")

grouped = {}
for path in files:
    set_id = get_set_id(path.name)
    grouped.setdefault(set_id, []).append(path)

# =========================================================
# PROCESS EACH SET
# =========================================================
for set_id in sorted(grouped.keys(), key=lambda x: int(x)):
    paths = grouped[set_id]
    family_tables = {}

    for path in paths:
        df = pd.read_csv(path)
        family = get_family(df, path.name)

        if family in family_tables:
            raise ValueError(f"Set {set_id}: duplicate family '{family}' found.")

        expanded = expand_episodes_to_ticks(df, family, set_id)
        family_tables[family] = expanded

    expected_families = {"kmeans", "hdbscan", "hmm"}
    if set(family_tables.keys()) != expected_families:
        raise ValueError(
            f"Set {set_id}: expected {expected_families}, found {set(family_tables.keys())}"
        )

    keys = ["set_id", "session_id", "source_file", "valid_run_id", "tick"]

    merged = family_tables["kmeans"].merge(
        family_tables["hdbscan"],
        on=keys,
        how="outer",
        validate="one_to_one"
    ).merge(
        family_tables["hmm"],
        on=keys,
        how="outer",
        validate="one_to_one"
    )

    merged = coalesce_times(merged)

    merged = merged.sort_values(
        by=["session_id", "source_file", "valid_run_id", "tick"]
    ).reset_index(drop=True)

    missing_summary = merged[["kmeans", "hdbscan", "hmm"]].isna().sum()
    print(f"\nSet {set_id}")
    print("Missing values after merge:")
    print(missing_summary.to_string())
    
    # Keep only ticks present in all three designs
    before_n = len(merged)
    merged = merged.dropna(subset=["kmeans", "hdbscan", "hmm"]).copy()
    after_n = len(merged)
    
    print(f"Removed rows with incomplete design coverage: {before_n - after_n:,}")
    print(f"Remaining fully aligned rows: {after_n:,}")
    
    # Optional: cast meta-modes back to integer
    merged["kmeans"] = merged["kmeans"].astype(int)
    merged["hdbscan"] = merged["hdbscan"].astype(int)
    merged["hmm"] = merged["hmm"].astype(int)
    
    final_cols = [
        "set_id", "session_id", "source_file", "valid_run_id",
        "tick", "time", "kmeans", "hdbscan", "hmm"
    ]
    merged["time"] = pd.to_datetime(merged["time"]).dt.strftime("%Y-%m-%d %H:%M:%S")
    merged = merged[final_cols]
    
    out_path = OUTPUT_DIR / f"{set_id}.csv"
    merged.to_csv(out_path, index=False)
    print(f"Saved: {out_path} | rows: {len(merged):,}")


Set 1
Missing values after merge:
kmeans       0
hdbscan     36
hmm        238
Removed rows with incomplete design coverage: 238
Remaining fully aligned rows: 331,254
Saved: Stage4\stage4_timewise_sequences\1.csv | rows: 331,254

Set 2
Missing values after merge:
kmeans       0
hdbscan      0
hmm        308
Removed rows with incomplete design coverage: 308
Remaining fully aligned rows: 331,254
Saved: Stage4\stage4_timewise_sequences\2.csv | rows: 331,254

Set 3
Missing values after merge:
kmeans     106
hdbscan      0
hmm        308
Removed rows with incomplete design coverage: 308
Remaining fully aligned rows: 331,254
Saved: Stage4\stage4_timewise_sequences\3.csv | rows: 331,254

Set 4
Missing values after merge:
kmeans      70
hdbscan      0
hmm        308
Removed rows with incomplete design coverage: 308
Remaining fully aligned rows: 331,254
Saved: Stage4\stage4_timewise_sequences\4.csv | rows: 331,254

Set 5
Missing values after merge:
kmeans     106
hdbscan      0
hmm        308


### 4.3 Fragmentation and switching summaries

In [15]:
from pathlib import Path
import pandas as pd
import re

# =========================================================
# SETTINGS
# =========================================================
INPUT_DIR = Path("Stage4/stage4_metamode_mapped")
OUTPUT_FILE = Path("Stage4/stage4_fragmentation_switch_summary.csv")

def discover_metamode_files(input_dir: Path):
    pattern = re.compile(r"^(\d+)_.*__metamode_q\d+_dmin\d+\.csv$")
    files = [p.name for p in input_dir.glob("*.csv") if pattern.match(p.name)]
    if not files:
        raise FileNotFoundError(
            f"No mapped meta-mode files were found in {input_dir}. "
            "Run the Stage 4 meta-mode mapping cell first."
        )
    return sorted(files, key=lambda x: (int(re.match(r"^(\d+)_", x).group(1)), x))

FILES = discover_metamode_files(INPUT_DIR)

# =========================================================
# HELPERS
# =========================================================
def extract_group_id(filename):
    m = re.match(r"^(\d+)_", filename)
    if not m:
        raise ValueError(f"Cannot extract group_id from filename: {filename}")
    return int(m.group(1))

def extract_q_and_dmin(filename):
    mq = re.search(r"_q(\d+)_", filename)
    md = re.search(r"_dmin(\d+)\.csv$", filename)
    if not mq or not md:
        raise ValueError(f"Cannot extract q/dmin from filename: {filename}")
    return int(mq.group(1)), int(md.group(1))

# =========================================================
# MAIN
# =========================================================
summary_rows = []

for fname in FILES:
    path = INPUT_DIR / fname
    df = pd.read_csv(path)

    required_cols = [
        "design_id", "design_family",
        "session_id", "source_file", "valid_run_id",
        "s_k", "l_k", "meta_mode"
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{fname} is missing required columns: {missing}")

    # Standardize and sort
    df = df.copy()
    df["s_k"] = pd.to_numeric(df["s_k"], errors="coerce")
    df["l_k"] = pd.to_numeric(df["l_k"], errors="coerce")
    df["meta_mode"] = pd.to_numeric(df["meta_mode"], errors="coerce")

    df = df.sort_values(
        by=["session_id", "source_file", "valid_run_id", "s_k", "episode_id"],
        kind="stable"
    ).reset_index(drop=True)

    group_cols = ["session_id", "source_file", "valid_run_id"]

    # Previous labels only within the same run
    df["prev_l_k"] = df.groupby(group_cols)["l_k"].shift(1)
    df["prev_meta_mode"] = df.groupby(group_cols)["meta_mode"].shift(1)

    # Count switches only where a previous episode exists
    df["raw_switch"] = (
        df["prev_l_k"].notna() &
        df["l_k"].notna() &
        (df["l_k"] != df["prev_l_k"])
    ).astype(int)

    df["meta_switch"] = (
        df["prev_meta_mode"].notna() &
        df["meta_mode"].notna() &
        (df["meta_mode"] != df["prev_meta_mode"])
    ).astype(int)

    # Totals
    n_episodes_total = len(df)
    n_runs = df[group_cols].drop_duplicates().shape[0]
    n_transitions_total = n_episodes_total - n_runs
    S_raw_total = int(df["raw_switch"].sum())
    S_meta_total = int(df["meta_switch"].sum())

    group_id = extract_group_id(fname)
    q, stabilization_dmin = extract_q_and_dmin(fname)

    summary_rows.append({
        "group_id": group_id,
        "design_id": str(df["design_id"].iloc[0]),
        "design_family": str(df["design_family"].iloc[0]),
        "q": q,
        "stabilization_dmin": stabilization_dmin,
        "n_runs": int(n_runs),
        "n_episodes_total": int(n_episodes_total),
        "n_transitions_total": int(n_transitions_total),
        "S_raw_total": S_raw_total,
        "S_meta_total": S_meta_total,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(
    by=["group_id", "design_family", "design_id"]
).reset_index(drop=True)

summary_df.to_csv(OUTPUT_FILE, index=False)

print(summary_df)
print(f"\nSaved: {OUTPUT_FILE}")

    group_id                  design_id design_family  q  stabilization_dmin  \
0          1  hdbscan_15s_mcs3000_ms200       hdbscan  5                   5   
1          1           hmm_ns12_covdiag           hmm  5                   5   
2          1             kmeans_10s_k10        kmeans  5                   5   
3          2   hdbscan_5s_mcs3000_ms200       hdbscan  6                   5   
4          2           hmm_ns12_covdiag           hmm  6                   5   
5          2               kmeans_5s_k7        kmeans  6                   5   
6          3   hdbscan_5s_mcs3000_ms200       hdbscan  5                   5   
7          3           hmm_ns12_covdiag           hmm  5                   5   
8          3              kmeans_15s_k4        kmeans  5                   5   
9          4   hdbscan_5s_mcs3000_ms200       hdbscan  5                  10   
10         4           hmm_ns12_covdiag           hmm  5                  10   
11         4             kmeans_10s_k10 

### 4.4 Meta-mode coverage

In [17]:
from pathlib import Path
import pandas as pd
import re

# =========================================================
# SETTINGS
# =========================================================
INPUT_DIR = Path("Stage4/stage4_metamode_mapped")
OUTPUT_CSV = Path("Stage4/stage4_metamode_coverage_long.csv")

def discover_metamode_files(input_dir: Path):
    pattern = re.compile(r"^(\d+)_.*__metamode_q\d+_dmin\d+\.csv$")
    files = [p.name for p in input_dir.glob("*.csv") if pattern.match(p.name)]
    if not files:
        raise FileNotFoundError(
            f"No mapped meta-mode files were found in {input_dir}. "
            "Run the Stage 4 meta-mode mapping cell first."
        )
    return sorted(files, key=lambda x: (int(re.match(r"^(\d+)_", x).group(1)), x))

FILES = discover_metamode_files(INPUT_DIR)

# =========================================================
# HELPERS
# =========================================================
def extract_group_id(filename):
    m = re.match(r"^(\d+)_", filename)
    if not m:
        raise ValueError(f"Cannot extract group_id from filename: {filename}")
    return int(m.group(1))

def extract_q_and_dmin(filename):
    mq = re.search(r"_q(\d+)_", filename)
    md = re.search(r"_dmin(\d+)\.csv$", filename)
    if not mq or not md:
        raise ValueError(f"Cannot extract q/dmin from filename: {filename}")
    return int(mq.group(1)), int(md.group(1))

# =========================================================
# MAIN
# =========================================================
rows = []

for fname in FILES:
    path = INPUT_DIR / fname
    df = pd.read_csv(path)

    required_cols = ["design_id", "design_family", "d_k", "meta_mode"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{fname} is missing required columns: {missing}")

    df = df.copy()
    df["d_k"] = pd.to_numeric(df["d_k"], errors="coerce")
    df["meta_mode"] = pd.to_numeric(df["meta_mode"], errors="coerce")

    if df["d_k"].isna().any():
        raise ValueError(f"{fname} contains invalid d_k values.")
    if df["meta_mode"].isna().any():
        raise ValueError(f"{fname} contains invalid meta_mode values.")

    group_id = extract_group_id(fname)
    q, stabilization_dmin = extract_q_and_dmin(fname)
    design_id = str(df["design_id"].iloc[0])
    design_family = str(df["design_family"].iloc[0]).strip().lower()

    total_duration = df["d_k"].sum()

    cov = (
        df.groupby("meta_mode", as_index=False)["d_k"]
          .sum()
          .rename(columns={"d_k": "duration_in_mode"})
    )
    cov["coverage"] = cov["duration_in_mode"] / total_duration
    cov["group_id"] = group_id
    cov["design_id"] = design_id
    cov["design_family"] = design_family
    cov["q"] = q
    cov["stabilization_dmin"] = stabilization_dmin
    cov["total_duration"] = total_duration

    rows.append(cov)

coverage_long = pd.concat(rows, ignore_index=True)

coverage_long = coverage_long[
    [
        "group_id", "design_id", "design_family", "q", "stabilization_dmin",
        "meta_mode", "duration_in_mode", "total_duration", "coverage"
    ]
].sort_values(
    by=["group_id", "design_family", "design_id", "meta_mode"]
).reset_index(drop=True)

coverage_long.to_csv(OUTPUT_CSV, index=False)

print("Saved:", OUTPUT_CSV)
print("\nPreview:")
print(coverage_long.head(20).to_string(index=False))

Saved: Stage4\stage4_metamode_coverage_long.csv

Preview:
 group_id                 design_id design_family  q  stabilization_dmin  meta_mode  duration_in_mode  total_duration  coverage
        1 hdbscan_15s_mcs3000_ms200       hdbscan  5                   5          0            123163          331456  0.371582
        1 hdbscan_15s_mcs3000_ms200       hdbscan  5                   5          1             79148          331456  0.238789
        1 hdbscan_15s_mcs3000_ms200       hdbscan  5                   5          3             42593          331456  0.128503
        1 hdbscan_15s_mcs3000_ms200       hdbscan  5                   5          4             86552          331456  0.261127
        1          hmm_ns12_covdiag           hmm  5                   5          0            154901          331254  0.467620
        1          hmm_ns12_covdiag           hmm  5                   5          1             13364          331254  0.040344
        1          hmm_ns12_covdiag           

### 4.5 Transition structure

In [18]:
from pathlib import Path
import pandas as pd
import re

# =========================================================
# SETTINGS
# =========================================================
INPUT_DIR = Path("Stage4/stage4_metamode_mapped")
OUTPUT_CSV = Path("Stage4/stage4_transition_structure_long.csv")

def discover_metamode_files(input_dir: Path):
    pattern = re.compile(r"^(\d+)_.*__metamode_q\d+_dmin\d+\.csv$")
    files = [p.name for p in input_dir.glob("*.csv") if pattern.match(p.name)]
    if not files:
        raise FileNotFoundError(
            f"No mapped meta-mode files were found in {input_dir}. "
            "Run the Stage 4 meta-mode mapping cell first."
        )
    return sorted(files, key=lambda x: (int(re.match(r"^(\d+)_", x).group(1)), x))

FILES = discover_metamode_files(INPUT_DIR)

# =========================================================
# HELPERS
# =========================================================
def extract_group_id(filename):
    m = re.match(r"^(\d+)_", filename)
    if not m:
        raise ValueError(f"Cannot extract group_id from filename: {filename}")
    return int(m.group(1))

def extract_q_and_dmin(filename):
    mq = re.search(r"_q(\d+)_", filename)
    md = re.search(r"_dmin(\d+)\.csv$", filename)
    if not mq or not md:
        raise ValueError(f"Cannot extract q/dmin from filename: {filename}")
    return int(mq.group(1)), int(md.group(1))

# =========================================================
# MAIN
# =========================================================
rows = []

for fname in FILES:
    path = INPUT_DIR / fname
    df = pd.read_csv(path)

    required_cols = [
        "design_id", "design_family",
        "session_id", "source_file", "valid_run_id",
        "s_k", "meta_mode"
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{fname} is missing required columns: {missing}")

    df = df.copy()
    df["s_k"] = pd.to_numeric(df["s_k"], errors="coerce")
    df["meta_mode"] = pd.to_numeric(df["meta_mode"], errors="coerce")

    if df["s_k"].isna().any():
        raise ValueError(f"{fname} contains invalid s_k values.")
    if df["meta_mode"].isna().any():
        raise ValueError(f"{fname} contains invalid meta_mode values.")

    group_id = extract_group_id(fname)
    q, stabilization_dmin = extract_q_and_dmin(fname)
    design_id = str(df["design_id"].iloc[0])
    design_family = str(df["design_family"].iloc[0]).strip().lower()

    sort_cols = ["session_id", "source_file", "valid_run_id", "s_k"]
    if "episode_id" in df.columns:
        sort_cols.append("episode_id")

    df = df.sort_values(by=sort_cols, kind="stable").reset_index(drop=True)

    run_cols = ["session_id", "source_file", "valid_run_id"]

    df["prev_meta_mode"] = df.groupby(run_cols)["meta_mode"].shift(1)

    # Keep only within-run transitions and only a != b
    trans = df.loc[
        df["prev_meta_mode"].notna() &
        (df["meta_mode"] != df["prev_meta_mode"]),
        ["prev_meta_mode", "meta_mode"]
    ].copy()

    if trans.empty:
        continue

    trans["from_mode"] = trans["prev_meta_mode"].astype(int)
    trans["to_mode"] = trans["meta_mode"].astype(int)

    counts = (
        trans.groupby(["from_mode", "to_mode"], as_index=False)
             .size()
             .rename(columns={"size": "transition_count"})
    )

    total_transitions = counts["transition_count"].sum()
    counts["transition_share"] = counts["transition_count"] / total_transitions
    counts["group_id"] = group_id
    counts["design_id"] = design_id
    counts["design_family"] = design_family
    counts["q"] = q
    counts["stabilization_dmin"] = stabilization_dmin
    counts["total_transitions"] = total_transitions

    rows.append(counts)

transition_long = pd.concat(rows, ignore_index=True)

transition_long = transition_long[
    [
        "group_id", "design_id", "design_family", "q", "stabilization_dmin",
        "from_mode", "to_mode", "transition_count", "total_transitions", "transition_share"
    ]
].sort_values(
    by=["group_id", "design_family", "design_id", "from_mode", "to_mode"]
).reset_index(drop=True)

transition_long.to_csv(OUTPUT_CSV, index=False)

print("Saved:", OUTPUT_CSV)
print("\nPreview:")
print(transition_long.head(30).to_string(index=False))

Saved: Stage4\stage4_transition_structure_long.csv

Preview:
 group_id                 design_id design_family  q  stabilization_dmin  from_mode  to_mode  transition_count  total_transitions  transition_share
        1 hdbscan_15s_mcs3000_ms200       hdbscan  5                   5          0        1               413               4518          0.091412
        1 hdbscan_15s_mcs3000_ms200       hdbscan  5                   5          0        3               406               4518          0.089863
        1 hdbscan_15s_mcs3000_ms200       hdbscan  5                   5          1        0               356               4518          0.078796
        1 hdbscan_15s_mcs3000_ms200       hdbscan  5                   5          1        3               332               4518          0.073484
        1 hdbscan_15s_mcs3000_ms200       hdbscan  5                   5          1        4               126               4518          0.027888
        1 hdbscan_15s_mcs3000_ms200       hdbscan  

### 4.6 Top reciprocal transitions

In [20]:
from pathlib import Path
import pandas as pd

# =========================================================
# SETTINGS
# =========================================================
INPUT_CSV = Path("Stage4/stage4_transition_structure_long.csv")

# =========================================================
# LOAD
# =========================================================
df = pd.read_csv(INPUT_CSV)

required_cols = ["group_id", "design_family", "from_mode", "to_mode", "transition_count", "transition_share"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# =========================================================
# BUILD RECIPROCAL PAIRS
# a->b and b->a are merged into one pair a↔b
# =========================================================
df = df.copy()
df["pair_a"] = df[["from_mode", "to_mode"]].min(axis=1).astype(int)
df["pair_b"] = df[["from_mode", "to_mode"]].max(axis=1).astype(int)

pair_df = (
    df.groupby(["group_id", "design_family", "pair_a", "pair_b"], as_index=False)
      .agg(
          pair_count=("transition_count", "sum"),
          pair_share=("transition_share", "sum")
      )
)

pair_df["pair"] = pair_df["pair_a"].astype(str) + "↔" + pair_df["pair_b"].astype(str)

# =========================================================
# TAKE TOP 3 PAIRS PER GROUP + DESIGN
# =========================================================
pair_df = pair_df.sort_values(
    ["group_id", "design_family", "pair_share", "pair_count", "pair_a", "pair_b"],
    ascending=[True, True, False, False, True, True]
).reset_index(drop=True)

pair_df["rank"] = pair_df.groupby(["group_id", "design_family"]).cumcount() + 1
top3 = pair_df[pair_df["rank"] <= 3].copy()

# =========================================================
# RESHAPE TO WIDE TABLE
# =========================================================
pair_wide = (
    top3.pivot_table(
        index=["group_id", "design_family"],
        columns="rank",
        values="pair",
        aggfunc="first"
    )
    .rename(columns={1: "top_1_pair", 2: "top_2_pair", 3: "top_3_pair"})
)

share_wide = (
    top3.pivot_table(
        index=["group_id", "design_family"],
        columns="rank",
        values="pair_share",
        aggfunc="first"
    )
    .rename(columns={1: "top_1_share", 2: "top_2_share", 3: "top_3_share"})
)

result = pair_wide.join(share_wide).reset_index()

# =========================================================
# ORDER ROWS
# =========================================================
family_order = {"hmm": 0, "hdbscan": 1, "kmeans": 2}
result["_order"] = result["design_family"].map(family_order).fillna(99)

result = result.sort_values(["group_id", "_order", "design_family"]).drop(columns="_order").reset_index(drop=True)

# Round shares
for c in ["top_1_share", "top_2_share", "top_3_share"]:
    if c in result.columns:
        result[c] = result[c].round(4)

print(result.to_string(index=False))

 group_id design_family top_1_pair top_2_pair top_3_pair  top_1_share  top_2_share  top_3_share
        1           hmm        3↔4        0↔3        0↔4       0.5009       0.2684       0.0872
        1       hdbscan        3↔4        0↔3        0↔1       0.4471       0.1946       0.1702
        1        kmeans        3↔4        0↔3        0↔4       0.5487       0.2457       0.0483
        2           hmm        1↔3        0↔1        0↔3       0.5078       0.2235       0.0824
        2       hdbscan        1↔3        0↔1        0↔4       0.3645       0.1744       0.1458
        2        kmeans        1↔3        3↔5        1↔5       0.4661       0.2528       0.1503
        3           hmm        3↔4        0↔4        0↔3       0.5205       0.3145       0.0893
        3       hdbscan        3↔4        0↔4        0↔1       0.3846       0.1904       0.1870
        3        kmeans        1↔4        1↔2        0↔2       0.8714       0.0458       0.0394
        4           hmm        2↔3      

### 4.7 Pairwise cross-design stability

In [19]:
from pathlib import Path
import pandas as pd
import itertools

# =========================================================
# SETTINGS
# =========================================================
INPUT_DIR = Path("Stage4/stage4_timewise_sequences")
OUTPUT_CSV = Path("Stage4/stage4_pairwise_cross_design_stability.csv")

FILES = ["1.csv", "2.csv", "3.csv", "4.csv", "5.csv"]
DESIGNS = ["hmm", "hdbscan", "kmeans"]

# =========================================================
# MAIN
# =========================================================
rows = []

for fname in FILES:
    path = INPUT_DIR / fname
    df = pd.read_csv(path)

    required_cols = DESIGNS
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{fname} is missing required columns: {missing}")

    group_id = int(path.stem)

    # Keep only rows with complete aligned support for all compared designs
    df = df.copy()
    for col in DESIGNS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    for d1, d2 in itertools.combinations(DESIGNS, 2):
        sub = df[[d1, d2]].dropna().copy()

        T = len(sub)
        if T == 0:
            n_equal = 0
            n_unequal = 0
            agreement = pd.NA
        else:
            n_equal = int((sub[d1] == sub[d2]).sum())
            n_unequal = int(T - n_equal)
            agreement = n_equal / T

        rows.append({
            "group_id": group_id,
            "design_1": d1,
            "design_2": d2,
            "T": T,
            "n_equal": n_equal,
            "n_unequal": n_unequal,
            "agreement": agreement
        })

summary_df = pd.DataFrame(rows).sort_values(
    by=["group_id", "design_1", "design_2"]
).reset_index(drop=True)

summary_df.to_csv(OUTPUT_CSV, index=False)

print(summary_df)
print(f"\nSaved: {OUTPUT_CSV}")

    group_id design_1 design_2       T  n_equal  n_unequal  agreement
0          1  hdbscan   kmeans  331254   226628     104626   0.684152
1          1      hmm  hdbscan  331254   227408     103846   0.686506
2          1      hmm   kmeans  331254   241104      90150   0.727852
3          2  hdbscan   kmeans  331254   121980     209274   0.368237
4          2      hmm  hdbscan  331254   217191     114063   0.655663
5          2      hmm   kmeans  331254   142290     188964   0.429550
6          3  hdbscan   kmeans  331254    71806     259448   0.216770
7          3      hmm  hdbscan  331254   243853      87401   0.736151
8          3      hmm   kmeans  331254    50453     280801   0.152309
9          4  hdbscan   kmeans  331254   211856     119398   0.639558
10         4      hmm  hdbscan  331254   209560     121694   0.632626
11         4      hmm   kmeans  331254   240523      90731   0.726098
12         5  hdbscan   kmeans  331254   203349     127905   0.613876
13         5      hm

### 4.8 Robust episodic core

In [21]:
from pathlib import Path
import pandas as pd
import re
from collections import Counter

# =========================================================
# SETTINGS
# =========================================================
INPUT_DIR = Path("Stage4/stage4_metamode_mapped")
OUTPUT_CSV = Path("Stage4/stage4_robust_episodic_core_summary.csv")

def discover_metamode_files(input_dir: Path):
    pattern = re.compile(r"^(\d+)_.*__metamode_q\d+_dmin\d+\.csv$")
    files = [p.name for p in input_dir.glob("*.csv") if pattern.match(p.name)]
    if not files:
        raise FileNotFoundError(
            f"No mapped meta-mode files were found in {input_dir}. "
            "Run the Stage 4 meta-mode mapping cell first."
        )
    return sorted(files, key=lambda x: (int(re.match(r"^(\d+)_", x).group(1)), x))

FILES = discover_metamode_files(INPUT_DIR)

# Primary + sensitivity
N_VALUES = [2, 3]

# Support thresholds by pattern length
THETA_BY_N = {
    2: 0.01,
    3: 0.005,
}

# =========================================================
# HELPERS
# =========================================================
def extract_group_id(filename):
    m = re.match(r"^(\d+)_", filename)
    if not m:
        raise ValueError(f"Cannot extract group_id from filename: {filename}")
    return int(m.group(1))

def collapse_runs_to_sequences(df):
    """
    Build one collapsed meta-mode run sequence per run
    (session_id, source_file, valid_run_id).
    Consecutive equal meta_mode values are collapsed to one run.
    """
    required_cols = ["session_id", "source_file", "valid_run_id", "s_k", "meta_mode"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    tmp = df.copy()
    tmp["s_k"] = pd.to_numeric(tmp["s_k"], errors="coerce")
    tmp["meta_mode"] = pd.to_numeric(tmp["meta_mode"], errors="coerce")

    if tmp["s_k"].isna().any():
        raise ValueError("Found invalid s_k values.")
    if tmp["meta_mode"].isna().any():
        raise ValueError("Found invalid meta_mode values.")

    sort_cols = ["session_id", "source_file", "valid_run_id", "s_k"]
    if "episode_id" in tmp.columns:
        sort_cols.append("episode_id")

    tmp = tmp.sort_values(by=sort_cols, kind="stable").reset_index(drop=True)

    run_cols = ["session_id", "source_file", "valid_run_id"]
    sequences = []

    for _, g in tmp.groupby(run_cols, sort=False):
        seq = g["meta_mode"].astype(int).tolist()
        if not seq:
            continue

        collapsed = [seq[0]]
        for x in seq[1:]:
            if x != collapsed[-1]:
                collapsed.append(x)

        sequences.append(tuple(collapsed))

    return sequences

def extract_pattern_supports(sequences, n):
    """
    Relative support:
    count(pattern) / total number of n-windows across all sequences
    """
    counter = Counter()
    total_windows = 0

    for seq in sequences:
        if len(seq) < n:
            continue
        for i in range(len(seq) - n + 1):
            pat = seq[i:i+n]
            counter[pat] += 1
            total_windows += 1

    if total_windows == 0:
        return {}, 0

    supports = {pat: cnt / total_windows for pat, cnt in counter.items()}
    return supports, total_windows

def pattern_to_string(pat):
    return "→".join(str(x) for x in pat)

# =========================================================
# LOAD DESIGNS BY GROUP
# =========================================================
group_design_files = {}

for fname in FILES:
    gid = extract_group_id(fname)
    path = INPUT_DIR / fname
    df = pd.read_csv(path)

    if "design_family" not in df.columns:
        raise ValueError(f"{fname} is missing design_family")

    design_family = str(df["design_family"].iloc[0]).strip().lower()
    group_design_files.setdefault(gid, {})[design_family] = path

# Check all groups have all 3 designs
for gid in sorted(group_design_files.keys()):
    fams = set(group_design_files[gid].keys())
    expected = {"hmm", "hdbscan", "kmeans"}
    if fams != expected:
        raise ValueError(f"Group {gid} has {fams}, expected {expected}")

# =========================================================
# MAIN
# =========================================================
rows = []

for gid in sorted(group_design_files.keys()):
    design_supports = {}

    # Compute supports separately for each design in the triplet
    for design in ["hmm", "hdbscan", "kmeans"]:
        path = group_design_files[gid][design]
        df = pd.read_csv(path)

        sequences = collapse_runs_to_sequences(df)

        for n in N_VALUES:
            supports, total_windows = extract_pattern_supports(sequences, n)
            design_supports[(design, n)] = {
                "supports": supports,
                "total_windows": total_windows,
            }

    # Union of patterns within each group and n
    for n in N_VALUES:
        theta = THETA_BY_N[n]

        pattern_union = set()
        for design in ["hmm", "hdbscan", "kmeans"]:
            pattern_union.update(design_supports[(design, n)]["supports"].keys())

        for pat in sorted(pattern_union):
            supp_hmm = design_supports[("hmm", n)]["supports"].get(pat, 0.0)
            supp_hdbscan = design_supports[("hdbscan", n)]["supports"].get(pat, 0.0)
            supp_kmeans = design_supports[("kmeans", n)]["supports"].get(pat, 0.0)

            n_designs_ge_theta = sum([
                supp_hmm >= theta,
                supp_hdbscan >= theta,
                supp_kmeans >= theta
            ])

            rows.append({
                "group_id": gid,
                "n": n,
                "theta": theta,
                "pattern": pattern_to_string(pat),
                "supp_hmm": supp_hmm,
                "supp_hdbscan": supp_hdbscan,
                "supp_kmeans": supp_kmeans,
                "n_designs_ge_theta": n_designs_ge_theta,
                "in_core_rho2": int(n_designs_ge_theta >= 2),
                "in_core_rho3": int(n_designs_ge_theta >= 3),
                "total_windows_hmm": design_supports[("hmm", n)]["total_windows"],
                "total_windows_hdbscan": design_supports[("hdbscan", n)]["total_windows"],
                "total_windows_kmeans": design_supports[("kmeans", n)]["total_windows"],
            })

result = pd.DataFrame(rows)

# Keep only patterns that pass theta in at least one design
result = result.loc[result["n_designs_ge_theta"] >= 1].copy()

# Order columns
result = result[
    [
        "group_id", "n", "theta", "pattern",
        "supp_hmm", "supp_hdbscan", "supp_kmeans",
        "n_designs_ge_theta", "in_core_rho2", "in_core_rho3",
        "total_windows_hmm", "total_windows_hdbscan", "total_windows_kmeans"
    ]
].sort_values(
    by=["group_id", "n", "n_designs_ge_theta", "supp_hmm", "supp_hdbscan", "supp_kmeans", "pattern"],
    ascending=[True, True, False, False, False, False, True]
).reset_index(drop=True)

result.to_csv(OUTPUT_CSV, index=False)

print(result.head(60).to_string(index=False))
print(f"\nSaved: {OUTPUT_CSV}")

 group_id  n  theta pattern  supp_hmm  supp_hdbscan  supp_kmeans  n_designs_ge_theta  in_core_rho2  in_core_rho3  total_windows_hmm  total_windows_hdbscan  total_windows_kmeans
        1  2  0.010     3→4  0.281525      0.218902     0.260274                   3             1             1               5115                   4518                  5548
        1  2  0.010     4→3  0.219355      0.228198     0.288392                   3             1             1               5115                   4518                  5548
        1  2  0.010     0→3  0.167742      0.089863     0.109048                   3             1             1               5115                   4518                  5548
        1  2  0.010     3→0  0.100684      0.104692     0.136626                   3             1             1               5115                   4518                  5548
        1  2  0.010     0→1  0.021505      0.091412     0.025234                   3             1             1   